# Trading Strategy Assignment: Exploratory Data Analysis and Strategy Optimization

## Overview

This assignment explores the intersection of **technical analysis**, **natural language processing**, and **large-scale data processing** in the context of algorithmic trading. You will work with historical market data and earnings call transcripts to develop and evaluate trading strategies.

### Learning Objectives

- **Technical Analysis**: Implement and analyze technical indicators (Moving Averages, RSI, MACD, Bollinger Bands)
- **NLP for Finance**: Apply FinBERT (a financial BERT model) for sentiment analysis on earnings transcripts
- **Scalable Computing**: Explore parallelization techniques for computing indicators on large datasets—consider approaches that can reduce latency compared to iterative pandas operations
- **Strategy Development**: Systematically improve a baseline strategy through iterative experimentation
- **Performance Evaluation**: Use proper train/validation split methodology for model development

---

## Data Description

### Price Data
S&P 500 historical price data with the following schema:

| Column | Description |
|--------|-------------|
| `ticker` | Stock symbol (e.g., AAPL, MSFT) |
| `date` | Trading date |
| `open`, `high`, `low`, `close` | OHLC prices |
| `volume` | Trading volume |

### Earnings Data
Earnings call transcripts for sentiment analysis:

| Column | Description |
|--------|-------------|
| `ticker` | Stock symbol |
| `date` | Earnings call date |
| `transcript` | Full text of earnings call |
| `quarter` | Fiscal quarter (e.g., "Q1 2023") |

---

## Data Splits

The data is partitioned to enable proper model development:

| Split | Date Range | Purpose |
|-------|------------|---------|
| **Dev** | 2000-2017 | Create subsets for development and hyperparameter tuning |
| **Val** | 2018-2024 | Final performance reporting (unseen during development) |

**Important**: Use the dev split and create subsets to iterate and experiment. Reserve the val split for final performance reporting only.

---

## Provided Infrastructure

### Baseline Strategy
A simple moving average crossover strategy enhanced with FinBERT sentiment:
- **Entry**: Price > MA-50 AND (no earnings OR positive sentiment)
- **Exit**: Price < MA-50 OR stop-loss at 20%

### Evaluation Framework
Weekly rebalancing backtest simulation with the following metrics:

| Metric | Description | Target |
|--------|-------------|--------|
| **Total Return** | Portfolio value change from starting capital | Maximize |
| **Sharpe Ratio** | Risk-adjusted return (higher is better) | >1.0 is good |
| **Max Drawdown** | Largest peak-to-trough decline | Minimize |
| **Win Rate** | Percentage of profitable trades | >40% |
| **Volatility** | Standard deviation of returns | Lower for same return |

### Visualization Tools
- Portfolio value over time
- Drawdown analysis
- Returns distribution
- Rolling returns
- Comparison charts

---

## Assignment Workflow

1. **Run cells 1-10** to establish baseline performance
2. **Perform advanced EDA** to understand the data
3. **Implement improvements** in `EnhancedStrategy` (Cell 12) across these areas:
   - Data Quality & Cleaning
   - Technical Indicators
   - Enhanced NLP Analysis
   - Smarter Decision Logic
4. **Iterate based on subsets created in dev split** to tune your strategy
5. **Report final performance** on val split when satisfied

---

## Expected Outcomes

By the end of this assignment, you should have:

1. **Implemented** an enhanced trading strategy that improves upon the baseline
2. **Demonstrated** understanding of scalable computing for large-scale data processing
3. **Documented** your development process and key findings
4. **Achieved** measurable improvements in key metrics (Return, Sharpe, Drawdown)

## GPU Configuration (Optional)

**Important:** You can configure your runtime to use GPU acceleration before executing any cells. GPU acceleration provides 3-4x faster inference for the FinBERT model.

### Configuration Steps

Navigate to **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

### Verification

The first cell has print statement that informs the hardware being used

In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================

import pandas as pd
import numpy as np
import time
import warnings
from datetime import timedelta
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import pipeline
import torch

# Local data directory (use INF2006_Data_Students folder)
DATA_DIR = Path(__file__).parent / 'INF2006_Data_Students' if '__file__' in dir() else Path('.') / 'INF2006_Data_Students'
# Fallback: set explicitly if the above doesn't resolve correctly
if not DATA_DIR.exists():
    DATA_DIR = Path(r'c:\Users\Sebert\Desktop\SIT Lambert\Y2T2\INF2006 - Cloud Computing and Big Data\Project\Algorithmic Trading\INF2006_Data_Students')

warnings.filterwarnings('ignore')

# Configuration
STARTING_CASH = 100000
FINBERT_MODEL = "ProsusAI/finbert"
DEVICE = 0 if torch.cuda.is_available() else -1

print(f"Data directory: {DATA_DIR}")
print(f"Initial capital: ${STARTING_CASH:,.0f}")
print(f"Device: {'GPU' if DEVICE >= 0 else 'CPU'}")

In [ ]:
# ============================================================
# CELL 2: Initialize FinBERT
# ============================================================

print("Loading FinBERT (first run downloads ~420MB)...")

finbert_pipeline = pipeline(
    "sentiment-analysis",
    model=FINBERT_MODEL,
    tokenizer=FINBERT_MODEL,
    device=DEVICE,
    return_all_scores=True,
    truncation=True,
    max_length=512
)

print("FinBERT loaded.")

In [ ]:
# ============================================================
# CELL 3: Data Loading
# ============================================================

import shutil
import tempfile

def load_prices(split='dev'):
    filepath = DATA_DIR / f'prices_{split}.parquet'
    if not filepath.exists():
        raise FileNotFoundError(f"Not found: {filepath}")
    # Copy to short temp path to avoid Windows long-path issues
    tmp = Path(tempfile.gettempdir()) / f'prices_{split}.parquet'
    shutil.copy2(filepath, tmp)
    return pd.read_parquet(tmp)

def load_earnings(split='dev'):
    filepath = DATA_DIR / f'earnings_{split}.parquet'
    if not filepath.exists():
        raise FileNotFoundError(f"Not found: {filepath}")
    tmp = Path(tempfile.gettempdir()) / f'earnings_{split}.parquet'
    shutil.copy2(filepath, tmp)
    return pd.read_parquet(tmp)

prices_dev = load_prices('dev')
prices_val = load_prices('val')
earnings_dev = load_earnings('dev')
earnings_val = load_earnings('val')

print(f"Dev prices:   {len(prices_dev):,} records")
print(f"Val prices:   {len(prices_val):,} records")
print(f"Dev earnings: {len(earnings_dev):,} records")
print(f"Val earnings: {len(earnings_val):,} records")

In [ ]:
# ============================================================
# CELL 4: Exploratory Data Analysis
# ============================================================

print("="*60)
print("EXPLORATORY DATA ANALYSIS")
print("="*60)

print(f"\n[PRICE DATA]")
print(f"Date range: {prices_dev['date'].min()} to {prices_dev['date'].max()}")
print(f"Unique tickers: {prices_dev['ticker'].nunique()}")
print(f"\nPrice statistics:")
print(prices_dev[['open', 'high', 'low', 'close', 'volume']].describe())

print(f"\n[MISSING VALUES]")
print(prices_dev.isnull().sum())

print(f"\n[EARNINGS DATA]")
print(f"Date range: {earnings_dev['date'].min()} to {earnings_dev['date'].max()}")
print(f"Unique tickers: {earnings_dev['ticker'].nunique()}")

earnings_dev['transcript_length'] = earnings_dev['transcript'].str.len()
print(f"\nTranscript lengths:")
print(earnings_dev['transcript_length'].describe())

price_tickers = set(prices_dev['ticker'].unique())
earnings_tickers = set(earnings_dev['ticker'].unique())
overlap = price_tickers & earnings_tickers
print(f"\n[TICKER OVERLAP]")
print(f"Prices: {len(price_tickers)}, Earnings: {len(earnings_tickers)}, Overlap: {len(overlap)}")

print("="*60)

In [ ]:
# ============================================================
# CELL 5: Data Visualization
# ============================================================

sample_ticker = prices_dev['ticker'].value_counts().index[0]
sample_prices = prices_dev[prices_dev['ticker'] == sample_ticker].sort_values('date')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(pd.to_datetime(sample_prices['date']), sample_prices['close'])
axes[0, 0].set_title(f'{sample_ticker} Price Over Time')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Close Price')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(pd.to_datetime(sample_prices['date']), sample_prices['volume'])
axes[0, 1].set_title(f'{sample_ticker} Volume')
axes[0, 1].set_xlabel('Date')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(prices_dev['close'], bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Price Distribution')
axes[1, 0].set_xlabel('Close Price')
axes[1, 0].grid(True, alpha=0.3)

ticker_counts = prices_dev['ticker'].value_counts().head(20)
axes[1, 1].barh(range(len(ticker_counts)), ticker_counts.values)
axes[1, 1].set_yticks(range(len(ticker_counts)))
axes[1, 1].set_yticklabels(ticker_counts.index)
axes[1, 1].set_title('Top 20 Tickers by Data Points')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Infrastructure

The following classes provide the foundation. You can extend or modify these as you explore.

---

## How the Backtest Simulation Works

### Weekly Rebalancing

The simulation implements a **weekly rebalancing** strategy:

1. **Every Friday**, the strategy evaluates all 400+ S&P 500 stocks
2. For each stock, the strategy makes a decision: `BUY`, `SELL`, or `HOLD`
3. The portfolio is rebalanced based on these decisions
4. Portfolio value and positions are tracked weekly

### Role of Earnings Calls

Earnings call transcripts provide **qualitative sentiment** that complements technical indicators:

| Data Source | Type | Example Signal |
|-------------|------|---------------|
| **Price/Technical** | Quantitative | Price > MA-50 (bullish trend) |
| **Earnings Transcript** | Qualitative | Management sounds confident about Q4 guidance (positive sentiment) |

**FinBERT** (Financial BERT) analyzes earnings transcripts to classify sentiment:
- **Positive**: Management upbeat, strong guidance, growth opportunities
- **Negative**: Cautionary language, cost-cutting, headwinds
- **Neutral**: Factual reporting, balanced outlook

**Example Usage in Strategy:**
```python
# Entry: Price in uptrend AND positive (or no) earnings sentiment
if price > ma_50 and (sentiment == 'positive' or sentiment is None):
    return 'BUY'

# Exit: Price in downtrend OR negative earnings sentiment
if price < ma_50 or sentiment == 'negative':
    return 'SELL'
```


In [ ]:
# ============================================================
# CELL 6: Portfolio Class
# ============================================================

class Portfolio:
    def __init__(self, starting_cash=100000):
        self.cash = starting_cash
        self.positions = {}
        self.trades = []

    def buy_target(self, ticker, price, date, target_value=5000):
        if ticker in self.positions:
            return 0
        max_shares = int(target_value // price)
        if max_shares <= 0:
            return 0
        cost = min(max_shares * price, self.cash)
        shares = int(cost // price)
        if shares <= 0:
            return 0
        self.cash -= cost
        self.positions[ticker] = {'shares': shares, 'buy_price': price}
        self.trades.append({'date': date, 'ticker': ticker, 'action': 'BUY',
                          'shares': shares, 'price': price, 'value': cost})
        return shares

    def sell(self, ticker, price, date):
        if ticker not in self.positions:
            return 0
        position = self.positions[ticker]
        shares = position['shares']
        proceeds = shares * price
        del self.positions[ticker]
        self.cash += proceeds
        self.trades.append({'date': date, 'ticker': ticker, 'action': 'SELL',
                          'shares': shares, 'price': price, 'value': proceeds})
        return shares

    def get_value(self, current_prices):
        total = self.cash
        for ticker, pos in self.positions.items():
            if ticker in current_prices:
                total += pos['shares'] * current_prices[ticker]
        return total

    def get_state(self, current_prices):
        return {
            'cash': self.cash,
            'positions': {t: {'shares': p['shares'], 'buy_price': p['buy_price']}
                         for t, p in self.positions.items()},
            'total_value': self.get_value(current_prices)
        }

print("Portfolio class loaded")

In [ ]:
# ============================================================
# CELL 7: Trading Simulation
# ============================================================

class TradingSimulation:
    def __init__(self, prices, earnings, starting_cash=100000):
        self.prices = prices
        self.earnings = earnings
        self.portfolio = Portfolio(starting_cash)
        self.weekly_schedule = self._create_weekly_schedule()
        self._build_lookups()
        # Build price history for fallback to most recent price
        self._build_price_history()

    def _create_weekly_schedule(self):
        min_date = pd.to_datetime(self.prices['date']).min()
        max_date = pd.to_datetime(self.prices['date']).max()
        return pd.date_range(start=min_date, end=max_date, freq='W-FRI').strftime('%Y-%m-%d').tolist()

    def _build_lookups(self):
        # Pre-compute O(1) lookup dictionaries
        # IMPORTANT: Convert dates to strings for consistent matching
        self.prices_by_ticker_date = {}
        for ticker in self.prices['ticker'].unique():
            ticker_prices = self.prices[self.prices['ticker'] == ticker].sort_values('date')
            for _, row in ticker_prices.iterrows():
                # Convert date to string for consistent lookup
                date_str = row['date'] if isinstance(row['date'], str) else pd.to_datetime(row['date']).strftime('%Y-%m-%d')
                self.prices_by_ticker_date[(ticker, date_str)] = row['close']

        self.earnings_by_ticker_week = {}
        for ticker in self.earnings['ticker'].unique():
            ticker_earnings = self.earnings[self.earnings['ticker'] == ticker].sort_values('date')
            for _, row in ticker_earnings.iterrows():
                earnings_date = pd.to_datetime(row['date'])
                week_end = (earnings_date + timedelta (days=(4 - earnings_date.weekday()) % 7)).strftime('%Y-%m-%d')
                self.earnings_by_ticker_week[(ticker, week_end)] = row['transcript']

    def _build_price_history(self):
        """Build price history for each ticker to enable fallback to most recent price."""
        self.price_history = {}
        for ticker in self.prices['ticker'].unique():
            ticker_prices = self.prices[self.prices['ticker'] == ticker].sort_values('date')
            # Store as list of (date_string, price) tuples
            self.price_history[ticker] = [
                (row['date'] if isinstance(row['date'], str) else pd.to_datetime(row['date']).strftime('%Y-%m-%d'), row['close'])
                for _, row in ticker_prices.iterrows()
            ]

    def _get_price_on_date(self, ticker, date):
        """Get price for ticker on specific date, with fallback to most recent price."""
        # Try direct lookup first
        direct = self.prices_by_ticker_date.get((ticker, date))
        if direct is not None:
            return direct

        # Fallback: find most recent price before this date
        if ticker in self.price_history:
            for hist_date, price in reversed(self.price_history[ticker]):
                if hist_date <= date:
                    return price

        return None

    def _get_current_prices(self, date):
        """Get current prices for all tickers, with fallback to most recent prices."""
        current_prices = {}
        for ticker in self.prices['ticker'].unique():
            price = self._get_price_on_date(ticker, date)
            if price is not None:
                current_prices[ticker] = price
        return current_prices

    def _get_recent_earnings(self, ticker, current_date, days_lookback=7):
        return self.earnings_by_ticker_week.get((ticker, current_date))

    def run(self, strategy_function, analytics_lookup, verbose=False):
        portfolio_history = []
        all_tickers = sorted(self.prices['ticker'].unique())

        for i, week_date in enumerate(self.weekly_schedule):
            if verbose and i % 10 == 0:
                print(f"  Week {i+1}/{len(self.weekly_schedule)}: {week_date}")

            current_prices = self._get_current_prices(week_date)
            portfolio_state = self.portfolio.get_state(current_prices)

            for ticker in all_tickers:
                transcript = self._get_recent_earnings(ticker, week_date)

                ticker_data = analytics_lookup.get(ticker, [])
                latest_analytics = None
                for analytics_date, analytics_dict in ticker_data:
                    if analytics_date <= week_date:
                        latest_analytics = analytics_dict
                    else:
                        break
                if latest_analytics is None:
                    continue

                decision = strategy_function(ticker, week_date, transcript, portfolio_state, latest_analytics)
                price = self._get_price_on_date(ticker, week_date)
                if price is None:
                    continue

                if decision == 'BUY':
                    self.portfolio.buy_target(ticker, price, week_date, target_value=5000)
                elif decision == 'SELL':
                    self.portfolio.sell(ticker, price, week_date)

            portfolio_history.append({
                'date': week_date,
                'portfolio_value': self.portfolio.get_value(current_prices),
                'cash': self.portfolio.cash,
                'positions': len(self.portfolio.positions)
            })

        final_date = self.weekly_schedule[-1]
        final_prices = self._get_current_prices(final_date)
        return {
            'trades': self.portfolio.trades,
            'portfolio_history': portfolio_history,
            'final_portfolio': self.portfolio.get_state(final_prices),
            'final_prices': final_prices
        }

print("TradingSimulation class loaded")

In [ ]:
# ============================================================
# CELL 8: Performance Metrics & Visualization
# ============================================================

def calculate_metrics(results, starting_cash=STARTING_CASH):
    """Calculate comprehensive performance metrics."""
    history_df = pd.DataFrame(results['portfolio_history'])
    history_df['date'] = pd.to_datetime(history_df['date'])
    trades_df = pd.DataFrame(results['trades'])

    final_value = results['final_portfolio']['total_value']
    total_return = (final_value - starting_cash) / starting_cash

    history_df['daily_return'] = history_df['portfolio_value'].pct_change()

    mean_daily_return = history_df['daily_return'].mean()
    std_daily_return = history_df['daily_return'].std()
    sharpe_ratio = (mean_daily_return / std_daily_return * np.sqrt(252)) if std_daily_return > 0 else 0

    peak = history_df['portfolio_value'].cummax()
    drawdown = (history_df['portfolio_value'] - peak) / peak
    max_drawdown = drawdown.min()

    if len(trades_df) > 0:
        buy_trades = trades_df[trades_df['action'] == 'BUY']
        sell_trades = trades_df[trades_df['action'] == 'SELL']
        profitable_trades = 0
        total_trades = 0
        for _, sell in sell_trades.iterrows():
            buy = buy_trades[buy_trades['ticker'] == sell['ticker']]
            buy = buy[buy['date'] < sell['date']]
            if len(buy) > 0:
                buy = buy.iloc[-1]
                if sell['value'] > buy['value']:
                    profitable_trades += 1
                total_trades += 1
        win_rate = profitable_trades / total_trades if total_trades > 0 else 0
    else:
        win_rate = 0

    volatility = std_daily_return * np.sqrt(252) if std_daily_return > 0 else 0

    return {
        'total_return': total_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'volatility': volatility,
        'num_trades': len(trades_df),
        'final_positions': len(results['final_portfolio']['positions'])
    }


def plot_results(results, metrics, title="Strategy Results", starting_cash=STARTING_CASH):
    """Generate visualization plots for strategy results."""
    history_df = pd.DataFrame(results['portfolio_history'])
    history_df['date'] = pd.to_datetime(history_df['date'])

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    # Portfolio value
    axes[0, 0].plot(history_df['date'], history_df['portfolio_value'], linewidth=2, color='#2E86AB')
    axes[0, 0].axhline(y=starting_cash, color='r', linestyle='--', alpha=0.5, label='Starting Capital')
    axes[0, 0].set_title('Portfolio Value')
    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('Value ($)')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

    # Cash
    axes[0, 1].plot(history_df['date'], history_df['cash'], color='#22C55E', linewidth=2)
    axes[0, 1].set_title('Cash Position')
    axes[0, 1].set_xlabel('Date')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

    # Positions
    axes[0, 2].plot(history_df['date'], history_df['positions'], color='#9467BE', linewidth=2)
    axes[0, 2].set_title('Number of Positions')
    axes[0, 2].set_xlabel('Date')
    axes[0, 2].grid(True, alpha=0.3)

    # Drawdown
    peak = history_df['portfolio_value'].cummax()
    drawdown = (history_df['portfolio_value'] - peak) / peak * 100
    axes[1, 0].fill_between(history_df['date'], drawdown, 0, color='#E15759', alpha=0.3)
    axes[1, 0].plot(history_df['date'], drawdown, color='#E15759', linewidth=1)
    axes[1, 0].set_title('Drawdown %')
    axes[1, 0].set_xlabel('Date')
    axes[1, 0].grid(True, alpha=0.3)

    # Daily returns distribution - filter out inf and NaN values
    daily_returns = history_df['portfolio_value'].pct_change().dropna() * 100
    # Filter out infinite and extreme values that can occur when portfolio goes to zero
    daily_returns = daily_returns[np.isfinite(daily_returns)]
    daily_returns = daily_returns[daily_returns > -100]  # Remove -100% (total loss) outliers for cleaner histogram
    if len(daily_returns) > 0:
        axes[1, 1].hist(daily_returns, bins=30, edgecolor='black', alpha=0.7, color='#4DB6AC')
    axes[1, 1].set_title('Weekly Returns Distribution')
    axes[1, 1].set_xlabel('Return (%)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

    # Rolling returns - also filter out inf values
    rolling_return = history_df['portfolio_value'].pct_change(periods=20).dropna() * 100
    rolling_return = rolling_return[np.isfinite(rolling_return)]
    axes[1, 2].plot(history_df['date'][:len(rolling_return)], rolling_return, linewidth=2, color='#FF9845')
    axes[1, 2].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[1, 2].set_title('20-Week Rolling Return')
    axes[1, 2].set_xlabel('Date')
    axes[1, 2].set_ylabel('Return (%)')
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_comparison(results_baseline, metrics_baseline, results_enhanced, metrics_enhanced, starting_cash=STARTING_CASH):
    """Plot side-by-side comparison of baseline vs enhanced strategy."""
    baseline_df = pd.DataFrame(results_baseline['portfolio_history'])
    baseline_df['date'] = pd.to_datetime(baseline_df['date'])
    baseline_df['strategy'] = 'Baseline'

    enhanced_df = pd.DataFrame(results_enhanced['portfolio_history'])
    enhanced_df['date'] = pd.to_datetime(enhanced_df['date'])
    enhanced_df['strategy'] = 'Enhanced'

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Baseline vs Enhanced Strategy Comparison', fontsize=16, fontweight='bold')

    # Portfolio Value Over Time
    axes[0, 0].plot(baseline_df['date'], baseline_df['portfolio_value'], linewidth=2, label='Baseline', color='#E15759', alpha=0.8)
    axes[0, 0].plot(enhanced_df['date'], enhanced_df['portfolio_value'], linewidth=2, label='Enhanced', color='#2E86AB', alpha=0.8)
    axes[0, 0].axhline(y=starting_cash, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    axes[0, 0].set_title('Portfolio Value Over Time')
    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('Portfolio Value ($)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

    # Drawdown Comparison
    peak_baseline = baseline_df['portfolio_value'].cummax()
    drawdown_baseline = (baseline_df['portfolio_value'] - peak_baseline) / peak_baseline * 100
    peak_enhanced = enhanced_df['portfolio_value'].cummax()
    drawdown_enhanced = (enhanced_df['portfolio_value'] - peak_enhanced) / peak_enhanced * 100

    axes[0, 1].fill_between(baseline_df['date'], drawdown_baseline, 0, color='#E15759', alpha=0.3, label='Baseline')
    axes[0, 1].plot(baseline_df['date'], drawdown_baseline, color='#E15759', linewidth=1, alpha=0.7)
    axes[0, 1].fill_between(enhanced_df['date'], drawdown_enhanced, 0, color='#2E86AB', alpha=0.3, label='Enhanced')
    axes[0, 1].plot(enhanced_df['date'], drawdown_enhanced, color='#2E86AB', linewidth=1, alpha=0.7)
    axes[0, 1].set_title('Drawdown Comparison (%)')
    axes[0, 1].set_xlabel('Date')
    axes[0, 1].set_ylabel('Drawdown %')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Metrics Comparison Bar Chart
    metrics_names = ['Total Return', 'Sharpe Ratio', 'Win Rate', 'Max Drawdown']
    baseline_values = [
        metrics_baseline['total_return'] * 100,
        metrics_baseline['sharpe_ratio'],
        metrics_baseline['win_rate'] * 100,
        abs(metrics_baseline['max_drawdown']) * 100
    ]
    enhanced_values = [
        metrics_enhanced['total_return'] * 100,
        metrics_enhanced['sharpe_ratio'],
        metrics_enhanced['win_rate'] * 100,
        abs(metrics_enhanced['max_drawdown']) * 100
    ]

    x = np.arange(len(metrics_names))
    width = 0.35

    bars1 = axes[1, 0].bar(x - width/2, baseline_values, width, label='Baseline', color='#E15759', alpha=0.8)
    bars2 = axes[1, 0].bar(x + width/2, enhanced_values, width, label='Enhanced', color='#2E86AB', alpha=0.8)
    axes[1, 0].set_title('Key Metrics Comparison')
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(metrics_names, rotation=15, ha='right')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)

    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.1f}', ha='center', va='bottom', fontsize=8)

    # Number of Positions Over Time
    axes[1, 1].plot(baseline_df['date'], baseline_df['positions'], linewidth=2, label='Baseline', color='#E15759', alpha=0.8)
    axes[1, 1].plot(enhanced_df['date'], enhanced_df['positions'], linewidth=2, label='Enhanced', color='#2E86AB', alpha=0.8)
    axes[1, 1].set_title('Number of Positions Over Time')
    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('Number of Positions')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def print_detailed_comparison(baseline_metrics, enhanced_metrics):
    """Print detailed metrics comparison table with improvement indicators."""
    print("\n" + "="*70)
    print("PERFORMANCE COMPARISON")
    print("="*70)

    print(f"{'Metric':<20} {'Baseline':<15} {'Enhanced':<15} {'Improvement':<15}")
    print(f"{'-'*70}")

    # Total Return
    return_diff = enhanced_metrics['return'] - baseline_metrics['return']
    return_arrow = "↑" if return_diff > 0 else "↓"
    print(f"{'Total Return':<20} {baseline_metrics['return']:>13.2%} {enhanced_metrics['return']:>13.2%} {return_arrow} {abs(return_diff):>+9.2%}")

    # Sharpe Ratio
    sharpe_diff = enhanced_metrics['sharpe'] - baseline_metrics['sharpe']
    sharpe_arrow = "↑" if sharpe_diff > 0 else "↓"
    print(f"{'Sharpe Ratio':<20} {baseline_metrics['sharpe']:>13.2f} {enhanced_metrics['sharpe']:>13.2f} {sharpe_arrow} {abs(sharpe_diff):>+9.2f}")

    # Max Drawdown (lower is better, so arrow direction flips)
    drawdown_diff = enhanced_metrics['drawdown'] - baseline_metrics['drawdown']
    drawdown_arrow = "↓" if drawdown_diff > 0 else "↑"
    print(f"{'Max Drawdown':<20} {baseline_metrics['drawdown']:>13.2%} {enhanced_metrics['drawdown']:>13.2%} {drawdown_arrow} {abs(drawdown_diff):>+9.2%}")

    # Win Rate
    winrate_diff = enhanced_metrics['win_rate'] - baseline_metrics['win_rate']
    winrate_arrow = "↑" if winrate_diff > 0 else "↓"
    print(f"{'Win Rate':<20} {baseline_metrics['win_rate']:>13.2%} {enhanced_metrics['win_rate']:>13.2%} {winrate_arrow} {abs(winrate_diff):>+9.2%}")

    # Volatility (lower is better for same return)
    volatility_diff = enhanced_metrics['volatility'] - baseline_metrics['volatility']
    volatility_arrow = "↓" if volatility_diff > 0 else "↑"
    print(f"{'Volatility':<20} {baseline_metrics['volatility']:>13.2%} {enhanced_metrics['volatility']:>13.2%} {volatility_arrow} {abs(volatility_diff):>+9.2%}")

    # Total Trades
    trades_diff = enhanced_metrics['trades'] - baseline_metrics['trades']
    trades_arrow = "↑" if trades_diff > 0 else "↓"
    print(f"{'Total Trades':<20} {baseline_metrics['trades']:>13d} {enhanced_metrics['trades']:>13d} {trades_arrow} {abs(trades_diff):>+9d}")

    print("="*70)

In [ ]:
# ============================================================
# CELL 9: Strategy Base Class
# ============================================================

class BaseStrategy:
    def __init__(self, finbert_pipeline=None):
        self.finbert_pipeline = finbert_pipeline
        self.llm_cache = {}
        self.llm_cache_hits = 0
        self.llm_cache_misses = 0
        self.prices = None
        self.earnings = None

    def set_data(self, prices_df, earnings_df):
        """
        Set and preprocess data for evaluation.
        Call this before evaluate().
        """
        print("Cleaning and preprocessing data...")
        self.prices, self.earnings = self.clean_data(prices_df, earnings_df)
        print(f"Data ready: {len(self.prices):,} price records")

    def clean_data(self, prices_df, earnings_df):
        """Clean and validate data. Override for enhanced cleaning."""
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]
        return prices, earnings

    def calculate_analytics(self, prices_df):
        """Calculate technical indicators. Override for enhanced analytics."""
        print("Computing technical indicators (MA-50)...")
        results = []
        for ticker in prices_df['ticker'].unique():
            df = prices_df[prices_df['ticker'] == ticker].copy().sort_values('date')
            df['ma_50'] = df['close'].rolling(50, min_periods=1).mean()
            results.append(df[['ticker', 'date', 'ma_50', 'close']])
        result_df = pd.concat(results, ignore_index=True)
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')
        print(f"Technical indicators computed: {len(result_df):,} rows")
        return result_df

    def llm_analysis(self, ticker, transcript, date):
        """Extract sentiment from earnings. Override for enhanced NLP."""
        if transcript is None or self.finbert_pipeline is None:
            return None

        # Use last 2000 characters of transcript
        text = transcript[-2000:] if len(transcript) > 2000 else transcript

        try:
            results = self.finbert_pipeline(text)
            # FinBERT returns sentiment classification
            sentiment = results[0][0]['label']  # 'positive', 'negative', or 'neutral'
            return {'sentiment': sentiment}
        except:
            return None

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        """
        Make trading decision. Override for enhanced logic.

        BASELINE STRATEGY:
        - Entry: Price > MA-50 AND (no earnings OR positive sentiment)
        - Exit: Price < MA-50 OR stop-loss at 20%
        """
        price = analytics.get('close', 0)
        ma_50 = analytics.get('ma_50', 0)
        has_position = ticker in portfolio_state.get('positions', {})

        sentiment = self.llm_analysis(ticker, transcript, date)

        if has_position:
            position = portfolio_state['positions'][ticker]
            buy_price = position['buy_price']

            # STOP-LOSS: Cut losses at 20% to prevent catastrophic losses
            if buy_price > 0:
                pnl_pct = (price - buy_price) / buy_price
                if pnl_pct < -0.20:
                    return 'SELL'

            # Normal exit: Price below MA-50
            return 'SELL' if price < ma_50 else 'HOLD'
        else:
            # Entry: Price above MA-50 AND (no earnings OR positive sentiment)
            if price > ma_50:
                if sentiment is None or sentiment['sentiment'] == 'positive':
                    return 'BUY'
        return 'HOLD'

    def _build_analytics_lookup(self, analytics_df):
        lookup = defaultdict(list)
        for _, row in analytics_df.iterrows():
            lookup[row['ticker']].append((row['date'], row.to_dict()))
        for ticker in lookup:
            lookup[ticker].sort(key=lambda x: x[0])
        return lookup

    def evaluate(self, verbose=False):
        """Evaluate strategy on previously set data."""
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        print("Running evaluation...")

        # Calculate analytics
        analytics = self.calculate_analytics(self.prices)
        analytics_lookup = self._build_analytics_lookup(analytics)

        # Run backtest
        print("Running backtest simulation...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        results = sim.run(
            lambda t, d, tr, ps, a: self.make_decision(t, d, tr, ps, a),
            analytics_lookup, verbose
        )

        return results

print("BaseStrategy class loaded")

In [ ]:
# ============================================================
# CELL 10: Evaluation Helper Function
# ============================================================

def run_evaluation(baseline_strategy=None, enhanced_strategy=None, strategy='baseline', split='val'):
    """
    Helper function to load data and evaluate strategies.

    Args:
        baseline_strategy: BaseStrategy instance (required if strategy='baseline')
        enhanced_strategy: EnhancedStrategy instance (required if strategy='enhanced')
        strategy: 'baseline' or 'enhanced'
        split: 'dev', 'val', or 'test'

    Returns:
        results: Dictionary with trades, portfolio_history, final_portfolio
    """
    if strategy == 'baseline':
        if baseline_strategy is None:
            raise ValueError("baseline_strategy must be provided when strategy='baseline'")
        selected_strategy = baseline_strategy
    elif strategy == 'enhanced':
        if enhanced_strategy is None:
            raise ValueError("enhanced_strategy must be provided when strategy='enhanced'")
        selected_strategy = enhanced_strategy
    else:
        raise ValueError(f"strategy must be 'baseline' or 'enhanced', got '{strategy}'")

    # Load data
    print(f"Loading {split.upper()} split data...")
    prices = load_prices(split)
    earnings = load_earnings(split)

    # Set data and evaluate
    selected_strategy.set_data(prices, earnings)
    results = selected_strategy.evaluate()

    return results

print("Evaluation helper function loaded")


In [ ]:
# ============================================================
# CELL 11: Baseline Strategy Evaluation
# ============================================================
#
# This cell establishes the baseline performance on the validation split.
#
# IMPORTANT: Data Split Explanation
# --------------------------------
# - **Dev Split (2000-2019)**: For experimentation and hyperparameter tuning
#   - Use this split to iterate on your strategy
#   - Test different RSI periods, stop-loss levels, signal thresholds, etc.
#
# - **Val Split (2020-2024)**: For final performance reporting
#   - This is the "test" set for comparing strategies
#   - Do NOT tune hyperparameters on this split
#   - Report your final metrics on this split
#
# Workflow:
#   1. Run baseline on val split (establishes reference performance)
#   2. Develop your enhanced strategy, tune on dev split
#   3. Run final comparison on val split

print("\n" + "="*70)
print("BASELINE STRATEGY EVALUATION")
print("="*70)

# Create baseline strategy instance (no data loaded yet)
baseline = BaseStrategy(finbert_pipeline)

# Evaluate on Validation split (final performance reporting)
print("\n[VAL SPLIT - Final Performance]")
results_baseline_val = run_evaluation(baseline_strategy=baseline, strategy='baseline', split='val')

metrics_baseline_val = calculate_metrics(results_baseline_val)
BASELINE_METRICS_VAL = {
    'return': metrics_baseline_val['total_return'],
    'sharpe': metrics_baseline_val['sharpe_ratio'],
    'drawdown': metrics_baseline_val['max_drawdown'],
    'win_rate': metrics_baseline_val['win_rate'],
    'volatility': metrics_baseline_val['volatility'],
    'trades': metrics_baseline_val['num_trades']
}

print(f"Return: {BASELINE_METRICS_VAL['return']:.2%}")
print(f"Sharpe: {BASELINE_METRICS_VAL['sharpe']:.2f}")
print(f"Max Drawdown: {BASELINE_METRICS_VAL['drawdown']:.2%}")
print(f"Win Rate: {BASELINE_METRICS_VAL['win_rate']:.1%}")
print(f"Volatility: {BASELINE_METRICS_VAL['volatility']:.2%}")
print(f"Total Trades: {BASELINE_METRICS_VAL['trades']:,}")

print("\n" + "="*70)
print("BASELINE METRICS STORED")
print("="*70)
print("Baseline metrics saved to BASELINE_METRICS_VAL for comparison.")
print("="*70)

# Visualization
plot_results(results_baseline_val, metrics_baseline_val, title="Baseline Strategy - Validation Split")

# ============================================================
# OPTIONAL: Dev Split Evaluation (For Experimentation)
# ============================================================
# Uncomment below to evaluate baseline on dev split for experimentation
#
print("\n[DEV SPLIT - For Experimentation]")
results_baseline_dev = run_evaluation(baseline_strategy=baseline, strategy='baseline', split='dev')
metrics_baseline_dev = calculate_metrics(results_baseline_dev)
print(f"Return: {metrics_baseline_dev['total_return']:.2%}")
print(f"Sharpe: {metrics_baseline_dev['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {metrics_baseline_dev['max_drawdown']:.2%}")

# Visualization
plot_results(results_baseline_dev, metrics_baseline_dev, title="Baseline Strategy - Dev Split")

---

## Exploration Areas

Work on all of these to improve upon the baseline and add the name of the key contributor of each area

#### Area 1: Enhanced EDA

Add more visualizations: correlation heatmaps, time series decomposition, word frequency analysis, volume-price relationships.

```python
# Example ideas:
# - Correlation heatmap of price features
# - Per-ticker performance comparison  
# - Earnings transcript word frequency
# - Sector analysis (if metadata available)
```

### Area 2: Data Quality & Cleaning

Override `clean_data()` to handle missing prices, filter outliers, remove tickers with insufficient data.

```python
def clean_data(self, prices_df, earnings_df):
    prices = prices_df.copy().drop_duplicates().sort_values(['ticker', 'date'])
    # Forward-fill missing prices by ticker
    # Filter tickers with minimum data points
    # Remove outliers
    return prices, earnings
```

### Area 3: More Technical Indicators

Override `calculate_analytics()` to add RSI, MACD, Bollinger Bands, ATR, Stochastic.

**Performance Consideration**: Computing indicators across 400+ tickers and 10+ years of data can be slow with iterative pandas operations. Think about how you might parallelize these computations or use more efficient approaches.

```python
def calculate_analytics(self, prices_df):
    # Add RSI: 14-period relative strength
    # Add MACD: EMA-12, EMA-26, signal line  
    # Add Bollinger Bands: 20-day ±2 std dev
    return analytics_df
```

### Area 4: Enhanced LLM Analysis

Override `llm_analysis()` to use relevant text to improve confidence, calculate sentiment strength, weight by confidence.

```python
def llm_analysis(self, ticker, transcript, date):
    # Use relevant text
    # Calculate sentiment strength (positive - negative)
    # Weight decisions by confidence
    return result
```

### Area 5: Smarter Decision Logic

Override `make_decision()` for multi-signal confirmation, position sizing, stop-loss, risk management.

```python
def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
    # Combine multiple indicators (RSI, MACD, sentiment)
    # Add confidence-based position sizing
    # Implement stop-loss / take-profit
    return decision
```

---

## Additional Exploratory Data Analysis

Before implementing your enhanced strategy, perform additional EDA to gain insights into the data that may inform your strategy design.

Consider exploring:
- Correlation analysis between price features
- Sector or industry patterns (if metadata available)
- Earnings transcript sentiment distribution
- Volume-price relationships
- Time series patterns or seasonality

Run the cell below for additional visualizations.

## Important
Mention the name of the key contributor of each area

# EDA for 20MA Crossover + Lourvain Cluster Strategy

In [ ]:
# ============================================================
# CELL 12: Additional EDA  (VECTORISED)
# ============================================================
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.diagnostic import acorr_ljungbox
import networkx as nx
from tqdm import tqdm
import re

# ============================================================
# Earnings Call Transcript Availability
# ============================================================
print("\n[EARNINGS TRANSCRIPTS] Listing all available transcripts sorted by ticker and date...")

earnings_list = earnings_dev[['ticker', 'date', 'quarter']].copy()
earnings_list['date'] = pd.to_datetime(earnings_list['date']).dt.tz_localize(None)
earnings_list = earnings_list.sort_values(['ticker', 'date']).reset_index(drop=True)
earnings_list['transcript_length'] = earnings_dev['transcript'].str.len().values

print(f"\nTotal transcripts available: {len(earnings_list):,}")
print(f"Unique tickers with transcripts: {earnings_list['ticker'].nunique()}")
print(f"Date range: {earnings_list['date'].min().date()} to {earnings_list['date'].max().date()}")

print("\n" + "="*70)
print(f"{'#':<6} {'Ticker':<10} {'Date':<14} {'Quarter':<12} {'Length':>10}")
print("="*70)

for i, row in earnings_list.iterrows():
    print(f"{i+1:<6} {row['ticker']:<10} {str(row['date'].date()):<14} {str(row.get('quarter', 'N/A')):<12} {row['transcript_length']:>10,}")

print("="*70)

# Summary per ticker
print("\n[PER-TICKER SUMMARY]")
summary = earnings_list.groupby('ticker').agg(
    count=('date', 'count'),
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    avg_length=('transcript_length', 'mean')
).reset_index().sort_values('ticker')

print(f"\n{'Ticker':<10} {'Count':>6} {'First Date':<14} {'Last Date':<14} {'Avg Length':>12}")
print("-"*60)
for _, row in summary.iterrows():
    print(f"{row['ticker']:<10} {row['count']:>6} {str(row['first_date'].date()):<14} {str(row['last_date'].date()):<14} {row['avg_length']:>12,.0f}")


# ============================================================
# Interactive Price Time Series — All Tickers (vectorised normalisation)
# ============================================================
print("\n[PRICE TIME SERIES] Plotting normalised interactive price chart for all tickers...")

prices_ts = prices_dev.copy()
prices_ts['date'] = pd.to_datetime(prices_ts['date']).dt.tz_localize(None)
prices_ts = prices_ts.sort_values(['ticker', 'date'])

# Vectorised normalisation: divide by first close per ticker
prices_ts['first_close'] = prices_ts.groupby('ticker')['close'].transform('first')
prices_ts['normalised'] = (prices_ts['close'] / prices_ts['first_close']) * 100
prices_ts = prices_ts[prices_ts['first_close'] > 0]

all_tickers = sorted(prices_ts['ticker'].unique())

fig = go.Figure()

for ticker in all_tickers:
    mask = prices_ts['ticker'] == ticker
    tdf = prices_ts.loc[mask]
    fig.add_trace(go.Scattergl(
        x=tdf['date'],
        y=tdf['normalised'],
        mode='lines',
        name=ticker,
        line=dict(width=1),
        visible='legendonly',
        hovertemplate=(
            f'<b>{ticker}</b><br>'
            'Date: %{x|%Y-%m-%d}<br>'
            'Indexed: %{y:.1f}<br>'
            'Close: $%{customdata:.2f}'
            '<extra></extra>'
        ),
        customdata=tdf['close'].values
    ))

fig.add_hline(y=100, line=dict(color='black', dash='dash', width=1),
              annotation_text='Base (100)', annotation_position='bottom right')

fig.update_layout(
    title='S&P 500 Normalised Price Paths (Dev Split) — Base 100 at first trading day',
    xaxis_title='Date', yaxis_title='Normalised Price (Base = 100)',
    height=650, width=1400, hovermode='closest',
    legend=dict(orientation='v', x=1.01, y=1, font=dict(size=9),
                itemclick='toggle', itemdoubleclick='toggleothers'),
    xaxis=dict(
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=1, label='1Y', step='year', stepmode='backward'),
            dict(count=3, label='3Y', step='year', stepmode='backward'),
            dict(count=5, label='5Y', step='year', stepmode='backward'),
            dict(count=10, label='10Y', step='year', stepmode='backward'),
            dict(step='all', label='All')
        ])
    ),
    yaxis=dict(fixedrange=False),
    updatemenus=[dict(type='buttons', direction='left', x=0.0, y=1.12, buttons=[
        dict(label='Show All', method='restyle', args=[{'visible': True}]),
        dict(label='Hide All', method='restyle', args=[{'visible': 'legendonly'}])
    ])]
)
fig.show()
print(f"Normalised chart rendered with {len(all_tickers)} tickers.")

# --------------- Helpers ---------------
prices_eda = prices_dev.copy()
prices_eda['date'] = pd.to_datetime(prices_eda['date']).dt.tz_localize(None)
prices_eda = prices_eda.sort_values(['ticker', 'date'])

earnings_eda = earnings_dev.copy()
earnings_eda['date'] = pd.to_datetime(earnings_eda['date']).dt.tz_localize(None)

top30 = prices_eda['ticker'].value_counts().head(30).index.tolist()
print(f"Top 30 tickers for EDA: {top30}")

# Vectorised log returns
prices_eda['log_return'] = prices_eda.groupby('ticker')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

# ============================================================
# DATA AVAILABILITY — SINGLE TRACE (vectorised)
# ============================================================
print("\n[1] Plotting data availability for all tickers...")

all_tickers_sorted = sorted(prices_eda['ticker'].unique())

fig = go.Figure()
# >>> VECTORISED: one single Scattergl trace instead of ~400 <<<
fig.add_trace(go.Scattergl(
    x=prices_eda['date'].values,
    y=prices_eda['ticker'].values,
    mode='markers',
    marker=dict(size=1.5, color='steelblue', opacity=0.6),
    showlegend=False,
    hovertemplate='%{y}<br>Date: %{x}<extra></extra>'
))

fig.update_layout(
    title='Price Data Availability by Ticker (gaps indicate delisting / missing data)',
    xaxis_title='Date', yaxis_title='Ticker',
    height=max(600, len(all_tickers_sorted) * 8), width=1200,
    yaxis=dict(tickfont=dict(size=6), categoryorder='array',
               categoryarray=all_tickers_sorted)
)
fig.show()

# ============================================================
# LJUNG-BOX EFFICIENCY TEST — VECTORISED via groupby
# ============================================================
print("\n[EDA] Computing Ljung-Box test (lag=20) on log returns for all tickers...")

LAG_TO_TEST = 20

lb_results = {}
for ticker, group in prices_eda.groupby('ticker')['log_return']:
    lr = group.dropna()
    if len(lr) < 50:
        continue
    try:
        res = acorr_ljungbox(lr, lags=[LAG_TO_TEST], return_df=True)
        lb_results[ticker] = {
            'lb_pvalue': res['lb_pvalue'].iloc[0],
            'lb_stat':   res['lb_stat'].iloc[0]
        }
    except Exception:
        continue

lb_df = (pd.DataFrame.from_dict(lb_results, orient='index')
         .rename_axis('ticker')
         .reset_index()
         .sort_values('lb_pvalue')
         .reset_index(drop=True))

alpha = 0.05
n_total = len(lb_df)
n_inefficient = (lb_df['lb_pvalue'] < alpha).sum()
n_efficient = n_total - n_inefficient

print(f"\nTotal tickers tested: {n_total}")
print(f"Inefficient (p < {alpha}): {n_inefficient}  ({n_inefficient/n_total:.1%})")
print(f"Efficient   (p >= {alpha}): {n_efficient}  ({n_efficient/n_total:.1%})")

colors = np.where(lb_df['lb_pvalue'] < alpha, '#E15759', '#76B7B2')

fig_lb = go.Figure()
fig_lb.add_trace(go.Bar(
    x=lb_df['ticker'], y=lb_df['lb_pvalue'], marker_color=colors, showlegend=False,
    hovertemplate='<b>%{x}</b><br>Ljung-Box p-value: %{y:.4e}<br>Q-Stat: %{customdata:.2f}<extra></extra>',
    customdata=lb_df['lb_stat']
))
fig_lb.add_hline(y=alpha, line=dict(color='black', width=2, dash='dash'),
                 annotation_text=f'α = {alpha}', annotation_position='top right',
                 annotation_font=dict(size=14, color='black'))
fig_lb.add_trace(go.Bar(x=[None], y=[None], marker_color='#E15759',
                        name=f'Inefficient / Tradable p < {alpha}  (n={n_inefficient})', showlegend=True))
fig_lb.add_trace(go.Bar(x=[None], y=[None], marker_color='#76B7B2',
                        name=f'Efficient / Random Walk p ≥ {alpha}  (n={n_efficient})', showlegend=True))
fig_lb.update_layout(
    title=(f'Ljung-Box Test for Autocorrelation — {n_total} Tickers<br>'
           f'<sup>Hypothesis: Stocks below the dashed line (p < {alpha}) exhibit serial correlation and reject the random walk.</sup>'),
    xaxis_title='Ticker (sorted by p-value)',
    yaxis_title=f'Ljung-Box p-value (Lag = {LAG_TO_TEST})',
    height=600, width=1400,
    xaxis=dict(tickfont=dict(size=6), tickangle=90,
               categoryorder='array', categoryarray=lb_df['ticker'].tolist()),
    yaxis=dict(type='log', title='p-value (Log Scale)'),
    legend=dict(x=0.01, y=0.98, font=dict(size=12),
                bgcolor='rgba(255,255,255,0.9)', bordercolor='black', borderwidth=1),
    plot_bgcolor='white', hovermode='closest'
)
fig_lb.show()

LB_INEFFICIENT_TICKERS = set(lb_df[lb_df['lb_pvalue'] < alpha]['ticker'].tolist())
print(f"\nStored as LB_INEFFICIENT_TICKERS (set of {len(LB_INEFFICIENT_TICKERS)} tickers)")

INEFFICIENT_TICKERS = LB_INEFFICIENT_TICKERS
print(f"\nUsing INEFFICIENT_TICKERS from Ljung-Box (set of {len(INEFFICIENT_TICKERS)} tickers)")

# ============================================================
# CORRELATION MATRIX HEATMAP — INEFFICIENT TICKERS ONLY
# ============================================================
print("\n[9] Computing return correlation matrix for INEFFICIENT tickers only...")

returns_pivot = prices_eda.pivot_table(index='date', columns='ticker', values='log_return')
min_obs = 252
valid_tickers = returns_pivot.columns[returns_pivot.notna().sum() >= min_obs]
returns_pivot = returns_pivot[valid_tickers]

inefficient_valid = [t for t in valid_tickers if t in INEFFICIENT_TICKERS]
returns_pivot = returns_pivot[inefficient_valid]
corr_matrix = returns_pivot.corr()

print(f"Filtered from {len(valid_tickers)} → {len(inefficient_valid)} inefficient tickers")

# ================================================================
# HIERARCHICAL CLUSTERING (Ward)
# ================================================================
print("\n[10] Hierarchical clustering on inefficient ticker correlations...")

corr_clean = corr_matrix.copy()
corr_clean_vals = corr_clean.to_numpy(copy=True, dtype=float)
np.fill_diagonal(corr_clean_vals, 1.0)
corr_clean = pd.DataFrame(corr_clean_vals, index=corr_clean.index, columns=corr_clean.columns)
corr_clean = corr_clean.clip(-1, 1)
dist_matrix = 1 - corr_clean
dist_condensed = squareform(dist_matrix.values, checks=False)
Z = linkage(dist_condensed, method='ward')

dendro_data = dendrogram(Z, labels=corr_matrix.columns.tolist(),
                         no_plot=True, color_threshold=0.7 * max(Z[:, 2]))

for n_clusters in [5, 10, 15]:
    labels = fcluster(Z, t=n_clusters, criterion='maxclust')
    cluster_df = pd.DataFrame({'ticker': corr_matrix.columns, 'cluster': labels})
    print(f"\n--- {n_clusters} Clusters ---")
    for c in sorted(cluster_df['cluster'].unique()):
        members = cluster_df[cluster_df['cluster'] == c]['ticker'].tolist()
        print(f"  Cluster {c} ({len(members):>3} tickers): {', '.join(members[:15])}"
              f"{'...' if len(members) > 15 else ''}")

# Reordered heatmap
print("\nPlotting cluster-ordered correlation heatmap (inefficient tickers only)...")
reorder_idx = dendro_data['leaves']
reordered_corr = corr_matrix.iloc[reorder_idx, reorder_idx]

fig_heatmap = go.Figure(data=go.Heatmap(
    z=reordered_corr.values,
    x=reordered_corr.columns.tolist(),
    y=reordered_corr.index.tolist(),
    colorscale='RdBu_r', zmid=0, zmin=-1, zmax=1,
    colorbar=dict(title='ρ'),
    hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>'
))
fig_heatmap.update_layout(
    title=f'Correlation Heatmap — Inefficient Tickers Only ({len(inefficient_valid)}) — Ordered by Hierarchical Clustering',
    height=950, width=1000,
    xaxis=dict(tickfont=dict(size=6), tickangle=90),
    yaxis=dict(tickfont=dict(size=6), autorange='reversed')
)
fig_heatmap.show()

# ================================================================
# PRE-COMPUTE AVERAGE DAILY VOLUME
# ================================================================
print("\nComputing average daily volume per ticker for leader identification...")
avg_volume = (
    prices_eda[prices_eda['ticker'].isin(INEFFICIENT_TICKERS)]
    .groupby('ticker')['volume'].mean().to_dict()
)

# ================================================================
# LOUVAIN COMMUNITY DETECTION — VECTORISED edge building
# ================================================================
print("\n[11] Louvain community detection (INEFFICIENT tickers only, ρ ≥ 0.6)...")

CORR_THRESHOLD = 0.6

G = nx.Graph()
tickers_list = corr_matrix.columns.tolist()
G.add_nodes_from(tickers_list)

# >>> VECTORISED: NumPy upper-triangle mask instead of O(n²) Python loop <<<
corr_vals = corr_matrix.values
row_idx, col_idx = np.triu_indices(len(tickers_list), k=1)
rho_vals = corr_vals[row_idx, col_idx]
edge_mask = rho_vals >= CORR_THRESHOLD

edges = [
    (tickers_list[r], tickers_list[c], {'weight': float(rho)})
    for r, c, rho in zip(row_idx[edge_mask], col_idx[edge_mask], rho_vals[edge_mask])
]
G.add_edges_from(edges)
edge_count = len(edges)

print(f"Graph: {G.number_of_nodes()} nodes, {edge_count} edges (ρ ≥ {CORR_THRESHOLD})")

# Remove isolated nodes
isolated = [n for n in G.nodes() if G.degree(n) == 0]
G_connected = G.copy()
G_connected.remove_nodes_from(isolated)
print(f"After removing {len(isolated)} isolated nodes: {G_connected.number_of_nodes()} nodes, {G_connected.number_of_edges()} edges")

# Louvain communities
communities = nx.community.louvain_communities(G_connected, weight='weight', seed=42)
print(f"Detected {len(communities)} communities")

# Leader of each community by average volume
community_leaders = {}
for idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    members = sorted(comm)
    leader = max(members, key=lambda t: avg_volume.get(t, 0))
    leader_vol = avg_volume.get(leader, 0)
    community_leaders[idx] = leader
    print(f"  Community {idx} ({len(members):>3} tickers) | "
          f"Leader: {leader} (avg vol: {leader_vol:,.0f}) | "
          f"Members: {', '.join(members[:18])}"
          f"{'...' if len(members) > 18 else ''}")

# ---------- Interactive network visualisation ----------
print("\nComputing network layout (spring layout)...")

pos = nx.spring_layout(G_connected, k=1.5 / np.sqrt(G_connected.number_of_nodes()),
                        iterations=80, seed=42, weight='weight')

palette = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
community_colors = {i: palette[i % len(palette)] for i in range(len(communities))}

# Edge traces (already vectorised — list concat)
edge_x, edge_y = [], []
for u, v, data in G_connected.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

fig_net = go.Figure()
fig_net.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.3, color='#cccccc'),
    hoverinfo='none', showlegend=False
))

leader_set = set(community_leaders.values())

for comm_idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    comm_nodes = sorted(comm)
    leader = community_leaders[comm_idx]

    regular = [n for n in comm_nodes if n != leader]
    if regular:
        node_x = [pos[n][0] for n in regular]
        node_y = [pos[n][1] for n in regular]
        node_sizes = [5 + G_connected.degree(n) * 1.5 for n in regular]
        hover_texts = [
            f"<b>{n}</b><br>Community: {comm_idx}<br>"
            f"Degree: {G_connected.degree(n)}<br>"
            f"Avg Volume: {avg_volume.get(n, 0):,.0f}<br>"
            f"Connections: {', '.join(sorted(G_connected.neighbors(n)))}"
            for n in regular
        ]
        fig_net.add_trace(go.Scatter(
            x=node_x, y=node_y, mode='markers+text',
            marker=dict(size=node_sizes, color=community_colors[comm_idx],
                        line=dict(width=0.5, color='white'), symbol='circle'),
            text=regular, textposition='top center', textfont=dict(size=7),
            hovertext=hover_texts, hoverinfo='text',
            name=f'Community {comm_idx} ({len(comm_nodes)})', showlegend=True
        ))

    fig_net.add_trace(go.Scatter(
        x=[pos[leader][0]], y=[pos[leader][1]], mode='markers+text',
        marker=dict(size=20 + G_connected.degree(leader) * 2,
                    color=community_colors[comm_idx],
                    line=dict(width=2, color='black'), symbol='star'),
        text=[f'★ {leader}'], textposition='top center',
        textfont=dict(size=10, color='black'),
        hovertext=[
            f"<b>★ LEADER: {leader}</b><br>Community: {comm_idx}<br>"
            f"Degree: {G_connected.degree(leader)}<br>"
            f"Avg Volume: {avg_volume.get(leader, 0):,.0f}<br>"
            f"Connections: {', '.join(sorted(G_connected.neighbors(leader)))}"
        ],
        hoverinfo='text', name=f'★ Leader: {leader}', showlegend=True
    ))

fig_net.update_layout(
    title=f'Louvain Community Network — INEFFICIENT Tickers Only (ρ ≥ {CORR_THRESHOLD})<br>'
          f'<sub>{G_connected.number_of_nodes()} stocks, {G_connected.number_of_edges()} edges, '
          f'{len(communities)} communities — ★ = Leader by Avg Daily Volume</sub>',
    height=850, width=1200,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    hovermode='closest',
    legend=dict(title='Communities & Leaders', font=dict(size=10),
                itemclick='toggle', itemdoubleclick='toggleothers'),
    plot_bgcolor='white'
)
fig_net.show()

# ---------- Community statistics — VECTORISED intra-ρ ----------
print("\n[COMMUNITY STATISTICS — INEFFICIENT TICKERS ONLY]")
print(f"{'Comm':<6} {'Size':>5} {'Leader':<8} {'Leader Vol':>14} {'Avg Degree':>11} {'Avg Intra-ρ':>12} {'Density':>9}")
print("-" * 75)

for comm_idx, comm in enumerate(sorted(communities, key=len, reverse=True)):
    comm_nodes = sorted(comm)
    leader = community_leaders[comm_idx]
    leader_vol = avg_volume.get(leader, 0)
    subG = G_connected.subgraph(comm_nodes)
    avg_deg = np.mean([G_connected.degree(n) for n in comm_nodes])

    if len(comm_nodes) > 1:
        # >>> VECTORISED: NumPy submatrix + triu_indices instead of nested loop <<<
        valid_members = [t for t in comm_nodes if t in corr_matrix.columns]
        if len(valid_members) > 1:
            sub_corr = corr_matrix.loc[valid_members, valid_members].values
            tri_r, tri_c = np.triu_indices(len(valid_members), k=1)
            avg_intra = float(np.mean(sub_corr[tri_r, tri_c]))
        else:
            avg_intra = 0
        density = nx.density(subG)
    else:
        avg_intra = 1.0
        density = 0.0

    print(f"{comm_idx:<6} {len(comm_nodes):>5} {leader:<8} {leader_vol:>14,.0f} {avg_deg:>11.1f} {avg_intra:>12.3f} {density:>9.3f}")

print(f"\nIsolated inefficient tickers (ρ < {CORR_THRESHOLD} with all others): {len(isolated)}")
if isolated:
    print(f"  {', '.join(sorted(isolated)[:30])}{'...' if len(isolated) > 30 else ''}")

COMMUNITY_LEADERS = community_leaders
print(f"\nStored COMMUNITY_LEADERS: {community_leaders}")

# ============================================================
# DAILY VOLATILITY — VECTORISED earnings matching via merge_asof
# ============================================================
print("\n[12] Daily Annualised Volatility (non-rolling) for Inefficient Community Leaders...")

leader_tickers_list = sorted(set(COMMUNITY_LEADERS.values()))
print(f"Leaders to plot: {leader_tickers_list}")

vol_summary = []

for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]
    color = palette[comm_idx % len(palette)]

    tdf = prices_eda[prices_eda['ticker'] == ticker].copy().sort_values('date')
    tdf['log_ret'] = np.log(tdf['close'] / tdf['close'].shift(1))

    ann_vol = tdf['log_ret'].std() * np.sqrt(252)
    daily_std = tdf['log_ret'].std()

    fig = go.Figure()

    # >>> VECTORISED: np.where instead of list comp for bar colours <<<
    bar_colors = np.where(tdf['log_ret'].fillna(0) >= 0, color, '#DC3232')

    fig.add_trace(go.Bar(
        x=tdf['date'], y=tdf['log_ret'],
        marker_color=bar_colors, marker_line_width=0,
        name='Log Return',
        hovertemplate='Date: %{x|%Y-%m-%d}<br>Log Ret: %{y:.4f}<extra></extra>'
    ))

    for mult, band_color, dash in [
        (2, 'cornflowerblue', 'dash'),
        (3, 'green', 'dot'),
    ]:
        fig.add_hline(y= mult * daily_std, line=dict(color=band_color, width=1.2, dash=dash),
                      annotation_text=f'+{mult}σ', annotation_position='top right',
                      annotation_font=dict(size=9, color=band_color))
        fig.add_hline(y=-mult * daily_std, line=dict(color=band_color, width=1.2, dash=dash),
                      annotation_text=f'-{mult}σ', annotation_position='bottom right',
                      annotation_font=dict(size=9, color=band_color))

    fig.add_hline(y=0, line=dict(color='grey', width=0.8, dash='dot'))

    # >>> VECTORISED: merge_asof for earnings date matching <<<
    ticker_earnings = earnings_eda[earnings_eda['ticker'] == ticker].copy().sort_values('date')

    if len(ticker_earnings) > 0:
        tdf_sorted = tdf[['date', 'log_ret']].sort_values('date')
        merged = pd.merge_asof(
            ticker_earnings[['date', 'quarter']].sort_values('date'),
            tdf_sorted,
            on='date',
            direction='nearest'
        )
        merged = merged.dropna(subset=['log_ret'])

        if len(merged) > 0:
            fig.add_trace(go.Scatter(
                x=merged['date'], y=merged['log_ret'],
                mode='markers',
                marker=dict(symbol='diamond', size=10, color='yellow',
                            line=dict(color='black', width=1.5)),
                name='Earnings Call',
                customdata=merged['quarter'].astype(str),
                hovertemplate=(
                    '<b>Earnings Call</b><br>'
                    'Date: %{x|%Y-%m-%d}<br>'
                    'Log Ret: %{y:.4f}<br>'
                    'Quarter: %{customdata}<extra></extra>'
                )
            ))

    fig.update_layout(
        title=(f'★ {ticker} (Community {comm_idx}) — Daily Log Returns  '
               f'| Annualised Vol: {ann_vol:.1%}  | ±2σ / ±3σ (full-period constant)'
               f'  | ◆ = Earnings Call'),
        xaxis_title='Date', yaxis_title='Log Return',
        height=500, width=1400, hovermode='x unified',
        plot_bgcolor='white', bargap=0,
        legend=dict(orientation='h', y=-0.15, font=dict(size=10))
    )
    fig.show()

    vol_summary.append({
        'ticker': ticker, 'community': comm_idx,
        'ann_vol': ann_vol, 'daily_std': daily_std,
        'n_obs': tdf['log_ret'].notna().sum()
    })

print(f"\n{'='*70}")
print(f"{'Leader':<8} {'Comm':>5} {'Ann. Vol':>12} {'Daily σ':>12} {'N Obs':>10}")
print(f"{'-'*70}")
for s in sorted(vol_summary, key=lambda x: x['ann_vol'], reverse=True):
    print(f"{s['ticker']:<8} {s['community']:>5} {s['ann_vol']:>12.1%} "
          f"{s['daily_std']:>12.4f} {s['n_obs']:>10,}")
print(f"{'='*70}")

# ============================================================
# ROLLING HURST EXPONENT (60-day) vs ROLLING VOLATILITY (20-day)
# ============================================================
print("\n[15] Rolling Hurst (252-day) vs Rolling Volatility (20-day) — Community Leaders...")

HURST_WINDOW = 60
VOL_WINDOW   = 20

def compute_hurst_series(log_ret_series, window=252, min_lag=2, max_lag=50):
    """Rolling Hurst exponent via R/S analysis."""
    values = pd.to_numeric(log_ret_series, errors='coerce').values.astype(np.float64)
    n = len(values)
    result = np.full(n, np.nan)

    lags = np.unique(
        np.logspace(np.log10(min_lag), np.log10(max_lag), num=20).astype(int)
    )

    for i in range(window - 1, n):
        ts = values[i - window + 1 : i + 1]
        ts = ts[~np.isnan(ts)]
        if len(ts) < window * 0.8:
            continue

        rs_vals, valid_lags = [], []
        for lag in lags:
            if lag >= len(ts) // 2:
                continue
            chunks = [ts[j: j + lag] for j in range(0, len(ts) - lag, lag)]
            rs_chunk = []
            for chunk in chunks:
                chunk = chunk.astype(np.float64)
                s = np.std(chunk, ddof=1)
                if s > 0:
                    dev = np.cumsum(chunk - chunk.mean())
                    rs_chunk.append(float(dev.max() - dev.min()) / float(s))
            if rs_chunk:
                rs_vals.append(float(np.mean(rs_chunk)))
                valid_lags.append(int(lag))

        if len(valid_lags) >= 5:
            slope, *_ = np.polyfit(
                np.log(np.array(valid_lags, dtype=np.float64)),
                np.log(np.array(rs_vals,   dtype=np.float64)),
                1
            )
            result[i] = float(slope)

    return pd.Series(result, index=log_ret_series.index)


for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]
    color    = palette[comm_idx % len(palette)]

    tdf = (prices_eda[prices_eda['ticker'] == ticker]
           .copy().sort_values('date').reset_index(drop=True))
    tdf['log_ret'] = np.log(tdf['close'] / tdf['close'].shift(1))

    print(f"  Computing rolling Hurst for {ticker}...")
    tdf['rolling_hurst'] = compute_hurst_series(tdf['log_ret'], window=HURST_WINDOW)
    tdf['rolling_vol'] = tdf['log_ret'].rolling(VOL_WINDOW).std() * np.sqrt(252)

    ticker_earnings = earnings_eda[earnings_eda['ticker'] == ticker].sort_values('date')

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=tdf['date'], y=tdf['rolling_vol'], mode='lines',
        line=dict(color='rgba(220,80,80,0.85)', width=1),
        fill='tozeroy', fillcolor='rgba(220,80,80,0.12)',
        name=f'{VOL_WINDOW}-day Rolling Vol (ann.)',
        hovertemplate='Date: %{x|%Y-%m-%d}<br>Vol: %{y:.2%}<extra>Vol</extra>'
    ), secondary_y=True)

    fig.add_trace(go.Scatter(
        x=tdf['date'], y=tdf['rolling_hurst'], mode='lines',
        line=dict(color=color, width=2),
        name=f'{HURST_WINDOW}-day Rolling Hurst (H)',
        hovertemplate='Date: %{x|%Y-%m-%d}<br>H: %{y:.3f}<extra>Hurst</extra>'
    ), secondary_y=False)

    for h_val, label, dash, ann_color in [
        (0.5, 'H=0.5 Random Walk', 'dash', 'black'),
        (0.45, 'H=0.45 MR threshold', 'dot', 'cornflowerblue'),
        (0.55, 'H=0.55 Trend threshold', 'dot', 'tomato'),
    ]:
        fig.add_hline(y=h_val, secondary_y=False,
                      line=dict(color=ann_color, width=1, dash=dash),
                      annotation_text=label, annotation_position='top right',
                      annotation_font=dict(size=9, color=ann_color))

    fig.add_hrect(y0=0, y1=0.45, secondary_y=False,
                  fillcolor='rgba(100,149,237,0.05)', line_width=0)
    fig.add_hrect(y0=0.55, y1=1.0, secondary_y=False,
                  fillcolor='rgba(220,80,80,0.05)', line_width=0)

    # >>> Earnings vlines via merge_asof for nearest-date snapping <<<
    tdf_idx = tdf.set_index('date')
    for _, erow in ticker_earnings.iterrows():
        edate = erow['date']
        if edate not in tdf_idx.index:
            pos_val = tdf_idx.index.searchsorted(edate)
            if pos_val >= len(tdf_idx):
                continue
            edate = tdf_idx.index[pos_val]
        x_val = pd.to_datetime(edate).timestamp() * 1000
        fig.add_vline(x=x_val, line=dict(color='gold', width=1.2, dash='dot'),
                      annotation_text='◆', annotation_position='top',
                      annotation_font=dict(size=9, color='goldenrod'))

    fig.update_yaxes(title_text='Hurst Exponent (H)', range=[0.15, 0.85],
                     showgrid=True, gridcolor='rgba(200,200,200,0.35)', secondary_y=False)
    fig.update_yaxes(title_text=f'{VOL_WINDOW}-day Annualised Volatility',
                     tickformat='.0%', showgrid=False, secondary_y=True)
    fig.update_xaxes(rangeselector=dict(buttons=[
        dict(count=1, label='1Y', step='year', stepmode='backward'),
        dict(count=3, label='3Y', step='year', stepmode='backward'),
        dict(count=5, label='5Y', step='year', stepmode='backward'),
        dict(step='all', label='All')
    ]))
    fig.update_layout(
        title=(f'★ {ticker} (Community {comm_idx}) — '
               f'{HURST_WINDOW}-day Rolling Hurst vs {VOL_WINDOW}-day Rolling Volatility<br>'
               '<sup>Blue zone = Mean-Reverting · Red zone = Trending · '
               'Gold dashed lines = Earnings Calls</sup>'),
        height=520, width=1400, hovermode='x unified', plot_bgcolor='white',
        legend=dict(orientation='h', y=-0.15, font=dict(size=10))
    )
    fig.show()

# ============================================================
# HALF-LIFE OF INFORMATION DECAY — VECTORISED crossing detection
# ============================================================
print("\n[14] Half-life of information decay per earnings call — Inefficient Community Leaders...")

WINDOW_POST = 60

halflife_summary = []

for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]

    tdf = (prices_eda[prices_eda['ticker'] == ticker]
           .copy().sort_values('date').set_index('date'))

    ticker_earnings = (earnings_eda[earnings_eda['ticker'] == ticker]
                       .sort_values('date'))

    for _, erow in ticker_earnings.iterrows():
        edate = erow['date']
        quarter = erow.get('quarter', 'N/A')

        if edate not in tdf.index:
            idx_pos = tdf.index.searchsorted(edate)
            if idx_pos >= len(tdf.index):
                continue
            edate = tdf.index[idx_pos]

        loc = tdf.index.get_loc(edate)
        if isinstance(loc, slice):
            loc = loc.start
        if loc + WINDOW_POST >= len(tdf):
            continue

        pre_price = tdf.iloc[loc]['close']
        post_slice = tdf.iloc[loc: loc + WINDOW_POST + 1]
        car = (post_slice['close'].values - pre_price) / pre_price

        peak_idx = int(np.argmax(np.abs(car)))
        peak_car = car[peak_idx]
        if abs(peak_car) < 1e-6:
            continue

        # >>> VECTORISED: np.argmax on boolean array instead of inner loop <<<
        target = peak_car * 0.5
        half_life = None
        if peak_idx + 1 < len(car):
            remaining = car[peak_idx + 1:]
            if peak_car > 0:
                crossings = remaining <= target
            else:
                crossings = remaining >= target
            if crossings.any():
                half_life = int(np.argmax(crossings)) + 1  # +1 because offset from peak

        halflife_summary.append({
            'ticker': ticker, 'community': comm_idx,
            'date': edate, 'quarter': quarter,
            'half_life': half_life, 'peak_car': peak_car,
            'color': palette[comm_idx % len(palette)]
        })

hl_df = pd.DataFrame(halflife_summary)
hl_df['half_life_val'] = pd.to_numeric(hl_df['half_life'], errors='coerce')

agg = (hl_df.dropna(subset=['half_life_val'])
       .groupby(['ticker', 'community', 'color'])['half_life_val']
       .agg(mean_hl='mean', median_hl='median', n_calls='count')
       .reset_index()
       .sort_values('mean_hl'))

print(f"\n{'='*75}")
print(f"{'Leader':<8} {'Comm':>5} {'N Calls':>8} {'Mean HL (days)':>16} {'Median HL (days)':>18}")
print(f"{'-'*75}")
for _, row in agg.iterrows():
    print(f"{row['ticker']:<8} {int(row['community']):>5} {int(row['n_calls']):>8} "
          f"{row['mean_hl']:>16.1f} {row['median_hl']:>18.1f}")
print(f"{'='*75}")

# ================================================================
# GROUPED BAR CHART — Mean & Median HL per leader
# ================================================================
fig_hl = go.Figure()

fig_hl.add_trace(go.Bar(
    x=agg['ticker'], y=agg['mean_hl'], name='Mean Half-Life',
    marker=dict(color=agg['color'].tolist(), line=dict(color='black', width=1)),
    text=agg['mean_hl'].round(1), textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>Mean HL: %{y:.1f} days<br>N calls: %{customdata}<extra>Mean</extra>',
    customdata=agg['n_calls']
))
fig_hl.add_trace(go.Bar(
    x=agg['ticker'], y=agg['median_hl'], name='Median Half-Life',
    marker=dict(color=agg['color'].tolist(),
                pattern=dict(shape='/', size=6, solidity=0.4),
                line=dict(color='black', width=1)),
    text=agg['median_hl'].round(1), textposition='outside', textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>Median HL: %{y:.1f} days<br>N calls: %{customdata}<extra>Median</extra>',
    customdata=agg['n_calls']
))

fig_hl.update_layout(
    title=('Information Decay Half-Life per Earnings Call — Inefficient Community Leaders<br>'
           '<sup>Half-life = trading days for post-earnings CAR to retrace 50% of its peak move · '
           f'window = {WINDOW_POST} days · bars sorted by Mean HL · hatched = Median</sup>'),
    xaxis=dict(title='Community Leader (sorted by Mean Half-Life)',
               categoryorder='array', categoryarray=agg['ticker'].tolist(), tickangle=0),
    yaxis=dict(title='Half-Life (Trading Days)', showgrid=True,
               gridcolor='rgba(200,200,200,0.4)', zeroline=True,
               zerolinecolor='black', zerolinewidth=1),
    barmode='group', bargap=0.25, bargroupgap=0.05,
    height=550, width=1200, plot_bgcolor='white',
    legend=dict(orientation='h', y=-0.18, font=dict(size=11)),
    hovermode='x unified'
)
fig_hl.show()

# ================================================================
# PER-CALL SCATTER
# ================================================================
fig_hl_scatter = go.Figure()

for ticker in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == ticker][0]
    color = palette[comm_idx % len(palette)]

    sub = hl_df[(hl_df['ticker'] == ticker) & hl_df['half_life_val'].notna()].copy()
    if sub.empty:
        continue

    fig_hl_scatter.add_trace(go.Scatter(
        x=[ticker] * len(sub), y=sub['half_life_val'], mode='markers',
        marker=dict(size=8, color=color, line=dict(color='black', width=0.8), symbol='circle'),
        name=f'{ticker} (Comm {comm_idx})',
        customdata=list(zip(sub['date'].astype(str), sub['quarter'].astype(str),
                            sub['peak_car'].round(4))),
        hovertemplate=(f'<b>{ticker}</b><br>Date: %{{customdata[0]}}<br>'
                       'Quarter: %{customdata[1]}<br>Peak CAR: %{customdata[2]:.2%}<br>'
                       'Half-Life: %{y:.0f} days<extra></extra>'),
        showlegend=True
    ))

    row_agg = agg[agg['ticker'] == ticker]
    if not row_agg.empty:
        mean_val = row_agg['mean_hl'].values[0]
        median_val = row_agg['median_hl'].values[0]
        for val, label, dash in [(mean_val, 'mean', 'solid'), (median_val, 'median', 'dash')]:
            fig_hl_scatter.add_trace(go.Scatter(
                x=[ticker, ticker], y=[val, val], mode='lines+text',
                line=dict(color=color, width=2.5, dash=dash),
                text=['', f'{label}={val:.0f}d'],
                textposition='middle right', textfont=dict(size=8, color='black'),
                showlegend=False, hoverinfo='skip'
            ))

fig_hl_scatter.update_layout(
    title=('Per-Earnings-Call Half-Life Scatter — Inefficient Community Leaders<br>'
           '<sup>Each point = one earnings call · solid line = mean · dashed = median · '
           f'window = {WINDOW_POST} trading days</sup>'),
    xaxis=dict(title='Community Leader', categoryorder='array',
               categoryarray=agg['ticker'].tolist(), tickangle=0),
    yaxis=dict(title='Half-Life (Trading Days)', showgrid=True,
               gridcolor='rgba(200,200,200,0.4)', zeroline=False),
    height=550, width=1200, plot_bgcolor='white', hovermode='closest',
    legend=dict(title='Leaders', font=dict(size=9), x=1.01, y=1,
                bgcolor='rgba(255,255,255,0.8)')
)
fig_hl_scatter.show()

# ============================================================
# LEADER-FOLLOWER LEAD TIME (Cross-Correlation)
# ============================================================
print("\n[18] Computing Cross-Correlation between Leaders and Followers...")

MAX_LAG = 10
ccf_results = []
community_labels = []

tdf_prices = prices_eda.copy().set_index(['ticker', 'date'])

for comm_idx, comm_nodes in enumerate(communities):
    leader = COMMUNITY_LEADERS[comm_idx]
    followers = [n for n in comm_nodes if n != leader]

    if not followers:
        continue

    try:
        leader_ret = tdf_prices.xs(leader, level='ticker')['log_return'].dropna()

        follower_rets = []
        for f in followers:
            f_ret = tdf_prices.xs(f, level='ticker')['log_return'].dropna()
            follower_rets.append(f_ret)

        if not follower_rets:
            continue

        aligned_followers = pd.concat(follower_rets, axis=1).mean(axis=1)
        aligned_data = pd.concat([leader_ret, aligned_followers], axis=1).dropna()
        aligned_data.columns = ['leader', 'follower_avg']

        cross_corr = ccf(aligned_data['follower_avg'], aligned_data['leader'])[:MAX_LAG + 1]

        ccf_results.append(cross_corr)
        community_labels.append(f"Comm {comm_idx}: {leader} (+{len(followers)} followers)")

    except Exception as e:
        print(f"Skipping Community {comm_idx} due to alignment error: {e}")

ccf_matrix = np.array(ccf_results)

fig_ccf = go.Figure(data=go.Heatmap(
    z=ccf_matrix,
    x=[f"Lag {i}" for i in range(MAX_LAG + 1)],
    y=community_labels,
    colorscale='RdBu_r', zmid=0,
    hovertemplate='<b>%{y}</b><br>%{x}<br>Correlation: %{z:.3f}<extra></extra>'
))
fig_ccf.update_layout(
    title='<b>Leader-Follower Cross-Correlation (Lead Time)</b><br>' +
          '<sup>Positive values at Lag > 0 indicate the Followers are lagging the Leader\'s price action.</sup>',
    xaxis_title='Days Lagging the Leader', yaxis_title='Louvain Community',
    height=max(500, len(community_labels) * 30), width=1000, plot_bgcolor='white'
)
fig_ccf.show()

# ============================================================
# SENTIMENT DIVERGENCE: Prepared Remarks vs. Unscripted Q&A (VECTORIZED)
# ============================================================

print("\n[21] Computing Sentiment Divergence for Community Leaders (Dynamic Split)...")

# 1. Filter strictly for Community Leaders
leader_tickers = list(COMMUNITY_LEADERS.values())
div_df = earnings_eda[earnings_eda['ticker'].isin(leader_tickers)].copy()
div_df = div_df.dropna(subset=['transcript']).reset_index(drop=True)

print(f"Analyzing {len(div_df)} total earnings calls for your Community Leaders...")

# 2. Dynamic String Splitting Function
def extract_sections(text):
    # Regex catches variations like "Question-and-Answer Session" or "Q&A Session"
    pattern = re.compile(r'(question-and-answer session|question and answer session|q&a session)', re.IGNORECASE)
    match = pattern.search(text)
    
    if match:
        split_idx = match.start()
        prepared = text[:split_idx]
        qa = text[split_idx:]
        
        # Take the LAST 20000 chars of prepared remarks (usually forward guidance/summary)
        # Take the FIRST 20000 chars of Q&A (usually the most aggressive/important analyst questions)
        return prepared[-20000:], qa[:20000]
    else:
        # Fallback if the specific phrase is missing
        return text[:20000], text[-20000:]

# Apply the split
sections = div_df['transcript'].apply(extract_sections)
prepared_list = [sec[0] for sec in sections]
qa_list = [sec[1] for sec in sections]

# 3. Batch LLM Inference (Fast)
print(" -> Scoring Prepared Remarks (Pre-Split)...")
res_prepared = finbert_pipeline(prepared_list) 

print(" -> Scoring Q&A (Post-Split)...")
res_qa = finbert_pipeline(qa_list)

# 4. Vectorized Post-Processing
df_prepared = pd.DataFrame(res_prepared)
df_qa = pd.DataFrame(res_qa)

label_map = {'positive': 1, 'negative': -1, 'neutral': 0}
prep_multiplier = df_prepared['label'].map(label_map).values
qa_multiplier = df_qa['label'].map(label_map).values

div_df['prepared_sentiment'] = prep_multiplier * df_prepared['score'].values
div_df['qa_sentiment'] = qa_multiplier * df_qa['score'].values

# Vectorized Delta Calculation
div_df['sentiment_delta'] = div_df['qa_sentiment'] - div_df['prepared_sentiment']

# ---------- Visualization ----------
fig_div = go.Figure()

# Add a y=x reference line
fig_div.add_trace(go.Scatter(
    x=[-1, 1], y=[-1, 1],
    mode='lines',
    line=dict(color='black', dash='dash'),
    name='No Tone Change (y=x)'
))

# Colors based on Delta
colors = np.where(div_df['sentiment_delta'] < 0, '#E15759', '#76B7B2')

fig_div.add_trace(go.Scatter(
    x=div_df['prepared_sentiment'],
    y=div_df['qa_sentiment'],
    mode='markers',
    marker=dict(size=9, color=colors, line=dict(width=1, color='darkgray')),
    text=div_df['ticker'],
    customdata=np.column_stack((
        div_df['date'].dt.strftime('%Y-%m-%d'), 
        div_df['sentiment_delta']
    )),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Date: %{customdata[0]}<br>'
        'Prepared Remarks: %{x:.2f}<br>'
        'Q&A Session: %{y:.2f}<br>'
        'Sentiment Delta: %{customdata[1]:.2f}'
        '<extra></extra>'
    ),
    name='Leader Earnings Calls'
))

# Danger Zone
fig_div.add_shape(
    type="rect", x0=0.2, y0=-1.0, x1=1.0, y1=-0.2,
    fillcolor="rgba(225, 87, 89, 0.1)", line=dict(width=0), layer="below"
)
fig_div.add_annotation(
    x=0.6, y=-0.6,
    text="Danger Zone<br>(Management Hiding Bad News)",
    showarrow=False, font=dict(color="#E15759", size=12)
)

fig_div.update_layout(
    title='<b>Sentiment Divergence: Scripted Remarks vs. Unscripted Q&A</b><br>' +
          '<sup>Filtered for Community Leaders | Split dynamically at the Q&A marker</sup>',
    xaxis_title='Prepared Remarks Sentiment (Guidance)',
    yaxis_title='Q&A Sentiment (Analyst Scrutiny)',
    height=650, width=900, plot_bgcolor='white', hovermode='closest'
)

fig_div.add_hline(y=0, line_width=1, line_color="lightgray")
fig_div.add_vline(x=0, line_width=1, line_color="lightgray")

fig_div.show()

# ============================================================
# SENTIMENT EXHAUSTION STUDY — VECTORISED price lookups via merge_asof
# ============================================================
print("\n[19] Running Dual-Section Sentiment Exhaustion Study (Sampling Transcripts)...")

exhaustion_data = []
FORWARD_WINDOW = 5

# Build a sorted price lookup table
p_sorted = prices_eda[['ticker', 'date', 'close']].copy().sort_values(['ticker', 'date'])
sample_earnings = earnings_eda.dropna(subset=['transcript']).sample(200, random_state=42)

# Pre-build per-ticker price DataFrames for faster lookups
_price_cache = {}
for t in sample_earnings['ticker'].unique():
    _price_cache[t] = p_sorted[p_sorted['ticker'] == t][['date', 'close']].reset_index(drop=True)

for _, row in tqdm(sample_earnings.iterrows(), total=len(sample_earnings)):
    ticker = row['ticker']
    edate = pd.to_datetime(row['date'])
    full_transcript = row['transcript']

    # --- DYNAMIC SPLIT LOGIC ---
    pattern = re.compile(r'(question-and-answer session|question and answer session|q&a session)', re.IGNORECASE)
    match = pattern.search(full_transcript)
    
    if match:
        split_idx = match.start()
        # Last 5000 chars of prepared (usually forward guidance)
        prepared = full_transcript[:split_idx][-5000:] 
        # First 5000 chars of Q&A (usually the most aggressive analyst questions)
        qa = full_transcript[split_idx:][:5000]
    else:
        # Fallback if marker is missing
        prepared = full_transcript[:5000]
        qa = full_transcript[-5000:]

    # --- SCORING BOTH SECTIONS ---
    try:
        # Score Prepared Remarks
        res_p = finbert_pipeline(prepared, top_k=None)
        probs_p = {r['label']: r['score'] for r in res_p}
        prep_score = probs_p.get('positive', 0) - probs_p.get('negative', 0)
        
        # Score Q&A Session
        res_q = finbert_pipeline(qa, top_k=None)
        probs_q = {r['label']: r['score'] for r in res_q}
        qa_score = probs_q.get('positive', 0) - probs_q.get('negative', 0)
    except Exception:
        continue

    # --- FORWARD RETURN CALCULATION ---
    try:
        tp = _price_cache.get(ticker)
        if tp is None or len(tp) < FORWARD_WINDOW + 2:
            continue

        pos = tp['date'].searchsorted(edate)
        if pos >= len(tp) or pos < 1:
            continue
        if pos + FORWARD_WINDOW >= len(tp):
            continue

        price_t0 = tp.iloc[pos]['close']
        price_t_minus_1 = tp.iloc[pos - 1]['close']
        price_t_plus_5 = tp.iloc[pos + FORWARD_WINDOW]['close']

        day0_ret = (price_t0 - price_t_minus_1) / price_t_minus_1
        forward_5d_ret = (price_t_plus_5 - price_t0) / price_t0

        exhaustion_data.append({
            'ticker': ticker, 
            'date': edate,
            'prepared_sentiment': prep_score,
            'qa_sentiment': qa_score,
            'day0_return': day0_ret,
            'forward_5d_return': forward_5d_ret
        })
    except Exception:
        continue

ex_df = pd.DataFrame(exhaustion_data)
print(f"Collected {len(ex_df)} valid exhaustion data points.")

if ex_df.empty:
    print("No valid exhaustion data — skipping chart.")
else:
    # --- RESHAPE DATA FOR SIDE-BY-SIDE PLOTTING ---
    melted_df = ex_df.melt(
        id_vars=['ticker', 'date', 'day0_return', 'forward_5d_return'],
        value_vars=['prepared_sentiment', 'qa_sentiment'],
        var_name='Section',
        value_name='sentiment'
    )
    
    # Rename for cleaner chart titles
    melted_df['Section'] = melted_df['Section'].replace({
        'prepared_sentiment': 'Prepared Remarks (Structured)',
        'qa_sentiment': 'Q&A Session (Unstructured)'
    })

    # --- PLOTTING ---
    # facet_col creates the side-by-side charts automatically
    fig_ex = px.scatter(
        melted_df, x='sentiment', y='forward_5d_return', color='day0_return',
        facet_col='Section', 
        color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
        hover_data=['ticker', 'date'], trendline="ols"
    )
    
    # Add target lines across ALL subplots
    fig_ex.add_hline(y=0, line_color="black", row='all', col='all')
    fig_ex.add_vline(x=0.8, line_dash="dash", line_color="red", row='all', col='all',
                     annotation_text="Extreme Optimism (>0.8)")

    fig_ex.update_layout(
        title='<b>Sentiment Exhaustion: Structured Script vs. Unscripted Q&A</b><br>' +
              '<sup>Comparing predictive power of management narrative vs. analyst scrutiny on 5-Day Reversal</sup>',
        coloraxis_colorbar=dict(title="Day 0<br>Return"),
        height=600, width=1200, plot_bgcolor='white'
    )
    
    # Clean up axes labels
    fig_ex.update_xaxes(title_text="FinBERT Sentiment Score")
    fig_ex.update_yaxes(title_text="Forward 5-Day Return")

    fig_ex.show()

# ============================================================
# SAV ANALYSIS — np.where for bar colours
# ============================================================
print("\n[20] Generating SAV Visualizations for ALL Community Leaders...")

leader_tickers_list = sorted(set(COMMUNITY_LEADERS.values()))

for target_leader in leader_tickers_list:
    comm_idx = [k for k, v in COMMUNITY_LEADERS.items() if v == target_leader][0]

    tdf = prices_eda[prices_eda['ticker'] == target_leader].copy().sort_values('date')
    tdf['log_ret'] = np.log(tdf['close'] / tdf['close'].shift(1))

    tdf['vol_med20'] = tdf['volume'].rolling(20).median()
    tdf['vol_std20'] = tdf['volume'].rolling(20).std()
    tdf['sav'] = (tdf['volume'] - tdf['vol_med20']) / tdf['vol_std20']
    tdf = tdf.dropna(subset=['sav'])

    raw_e_dates = earnings_eda[earnings_eda['ticker'] == target_leader]['date'].dt.tz_localize(None).dt.normalize()
    tdf['date_only'] = tdf['date'].dt.normalize()

    valid_e_dates = []
    for edate in raw_e_dates:
        pos = tdf['date_only'].searchsorted(edate)
        if pos < len(tdf):
            valid_e_dates.append(tdf['date_only'].iloc[pos])

    tdf['is_earnings'] = tdf['date_only'].isin(valid_e_dates)
    earnings_df = tdf[tdf['is_earnings']]

    fig_sav = make_subplots(
        rows=2, cols=1, row_heights=[0.6, 0.4], vertical_spacing=0.1,
        subplot_titles=(
            f"Time Series: SAV Spikes for ★ {target_leader} (Comm {comm_idx})",
            f"Distribution: Normal Days vs. Earnings Days"
        )
    )

    # >>> VECTORISED: np.where for bar colours <<<
    bar_colors = np.where(tdf['log_ret'] >= 0, '#76B7B2', '#E15759')

    fig_sav.add_trace(go.Bar(
        x=tdf['date'], y=tdf['sav'], marker_color=bar_colors, name='SAV',
        hovertemplate='Date: %{x|%Y-%m-%d}<br>SAV: %{y:.2f}<extra></extra>'
    ), row=1, col=1)

    if not earnings_df.empty:
        fig_sav.add_trace(go.Scatter(
            x=earnings_df['date'], y=earnings_df['sav'] + 0.5, mode='markers',
            marker=dict(symbol='diamond', size=8, color='gold',
                        line=dict(color='black', width=1)),
            name='Earnings Call',
            hovertemplate='<b>Earnings Call</b><br>Date: %{x|%Y-%m-%d}<br>SAV: %{y:.2f}<extra></extra>'
        ), row=1, col=1)

    fig_sav.add_hline(y=2.0, row=1, col=1, line=dict(color='black', dash='dash'),
                      annotation_text="Strategy Trigger (SAV > 2.0)")

    fig_sav.add_trace(go.Histogram(
        x=tdf[~tdf['is_earnings']]['sav'], name='Normal Days',
        marker_color='lightgray', opacity=0.75, histnorm='probability density'
    ), row=2, col=1)

    if not earnings_df.empty:
        fig_sav.add_trace(go.Histogram(
            x=earnings_df['sav'], name='Earnings Days',
            marker_color='gold', opacity=0.75, histnorm='probability density'
        ), row=2, col=1)

    fig_sav.update_layout(
        title=f'<b>Institutional Conviction Filter (SAV) — {target_leader}</b><br><sup>Validating the >2.0 Volume Threshold</sup>',
        height=800, width=1200, plot_bgcolor='white', barmode='overlay', showlegend=True
    )
    fig_sav.update_yaxes(title_text="SAV Score", row=1, col=1)
    fig_sav.update_xaxes(title_text="Standardized Abnormal Volume", row=2, col=1)
    fig_sav.update_yaxes(title_text="Density", row=2, col=1)
    fig_sav.show()

# EDA for AMIHUD + Drawdown Management Strategy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import ttest_ind


POSITIVE_WORDS = {
    "growth",
    "strong",
    "profit",
    "success",
    "opportunity",
    "excellent",
    "positive",
    "gain",
    "improvement",
    "best",
    "higher",
    "increase",
    "rose",
    "record",
    "benefit",
    "up",
    "expanding",
    "revenue",
    "income",
    "outperform",
}

NEGATIVE_WORDS = {
    "loss",
    "decline",
    "negative",
    "risk",
    "fail",
    "difficult",
    "challenging",
    "weak",
    "worst",
    "uncertainty",
    "lower",
    "decrease",
    "fell",
    "drop",
    "down",
    "missed",
    "adverse",
    "struggle",
    "concerns",
    "volatile",
}


# Expected to be defined by the parent script / notebook:
# prices_dev, prices_val, earnings_dev, earnings_val


def calculate_sentiment(text):
    if not isinstance(text, str):
        return 0.0

    words = text.lower().split()
    pos_count = sum(1 for word in words if word in POSITIVE_WORDS)
    neg_count = sum(1 for word in words if word in NEGATIVE_WORDS)
    total = pos_count + neg_count

    if total == 0:
        return 0.0

    return (pos_count - neg_count) / total


def print_section(title):
    print(f"\n{'=' * 80}")
    print(title)
    print(f"{'=' * 80}")


def run_low_vs_high_volatility_anomaly():
    print_section("Hypothesis 1: Low vs High Volatility Anomaly")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    print("Preprocessing...")
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])
    df["daily_ret"] = df.groupby("ticker")["close"].pct_change()

    print("Calculating rolling volatility...")
    df["volatility"] = df.groupby("ticker")["daily_ret"].transform(
        lambda series: series.rolling(window=252, min_periods=200).std() * np.sqrt(252)
    )

    print("Resampling to monthly frequency...")
    df = df.set_index("date")

    try:
        monthly_df = df.groupby("ticker").resample("ME").agg({"close": "last", "volatility": "last"})
    except ValueError:
        monthly_df = df.groupby("ticker").resample("M").agg({"close": "last", "volatility": "last"})

    monthly_df = monthly_df.reset_index()
    monthly_df["fwd_ret"] = monthly_df.groupby("ticker")["close"].pct_change().shift(-1)
    monthly_df = monthly_df.dropna(subset=["volatility", "fwd_ret"])

    if monthly_df.empty:
        print("No data available after processing.")
        return

    print("Ranking stocks into deciles...")
    monthly_df["decile"] = monthly_df.groupby("date")["volatility"].transform(
        lambda series: pd.qcut(series, 10, labels=False, duplicates="drop") + 1
        if len(series) >= 20
        else np.nan
    )
    monthly_df = monthly_df.dropna(subset=["decile"])

    print("Computing portfolio stats...")
    portfolio_ts = monthly_df.groupby(["date", "decile"])["fwd_ret"].mean().reset_index()
    stats_df = portfolio_ts.groupby("decile")["fwd_ret"].agg(["mean", "std", "count"])

    stats_df["ann_ret"] = stats_df["mean"] * 12
    stats_df["ann_vol"] = stats_df["std"] * np.sqrt(12)
    stats_df["sharpe"] = stats_df["ann_ret"] / stats_df["ann_vol"]

    print("\n--- Low vs High Volatility Anomaly Results ---")
    print(stats_df[["ann_ret", "ann_vol", "sharpe"]])

    try:
        low_vol_sharpe = stats_df.loc[1.0, "sharpe"]
        high_vol_sharpe = stats_df.loc[10.0, "sharpe"]

        print(f"\nLow Volatility (Decile 1) Sharpe: {low_vol_sharpe:.4f}")
        print(f"High Volatility (Decile 10) Sharpe: {high_vol_sharpe:.4f}")

        if low_vol_sharpe > high_vol_sharpe:
            print("\nConclusion: Low Volatility Anomaly CONFIRMED (Low Vol > High Vol).")
        else:
            print("\nConclusion: Low Volatility Anomaly REJECTED (High Vol > Low Vol).")
    except KeyError:
        print("\nCould not extract Decile 1 or 10 statistics.")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    stats_df["sharpe"].plot(kind="bar", ax=ax1, color="skyblue", edgecolor="black")
    ax1.set_title("Sharpe Ratio by Volatility Decile")
    ax1.set_xlabel("Volatility Decile (1=Lowest, 10=Highest)")
    ax1.set_ylabel("Annualized Sharpe Ratio")
    ax1.grid(axis="y", alpha=0.3)

    ax2.scatter(stats_df["ann_vol"], stats_df["ann_ret"], c="blue", s=100, alpha=0.7)
    ax2.plot(stats_df["ann_vol"], stats_df["ann_ret"], "b--", alpha=0.3)

    for idx, row in stats_df.iterrows():
        ax2.annotate(
            f"D{int(idx)}",
            (row["ann_vol"], row["ann_ret"]),
            xytext=(5, 5),
            textcoords="offset points",
        )

    ax2.set_title("Risk-Return Profile")
    ax2.set_xlabel("Annualized Volatility")
    ax2.set_ylabel("Annualized Return")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def run_volume_confirmation_experiment():
    print_section("Hypothesis 2: High-Volume vs Low-Volume Up Days")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()
    print("Dataset loaded successfully.")

    df.sort_values(by=["ticker", "date"], inplace=True)
    df["prev_close"] = df.groupby("ticker")["close"].shift(1)
    df["return"] = (df["close"] - df["prev_close"]) / df["prev_close"]
    df["next_return"] = df.groupby("ticker")["return"].shift(-1)
    df["vol_ma_20"] = df.groupby("ticker")["volume"].transform(lambda series: series.rolling(window=20).mean())

    df_clean = df.dropna(subset=["return", "next_return", "vol_ma_20", "volume"]).copy()
    up_days = df_clean[df_clean["return"] > 0].copy()

    print(f"Total Up Days processed: {len(up_days)}")

    high_vol_mask = up_days["volume"] > (1.5 * up_days["vol_ma_20"])
    low_vol_mask = up_days["volume"] <= up_days["vol_ma_20"]

    high_vol_returns = up_days.loc[high_vol_mask, "next_return"]
    low_vol_returns = up_days.loc[low_vol_mask, "next_return"]

    n_high = len(high_vol_returns)
    n_low = len(low_vol_returns)

    if n_high == 0 or n_low == 0:
        print("Insufficient data points for one of the groups.")
        return

    mean_high = high_vol_returns.mean()
    mean_low = low_vol_returns.mean()
    std_high = high_vol_returns.std()
    std_low = low_vol_returns.std()

    print("\n--- Results ---")
    print(
        f"High Volume Up-Days (>150% MA) (n={n_high}): Mean Next-Day Return = "
        f"{mean_high:.6f} ({mean_high * 100:.4f}%), Std = {std_high:.6f}"
    )
    print(
        f"Low Volume Up-Days  (<=100% MA) (n={n_low}): Mean Next-Day Return = "
        f"{mean_low:.6f} ({mean_low * 100:.4f}%), Std = {std_low:.6f}"
    )

    t_stat, p_val = stats.ttest_ind(high_vol_returns, low_vol_returns, equal_var=False)

    print("\n--- T-Test (Welch's) ---")
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_val:.4e}")

    if p_val < 0.05:
        if t_stat > 0:
            print("Result: Significant Positive Difference (High Vol > Low Vol).")
        else:
            print("Result: Significant Negative Difference (High Vol < Low Vol).")
    else:
        print("Result: No Statistically Significant Difference.")

    plt.figure(figsize=(10, 6))
    means = [mean_high * 100, mean_low * 100]
    standard_errors = [std_high * 100 / np.sqrt(n_high), std_low * 100 / np.sqrt(n_low)]
    labels = ["High Volume (>150% MA)", "Low Volume (<=100% MA)"]

    plt.bar(labels, means, yerr=standard_errors, capsize=10, color=["#d62728", "#1f77b4"], alpha=0.8)
    plt.title("Mean Next-Day Return: High Vol vs Low Vol Up-Days")
    plt.ylabel("Average Next-Day Return (%)")
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    for idx, value in enumerate(means):
        plt.text(idx, value + (0.01 if value >= 0 else -0.05), f"{value:.4f}%", ha="center", fontweight="bold")

    plt.show()


def run_rsi_mean_reversion_experiment():
    print_section("Hypothesis 3: RSI Oversold vs Overbought")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

    print("Calculating RSI...")
    df["delta"] = df.groupby("ticker")["close"].diff()

    up = df["delta"].clip(lower=0)
    down = -1 * df["delta"].clip(upper=0)

    ma_up = up.groupby(df["ticker"]).ewm(alpha=1 / 14, adjust=False).mean()
    ma_down = down.groupby(df["ticker"]).ewm(alpha=1 / 14, adjust=False).mean()

    df["avg_gain"] = ma_up.reset_index(level=0, drop=True)
    df["avg_loss"] = ma_down.reset_index(level=0, drop=True)

    with np.errstate(divide="ignore", invalid="ignore"):
        rs = df["avg_gain"] / df["avg_loss"]
        df["rsi"] = 100 - (100 / (1 + rs))

    df.loc[df["avg_loss"] == 0, "rsi"] = 100
    df["next_close"] = df.groupby("ticker")["close"].shift(-1)
    df["next_ret"] = (df["next_close"] - df["close"]) / df["close"]

    valid_data = df.dropna(subset=["rsi", "next_ret"])
    oversold = valid_data[valid_data["rsi"] < 30]["next_ret"]
    overbought = valid_data[valid_data["rsi"] > 70]["next_ret"]

    print(f"Oversold (RSI < 30) count: {len(oversold)}")
    print(f"Overbought (RSI > 70) count: {len(overbought)}")

    mean_os = oversold.mean()
    mean_ob = overbought.mean()

    print(f"Mean Next-Day Return (Oversold): {mean_os:.6f}")
    print(f"Mean Next-Day Return (Overbought): {mean_ob:.6f}")

    t_stat, p_val = ttest_ind(oversold, overbought, equal_var=False)
    print(f"T-statistic: {t_stat:.4f}")
    print(f"P-value: {p_val:.4e}")

    plt.figure(figsize=(10, 6))
    plt.hist(oversold, bins=100, range=(-0.05, 0.05), density=True, alpha=0.5, label="Oversold (RSI < 30)")
    plt.hist(overbought, bins=100, range=(-0.05, 0.05), density=True, alpha=0.5, label="Overbought (RSI > 70)")
    plt.title("Next-Day Return Distribution: Oversold vs Overbought")
    plt.xlabel("Next Day Return")
    plt.ylabel("Density")
    plt.legend()
    plt.show()


def run_illiquidity_premium_experiment():
    print_section("Hypothesis 4: Illiquidity Premium")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    print("Calculating Amihud Ratio...")
    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["dollar_vol"] = (df["close"] * df["volume"]).replace(0, np.nan)
    df["amihud"] = (df["ret"].abs() / df["dollar_vol"]).replace([np.inf, -np.inf], np.nan)

    print("Aggregating to monthly frequency...")
    monthly_data = df.groupby(["ticker", pd.Grouper(key="date", freq="ME")]).agg({"amihud": "mean", "close": "last"}).reset_index()
    monthly_data["ret_m"] = monthly_data.groupby("ticker")["close"].pct_change()
    monthly_data["fwd_ret_m"] = monthly_data.groupby("ticker")["ret_m"].shift(-1)

    valid_data = monthly_data.dropna(subset=["amihud", "fwd_ret_m"]).copy()

    print("Ranking stocks into deciles...")

    def rank_deciles(series):
        try:
            return pd.qcut(series, 10, labels=False, duplicates="drop")
        except ValueError:
            return np.nan

    valid_data["decile"] = valid_data.groupby("date")["amihud"].transform(rank_deciles)
    valid_data = valid_data.dropna(subset=["decile"])

    print("Constructing portfolios...")
    portfolios = valid_data.groupby(["date", "decile"])["fwd_ret_m"].mean().unstack()

    if 0.0 not in portfolios.columns or 9.0 not in portfolios.columns:
        print("Error: Deciles 0 and 9 not found in aggregated data.")
        print("Columns found:", portfolios.columns)
        return

    illiquid_returns = portfolios[9.0]
    liquid_returns = portfolios[0.0]
    spread_returns = illiquid_returns - liquid_returns

    mean_spread = spread_returns.mean()
    t_stat, p_val = stats.ttest_1samp(spread_returns, 0)

    print("\n--- Illiquidity Premium Analysis Results ---")
    print(f"Observation Period: {valid_data['date'].min().date()} to {valid_data['date'].max().date()}")
    print(f"Total Months: {len(spread_returns)}")
    print(f"Average Monthly Spread Return: {mean_spread:.4%}")
    print(f"Annualized Spread Return: {(1 + mean_spread) ** 12 - 1:.4%}")
    print(f"T-Statistic: {t_stat:.4f}")
    print(f"P-Value: {p_val:.4f}")

    cum_illiquid = (1 + illiquid_returns).cumprod()
    cum_liquid = (1 + liquid_returns).cumprod()
    cum_spread = (1 + spread_returns).cumprod()

    plt.figure(figsize=(10, 6))
    plt.plot(cum_illiquid.index, cum_illiquid, label="Illiquid (Top Decile)")
    plt.plot(cum_liquid.index, cum_liquid, label="Liquid (Bottom Decile)")
    plt.plot(cum_spread.index, cum_spread, label="Long Illiquid / Short Liquid", linestyle="--", color="black")
    plt.title("Illiquidity Premium: Cumulative Returns")
    plt.xlabel("Date")
    plt.ylabel("Growth of $1")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def run_sentiment_acceleration_experiment():
    print_section("Hypothesis 5: Sentiment Acceleration vs Post-Earnings Sharpe")
    print("Using preloaded earnings_dev, earnings_val, prices_dev, and prices_val...")
    df_earnings = pd.concat([earnings_dev, earnings_val], ignore_index=True)
    df_prices = pd.concat([prices_dev, prices_val], ignore_index=True)

    print(f"Earnings events loaded: {len(df_earnings)}")
    print(f"Price records loaded: {len(df_prices)}")

    print("Calculating sentiment scores...")
    df_earnings["sentiment"] = df_earnings["transcript"].apply(calculate_sentiment)

    print("Calculating sentiment acceleration...")
    df_earnings["date"] = pd.to_datetime(df_earnings["date"]).dt.tz_localize(None).dt.normalize()
    df_earnings = df_earnings.sort_values(by=["ticker", "date"])
    df_earnings["prev_sentiment"] = df_earnings.groupby("ticker")["sentiment"].shift(1)
    df_earnings["sentiment_change"] = df_earnings["sentiment"] - df_earnings["prev_sentiment"]
    df_earnings_clean = df_earnings.dropna(subset=["sentiment_change"]).copy()

    print("Processing price data...")
    df_prices["date"] = pd.to_datetime(df_prices["date"]).dt.tz_localize(None).dt.normalize()
    df_prices = df_prices.sort_values(["ticker", "date"])
    df_prices["return"] = df_prices.groupby("ticker")["close"].pct_change()

    print("Building price lookup dictionary...")
    price_dict = {ticker: frame.set_index("date")["return"] for ticker, frame in df_prices.groupby("ticker")}

    sharpe_ratios = []
    valid_indices = []

    print(f"Calculating post-earnings Sharpe Ratio for {len(df_earnings_clean)} events...")

    for idx, row in df_earnings_clean.iterrows():
        ticker = row["ticker"]
        earnings_date = row["date"]

        if ticker not in price_dict:
            continue

        ticker_returns = price_dict[ticker]

        try:
            idx_loc = ticker_returns.index.searchsorted(earnings_date, side="left")

            if idx_loc >= len(ticker_returns):
                continue

            current_date_at_idx = ticker_returns.index[idx_loc]
            start_pos = idx_loc + 1 if current_date_at_idx == earnings_date else idx_loc
            end_pos = start_pos + 20

            if end_pos > len(ticker_returns):
                continue

            window_returns = ticker_returns.iloc[start_pos:end_pos].values
            window_returns = window_returns[~np.isnan(window_returns)]

            if len(window_returns) < 10:
                continue

            mean_ret = np.mean(window_returns)
            std_ret = np.std(window_returns)
            sharpe = 0 if std_ret == 0 or np.isnan(std_ret) else mean_ret / std_ret

            sharpe_ratios.append(sharpe)
            valid_indices.append(idx)
        except Exception:
            continue

    df_analysis = df_earnings_clean.loc[valid_indices].copy()
    df_analysis["sharpe_ratio"] = sharpe_ratios

    print(f"Successfully computed Sharpe Ratios for {len(df_analysis)} events.")

    if len(df_analysis) < 50:
        print("Insufficient data points for meaningful regression.")
        return

    print("\nRunning Regression Analysis...")
    df_analysis = df_analysis.dropna(subset=["sharpe_ratio", "sentiment", "sentiment_change"])
    df_analysis["sentiment_z"] = (df_analysis["sentiment"] - df_analysis["sentiment"].mean()) / df_analysis["sentiment"].std()
    df_analysis["sentiment_change_z"] = (
        (df_analysis["sentiment_change"] - df_analysis["sentiment_change"].mean()) / df_analysis["sentiment_change"].std()
    )

    X1 = sm.add_constant(df_analysis["sentiment_z"])
    y = df_analysis["sharpe_ratio"]
    model1 = sm.OLS(y, X1).fit()

    X2 = sm.add_constant(df_analysis["sentiment_change_z"])
    model2 = sm.OLS(y, X2).fit()

    X3 = sm.add_constant(df_analysis[["sentiment_z", "sentiment_change_z"]])
    model3 = sm.OLS(y, X3).fit()

    print("\n=== Regression 1: Sharpe ~ Absolute Sentiment (Z-scored) ===")
    print(model1.summary().tables[1])
    print(f"R-squared: {model1.rsquared:.6f}")

    print("\n=== Regression 2: Sharpe ~ Sentiment Change (Z-scored) ===")
    print(model2.summary().tables[1])
    print(f"R-squared: {model2.rsquared:.6f}")

    print("\n=== Regression 3: Sharpe ~ Absolute + Change ===")
    print(model3.summary().tables[1])
    print(f"R-squared: {model3.rsquared:.6f}")

    df_analysis["group"] = df_analysis["sentiment_change"].apply(lambda value: "Improving" if value > 0 else "Deteriorating")
    group_means = df_analysis.groupby("group")["sharpe_ratio"].mean()
    group_counts = df_analysis.groupby("group")["sharpe_ratio"].count()

    print("\n=== Group Comparison (Average Daily Sharpe Ratio) ===")
    print(group_means)
    print("\nGroup Counts:")
    print(group_counts)

    plt.figure(figsize=(10, 6))
    group_means.plot(kind="bar", color=["red", "green"], alpha=0.7)
    plt.title("Average Post-Earnings Sharpe Ratio (t+1 to t+20)\nby Sentiment Momentum")
    plt.ylabel("Average Daily Sharpe Ratio")
    plt.xlabel("Sentiment Change")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()


def run_volatility_acceleration_experiment():
    print_section("Hypothesis 6: Volatility Acceleration vs Forward Drawdown")
    df = pd.concat([prices_dev, prices_val], ignore_index=True)
    df.sort_values(["ticker", "date"], inplace=True)

    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["vol_10"] = df.groupby("ticker")["ret"].transform(lambda series: series.rolling(10, min_periods=10).std())
    df["vol_60"] = df.groupby("ticker")["ret"].transform(lambda series: series.rolling(60, min_periods=60).std())
    df["vol_ratio"] = df["vol_10"] / df["vol_60"]

    df["next_low"] = df.groupby("ticker")["low"].shift(-1)

    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=20)
    df["fwd_min_low"] = df.groupby("ticker")["next_low"].transform(
        lambda series: series.rolling(window=indexer, min_periods=20).min()
    )
    df["fwd_drawdown"] = (df["fwd_min_low"] - df["close"]) / df["close"]

    valid_data = df.dropna(subset=["vol_ratio", "fwd_drawdown"])
    shock_group = valid_data[valid_data["vol_ratio"] > 1.5]["fwd_drawdown"]
    control_group = valid_data[(valid_data["vol_ratio"] >= 0.9) & (valid_data["vol_ratio"] <= 1.1)]["fwd_drawdown"]

    ks_stat, p_value = stats.ks_2samp(shock_group, control_group)

    print("=== Volatility Acceleration Experiment Results ===")
    print(f"Shock Group Size: {len(shock_group)}")
    print(f"Control Group Size: {len(control_group)}")
    print(f"Shock Group Mean Forward Drawdown (Next 20 Days): {shock_group.mean():.4f}")
    print(f"Control Group Mean Forward Drawdown (Next 20 Days): {control_group.mean():.4f}")
    print(f"KS Test Statistic: {ks_stat:.4f}")
    print(f"KS Test p-value: {p_value:.4e}")

    shock_prob = (shock_group < -0.10).mean()
    control_prob = (control_group < -0.10).mean()
    print(f"Probability of >10% price drop in Shock group: {shock_prob:.2%}")
    print(f"Probability of >10% price drop in Control group: {control_prob:.2%}")

    plt.figure(figsize=(10, 6))
    plt.hist(control_group, bins=100, range=(-0.5, 0.1), density=True, alpha=0.5, label="Control (Normal Vol)", color="blue")
    plt.hist(shock_group, bins=100, range=(-0.5, 0.1), density=True, alpha=0.5, label="Shock (Vol Accel > 1.5x)", color="red")
    plt.title("Distribution of Maximum Forward Drawdown (Next 20 Days)")
    plt.xlabel("Max Drawdown (Relative to Entry Price)")
    plt.ylabel("Density")
    plt.axvline(-0.10, color="black", linestyle="--", alpha=0.7, label="10% Drop Threshold")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def run_trend_filter_crash_experiment():
    print_section("Hypothesis 7: Trend Filter During Crash Periods")
    print("Using preloaded prices_dev...")
    df = prices_dev.copy()

    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    print("Calculating SMA and Returns...")
    grouped = df.groupby("ticker")
    df["sma_200"] = grouped["close"].transform(lambda series: series.rolling(200).mean())
    df["prev_close"] = grouped["close"].shift(1)
    df["return"] = df["close"] / df["prev_close"] - 1.0
    df["signal"] = (df["close"] > df["sma_200"]).astype(int)
    df["strategy_pos"] = grouped["signal"].shift(1).fillna(0)
    df["strat_ret"] = df["return"] * df["strategy_pos"]

    market_series = df.groupby("date")["close"].mean()
    market_peak = market_series.cummax()
    market_dd = (market_series - market_peak) / market_peak

    global_low_date = market_dd.idxmin()
    peak_date = market_series.loc[:global_low_date].idxmax()

    crash_start = peak_date
    crash_end = global_low_date
    print(f"Crash Period Identified: {crash_start.date()} to {crash_end.date()}")
    print(f"Market Drawdown: {market_dd.min():.2%}")

    lookback_start = crash_start - pd.Timedelta(days=180)
    pre_crash_data = df[(df["date"] >= lookback_start) & (df["date"] < crash_start)]
    vol_stats = pre_crash_data.groupby("ticker")["return"].std()

    vol_threshold = vol_stats.quantile(0.8)
    high_vol_tickers = vol_stats[vol_stats >= vol_threshold].index
    print(f"High Volatility Threshold (Std Dev): {vol_threshold:.4f}")
    print(f"Selected {len(high_vol_tickers)} high volatility stocks.")

    sim_data = df[
        (df["date"] >= crash_start)
        & (df["date"] <= crash_end)
        & (df["ticker"].isin(high_vol_tickers))
    ].copy()

    results = []

    for ticker, group in sim_data.groupby("ticker"):
        if len(group) < 10:
            continue

        bh_equity = (1 + group["return"].fillna(0)).cumprod()
        bh_peak = bh_equity.cummax()
        bh_dd_series = (bh_equity - bh_peak) / bh_peak
        bh_mdd = bh_dd_series.min()

        tr_equity = (1 + group["strat_ret"].fillna(0)).cumprod()
        tr_peak = tr_equity.cummax()
        tr_dd_series = (tr_equity - tr_peak) / tr_peak
        tr_mdd = tr_dd_series.min()

        results.append({"ticker": ticker, "BH_MDD": bh_mdd, "Trend_MDD": tr_mdd})

    results_df = pd.DataFrame(results).dropna()

    mean_bh_mdd = results_df["BH_MDD"].mean()
    mean_tr_mdd = results_df["Trend_MDD"].mean()

    print("\n=== Simulation Results ===")
    print(f"Average Max Drawdown (Buy & Hold): {mean_bh_mdd:.2%}")
    print(f"Average Max Drawdown (Trend Filter): {mean_tr_mdd:.2%}")

    improvement = mean_tr_mdd - mean_bh_mdd
    print(f"Average Improvement: {improvement * 100:.2f} percentage points")

    t_stat, p_val = stats.ttest_rel(results_df["BH_MDD"], results_df["Trend_MDD"])
    print(f"Paired t-test: t-statistic={t_stat:.4f}, p-value={p_val:.4e}")

    if p_val < 0.05:
        print("Result: Statistically significant difference.")
    else:
        print("Result: No statistically significant difference.")

    plt.figure(figsize=(8, 6))
    plt.boxplot([results_df["BH_MDD"], results_df["Trend_MDD"]], tick_labels=["Buy & Hold", "Trend Filter"])
    plt.title("Distribution of Maximum Drawdowns (High Volatility Stocks)")
    plt.ylabel("Maximum Drawdown")
    plt.grid(True, axis="y", alpha=0.3)
    plt.show()

    piv_bh = sim_data.pivot(index="date", columns="ticker", values="return").mean(axis=1).fillna(0)
    piv_tr = sim_data.pivot(index="date", columns="ticker", values="strat_ret").mean(axis=1).fillna(0)

    cum_bh = (1 + piv_bh).cumprod()
    cum_tr = (1 + piv_tr).cumprod()

    plt.figure(figsize=(12, 6))
    plt.plot(cum_bh, label="Buy & Hold Portfolio", color="red")
    plt.plot(cum_tr, label="Trend Filter Portfolio", color="blue")
    plt.title(f"Performance of High Volatility Stocks during Crash ({crash_start.date()} - {crash_end.date()})")
    plt.xlabel("Date")
    plt.ylabel("Normalized Equity")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def run_amihud_vol_window_search_experiment():
    """
    Hypothesis 8: Amihud & Volatility Lookback Window Grid Search.

    Tests whether the 60-day lookback currently used for both the Amihud
    illiquidity ratio and the volatility metric in universe selection is
    optimal, or whether a shorter/longer window (20, 40, 60, 120 days)
    produces a better-quality investable universe.

    Method:
      For each (amihud_window, vol_window) combination:
        1. Compute rolling Amihud(W) and rolling Vol(W) per ticker.
        2. Monthly: rank tickers by each metric, form composite score,
           select top 40% (universe).
        3. Measure equal-weighted forward 1-month return of that universe.
        4. Compute annualised Sharpe of the strategy.

    A higher Sharpe = better predictive quality from that window.
    """
    print_section("Hypothesis 8: Amihud & Volatility Lookback Window Grid Search")
    print("Using preloaded prices_dev...")

    df = prices_dev.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values(["ticker", "date"])

    df["ret"] = df.groupby("ticker")["close"].pct_change()
    df["dollar_vol"] = (df["close"] * df["volume"]).replace(0, np.nan)
    df["amihud_daily"] = (df["ret"].abs() / df["dollar_vol"]).replace([np.inf, -np.inf], np.nan)

    windows = [20, 40, 60, 120]
    results = []

    for amihud_w in windows:
        for vol_w in windows:
            label = f"Amihud-{amihud_w}d / Vol-{vol_w}d"
            print(f"  Testing {label}...")

            # --- Compute windowed metrics for every row ---
            df["amihud_roll"] = (
                df.groupby("ticker")["amihud_daily"]
                .transform(lambda s: s.rolling(amihud_w, min_periods=max(10, amihud_w // 2)).mean())
            )
            df["vol_roll"] = (
                df.groupby("ticker")["ret"]
                .transform(lambda s: s.rolling(vol_w, min_periods=max(10, vol_w // 2)).std() * np.sqrt(252))
            )

            # --- Resample to month-end snapshots ---
            df_indexed = df.set_index("date")
            try:
                monthly = (
                    df_indexed.groupby("ticker")
                    .resample("ME")
                    .agg({"close": "last", "amihud_roll": "last", "vol_roll": "last"})
                )
            except ValueError:
                monthly = (
                    df_indexed.groupby("ticker")
                    .resample("M")
                    .agg({"close": "last", "amihud_roll": "last", "vol_roll": "last"})
                )
            monthly = monthly.reset_index()
            monthly["fwd_ret"] = monthly.groupby("ticker")["close"].pct_change().shift(-1)
            monthly = monthly.dropna(subset=["amihud_roll", "vol_roll", "fwd_ret"])

            # --- Monthly universe selection: top 40% composite score ---
            def score_month(group):
                if len(group) < 5:
                    group["selected"] = False
                    return group
                group = group.copy()
                group["vol_rank"] = group["vol_roll"].rank(ascending=True, method="average")
                group["amihud_rank"] = group["amihud_roll"].rank(ascending=False, method="average")
                group["composite"] = group["vol_rank"] + group["amihud_rank"]
                threshold = group["composite"].quantile(0.60)  # top 40%
                group["selected"] = group["composite"] >= threshold
                return group

            monthly = monthly.groupby("date", group_keys=False).apply(score_month)

            # --- Equal-weighted portfolio return of selected universe ---
            portfolio = (
                monthly[monthly["selected"]]
                .groupby("date")["fwd_ret"]
                .mean()
            )

            if len(portfolio) < 12:
                print(f"    Insufficient months ({len(portfolio)}) — skipping.")
                continue

            ann_ret = portfolio.mean() * 12
            ann_vol = portfolio.std() * np.sqrt(12)
            sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
            hit_rate = (portfolio > 0).mean()

            results.append({
                "label": label,
                "amihud_window": amihud_w,
                "vol_window": vol_w,
                "ann_return": ann_ret,
                "ann_vol": ann_vol,
                "sharpe": sharpe,
                "hit_rate": hit_rate,
                "months": len(portfolio),
            })
            print(f"    Sharpe={sharpe:.3f}  AnnRet={ann_ret:.2%}  AnnVol={ann_vol:.2%}  HitRate={hit_rate:.2%}")

    if not results:
        print("No valid results produced.")
        return

    results_df = pd.DataFrame(results).sort_values("sharpe", ascending=False)

    print("\n--- Grid Search Results (sorted by Sharpe) ---")
    print(results_df[["label", "ann_return", "ann_vol", "sharpe", "hit_rate", "months"]].to_string(index=False))

    best = results_df.iloc[0]
    current = results_df[results_df["label"] == "Amihud-60d / Vol-60d"]
    print(f"\nBest window  : {best['label']} (Sharpe={best['sharpe']:.4f})")
    if not current.empty:
        curr_sharpe = current.iloc[0]["sharpe"]
        print(f"Current (60d): Amihud-60d / Vol-60d (Sharpe={curr_sharpe:.4f})")
        diff = best["sharpe"] - curr_sharpe
        if diff > 0.05:
            print(f"Conclusion: A different window improves Sharpe by {diff:.4f} — consider changing the lookback.")
        else:
            print(f"Conclusion: 60-day window is competitive (difference < 0.05 Sharpe). Current setting is acceptable.")

    # --- Plots ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle("Hypothesis 8: Amihud & Vol Lookback Window Grid Search", fontsize=14, fontweight="bold")

    # Heatmap: Sharpe by (amihud_w, vol_w)
    pivot_sharpe = results_df.pivot(index="amihud_window", columns="vol_window", values="sharpe")
    im = axes[0].imshow(pivot_sharpe.values, cmap="RdYlGn", aspect="auto")
    axes[0].set_xticks(range(len(pivot_sharpe.columns)))
    axes[0].set_yticks(range(len(pivot_sharpe.index)))
    axes[0].set_xticklabels([f"{c}d" for c in pivot_sharpe.columns])
    axes[0].set_yticklabels([f"{r}d" for r in pivot_sharpe.index])
    axes[0].set_xlabel("Vol Window")
    axes[0].set_ylabel("Amihud Window")
    axes[0].set_title("Sharpe Ratio Heatmap")
    for i in range(len(pivot_sharpe.index)):
        for j in range(len(pivot_sharpe.columns)):
            val = pivot_sharpe.values[i, j]
            if not np.isnan(val):
                axes[0].text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                             color="black" if 0.2 < val < 0.8 else "white")
    fig.colorbar(im, ax=axes[0])

    # Bar: Sharpe by label
    colors = ["#2E86AB" if r["label"] != "Amihud-60d / Vol-60d" else "#E15759" for _, r in results_df.iterrows()]
    axes[1].barh(results_df["label"], results_df["sharpe"], color=colors, edgecolor="black", alpha=0.85)
    axes[1].axvline(x=0, color="black", linewidth=0.8)
    axes[1].set_xlabel("Annualised Sharpe Ratio")
    axes[1].set_title("Sharpe by Window Combination\n(red = current 60d/60d)")
    axes[1].grid(axis="x", alpha=0.3)

    # Scatter: Ann Return vs Ann Vol
    for _, row in results_df.iterrows():
        color = "#E15759" if row["label"] == "Amihud-60d / Vol-60d" else "#2E86AB"
        axes[2].scatter(row["ann_vol"], row["ann_return"], color=color, s=80, zorder=3)
        axes[2].annotate(
            row["label"].replace(" / ", "\n"),
            (row["ann_vol"], row["ann_return"]),
            xytext=(4, 4), textcoords="offset points", fontsize=7
        )
    axes[2].set_xlabel("Annualised Volatility")
    axes[2].set_ylabel("Annualised Return")
    axes[2].set_title("Risk-Return by Window")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def main():
    required_datasets = ("prices_dev", "prices_val", "earnings_dev", "earnings_val")
    missing = [name for name in required_datasets if name not in globals()]
    if missing:
        raise NameError(
            "Missing preloaded dataset variables: "
            + ", ".join(missing)
            + ". Define prices_dev, prices_val, earnings_dev, and earnings_val before running this file."
        )

    experiments = [
        run_low_vs_high_volatility_anomaly,
        run_volume_confirmation_experiment,
        run_rsi_mean_reversion_experiment,
        run_illiquidity_premium_experiment,
        run_sentiment_acceleration_experiment,
        run_volatility_acceleration_experiment,
        run_trend_filter_crash_experiment,
        run_amihud_vol_window_search_experiment,  # H8: Lookback window grid search
    ]

    for experiment in experiments:
        try:
            experiment()
        except Exception as exc:
            print(f"Experiment '{experiment.__name__}' failed: {exc}")


if __name__ == "__main__":
    main()

# Enchanced Strat (20MA Crossover w hard stop loss)

In [ ]:
# ============================================================
# CELL 13: Enhanced Strategy Implementation
# ============================================================

import re
import networkx as nx
from statsmodels.stats.diagnostic import acorr_ljungbox
from tqdm import tqdm

print("\n" + "="*70)
print("ENHANCED STRATEGY")
print("="*70)


class EnhancedStrategy(BaseStrategy):

    def __init__(self, finbert_pipeline=None):
        super().__init__(finbert_pipeline)
        self.entry_dates = {}                    # {ticker: {date, price, low, atr, ema, tp, sl}}
        self.inefficient_tickers = []
        self.efficient_tickers = []
        self.weekly_clusters = {}                # {date_str: {cid: [tickers]}}
        self.weekly_leaders = {}                 # {date_str: {cid: leader_ticker}}
        self.high_corr_flag = {}                 # {date_str: bool}
        self.avg_volume = {}                     # {ticker: float}
        self.sentiment_cache = {}                # {(ticker, date_str): float}
        self._sentiment_by_ticker = defaultdict(list)

    # ==============================================================
    # 1. DATA CLEANING
    # ==============================================================
    def clean_data(self, prices_df, earnings_df):
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]

        # Drop tickers with no earnings calls
        tickers_with_earnings = set(earnings['ticker'].unique())
        prices = prices[prices['ticker'].isin(tickers_with_earnings)]

        # Forward-fill close prices per ticker
        prices = prices.sort_values(['ticker', 'date'])
        prices['close'] = prices.groupby('ticker')['close'].ffill()

        return prices, earnings

    # ==============================================================
    # 2. ANALYTICS — Ljung-Box, Louvain, Correlation Flag, SAV, ATR, EMA
    # ==============================================================
    def calculate_analytics(self, prices_df):
        print("Computing enhanced analytics...")
        prices = prices_df.copy().sort_values(['ticker', 'date'])

        # ---- vectorised log returns ----
        prices['log_return'] = prices.groupby('ticker')['close'].transform(
            lambda x: np.log(x / x.shift(1))
        )

        # ---- LIST 1: Ljung-Box → inefficient / efficient tickers ----
        print("  [1/4] Ljung-Box test for market efficiency...")
        lb_pvals = {}
        for ticker, group in prices.groupby('ticker')['log_return']:
            lr = group.dropna()
            if len(lr) < 50:
                continue
            try:
                res = acorr_ljungbox(lr, lags=[20], return_df=True)
                lb_pvals[ticker] = res['lb_pvalue'].iloc[0]
            except Exception:
                continue

        alpha = 0.05
        self.inefficient_tickers = [t for t, p in lb_pvals.items() if p < alpha]
        self.efficient_tickers   = [t for t, p in lb_pvals.items() if p >= alpha]
        print(f"    Inefficient: {len(self.inefficient_tickers)}, "
              f"Efficient: {len(self.efficient_tickers)}")

        # ---- average volume for leader identification ----
        self.avg_volume = (
            prices[prices['ticker'].isin(self.inefficient_tickers)]
            .groupby('ticker')['volume'].mean().to_dict()
        )

        # ---- LIST 2 & 3 + BOOLEAN: weekly Louvain clusters,
        #      leaders, efficient-ticker correlation flag ----
        print("  [2/4] Weekly Louvain clustering & correlation regime flag...")
        returns_pivot = prices.pivot_table(
            index='date', columns='ticker', values='log_return'
        )
        ineff_cols = [t for t in self.inefficient_tickers if t in returns_pivot.columns]
        eff_cols   = [t for t in self.efficient_tickers   if t in returns_pivot.columns]
        ineff_ret  = returns_pivot[ineff_cols]
        eff_ret    = returns_pivot[eff_cols]

        weekly_dates = pd.date_range(
            prices['date'].min(), prices['date'].max(), freq='W-FRI'
        )
        CORR_WINDOW    = 60
        EDGE_THRESHOLD = 0.6

        for wd in tqdm(weekly_dates, desc="    Weeks"):
            ws = wd.strftime('%Y-%m-%d')

            # --- inefficient tickers → Louvain ---
            window = ineff_ret[ineff_ret.index <= wd].tail(CORR_WINDOW)
            valid  = window.columns[window.notna().sum() >= 20]

            if len(valid) < 3:
                self.weekly_clusters[ws] = {}
                self.weekly_leaders[ws]  = {}
                self.high_corr_flag[ws]  = False
                continue

            corr_m = window[valid].corr()
            tlist  = corr_m.columns.tolist()
            cvals  = corr_m.values
            ri, ci = np.triu_indices(len(tlist), k=1)
            rhos   = cvals[ri, ci]
            emask  = rhos >= EDGE_THRESHOLD

            G = nx.Graph()
            G.add_nodes_from(tlist)
            G.add_edges_from([
                (tlist[r], tlist[c], {'weight': float(rho)})
                for r, c, rho in zip(ri[emask], ci[emask], rhos[emask])
            ])

            iso = [n for n in G.nodes() if G.degree(n) == 0]
            Gc  = G.copy()
            Gc.remove_nodes_from(iso)

            if Gc.number_of_nodes() < 2:
                self.weekly_clusters[ws] = {}
                self.weekly_leaders[ws]  = {}
            else:
                comms = nx.community.louvain_communities(
                    Gc, weight='weight', seed=42
                )
                clusters, leaders = {}, {}
                for idx, comm in enumerate(sorted(comms, key=len, reverse=True)):
                    members = sorted(comm)
                    clusters[idx] = members
                    leaders[idx]  = max(
                        members, key=lambda t: self.avg_volume.get(t, 0)
                    )
                self.weekly_clusters[ws] = clusters
                self.weekly_leaders[ws]  = leaders

            # --- efficient tickers → correlation flag ---
            ew = eff_ret[eff_ret.index <= wd].tail(CORR_WINDOW)
            ve = ew.columns[ew.notna().sum() >= 20]
            if len(ve) < 2:
                self.high_corr_flag[ws] = False
            else:
                ec     = ew[ve].corr().values
                r2, c2 = np.triu_indices(len(ve), k=1)
                total  = len(r2)
                high   = (ec[r2, c2] > 0.85).sum()
                self.high_corr_flag[ws] = (high / total) >= 0.50

        # ---- LIST 4 & 5: per-ticker SAV, ATR, EMA (vectorised) ----
        print("  [3/4] SAV, ATR, EMA for inefficient tickers...")
        ip = prices[prices['ticker'].isin(self.inefficient_tickers)].copy()

        # 20-day rolling SAV
        ip['vol_med'] = ip.groupby('ticker')['volume'].transform(
            lambda x: x.rolling(20, min_periods=10).median()
        )
        ip['vol_std'] = ip.groupby('ticker')['volume'].transform(
            lambda x: x.rolling(20, min_periods=10).std()
        )
        ip['sav'] = (ip['volume'] - ip['vol_med']) / ip['vol_std']

        # 14-day ATR
        ip['prev_close'] = ip.groupby('ticker')['close'].shift(1)
        ip['tr'] = np.maximum(
            ip['high'] - ip['low'],
            np.maximum(
                np.abs(ip['high'] - ip['prev_close']),
                np.abs(ip['low']  - ip['prev_close'])
            )
        )
        ip['atr_14'] = ip.groupby('ticker')['tr'].transform(
            lambda x: x.rolling(14, min_periods=7).mean()
        )

        # 20-day EMA
        ip['ema_20'] = ip.groupby('ticker')['close'].transform(
            lambda x: x.ewm(span=20, adjust=False).mean()
        )

        result_df = ip[['ticker', 'date', 'close', 'low', 'high',
                         'volume', 'sav', 'atr_14', 'ema_20']].copy()
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')

        print(f"  [4/4] Analytics ready: {len(result_df):,} rows, "
              f"{result_df['ticker'].nunique()} tickers")
        return result_df

    # ==============================================================
    # 3. LLM ANALYSIS — split transcript, FinBERT confidence score
    # ==============================================================
    def llm_analysis(self, ticker, transcript, date):
        if transcript is None or self.finbert_pipeline is None:
            return None

        date_str = (date if isinstance(date, str)
                    else pd.Timestamp(date).strftime('%Y-%m-%d'))
        cache_key = (ticker, date_str)
        if cache_key in self.sentiment_cache:
            return {'confidence': self.sentiment_cache[cache_key]}

        pattern = re.compile(
            r'(question[- ]and[- ]answer session|q&a session|q n a)',
            re.IGNORECASE
        )
        match = pattern.search(transcript)

        if match:
            prepared = transcript[:match.start()][-2000:]
            qa       = transcript[match.start():][:2000]
        else:
            prepared = transcript[:2000]
            qa       = transcript[-2000:]

        try:
            res_p = self.finbert_pipeline(prepared, top_k=None)
            if isinstance(res_p[0], list):
                res_p = res_p[0]
            probs_p    = {r['label']: r['score'] for r in res_p}
            prep_score = probs_p.get('positive', 0) - probs_p.get('negative', 0)

            res_q = self.finbert_pipeline(qa, top_k=None)
            if isinstance(res_q[0], list):
                res_q = res_q[0]
            probs_q  = {r['label']: r['score'] for r in res_q}
            qa_score = probs_q.get('positive', 0) - probs_q.get('negative', 0)

            confidence = (prep_score + qa_score) / 2.0
        except Exception:
            confidence = 0.0

        self.sentiment_cache[cache_key] = confidence
        return {'confidence': confidence}

    # ==============================================================
    # HELPERS
    # ==============================================================
    def _build_fast_lookup(self, df):
        """Build {ticker: [(date_str, {col: val}), ...]} sorted by date."""
        df = df.sort_values(['ticker', 'date'])
        lookup = {}
        for ticker, grp in df.groupby('ticker'):
            records = grp.to_dict('records')
            lookup[ticker] = [(r['date'], r) for r in records]
        return lookup

    def _latest(self, ticker, date_str, lookup):
        """Return most recent analytics dict for ticker on or before date_str."""
        entries = lookup.get(ticker, [])
        best = None
        for d, rec in entries:
            if d <= date_str:
                best = rec
            else:
                break
        return best

    def _build_sentiment_lookup(self):
        """Organise cached sentiments by ticker for fast chronological search."""
        self._sentiment_by_ticker = defaultdict(list)
        for (ticker, ds), score in self.sentiment_cache.items():
            self._sentiment_by_ticker[ticker].append((ds, score))
        for t in self._sentiment_by_ticker:
            self._sentiment_by_ticker[t].sort()

    def _latest_sentiment(self, leader, date_str):
        """Most recent sentiment score for a leader on or before date_str."""
        best = None
        for ds, sc in self._sentiment_by_ticker.get(leader, []):
            if ds <= date_str:
                best = sc
            else:
                break
        return best

    # ==============================================================
    # 4. MAKE DECISION (stub — logic lives in evaluate)
    # ==============================================================
    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        return 'HOLD'

    # ==============================================================
    # 5. EVALUATE — custom backtest loop with dynamic bet sizing
    # ==============================================================
    def evaluate(self, verbose=False):
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        print("\n=== Enhanced Strategy Evaluation ===")

        # ---- step 1: compute all analytics ----
        analytics_df = self.calculate_analytics(self.prices)
        lookup       = self._build_fast_lookup(analytics_df)

        # ---- step 2: pre-compute leader sentiments ----
        print("Pre-computing leader sentiments...")
        all_leaders = set()
        for leaders_dict in self.weekly_leaders.values():
            all_leaders.update(leaders_dict.values())

        leader_earn = (
            self.earnings[self.earnings['ticker'].isin(all_leaders)]
            .sort_values(['ticker', 'date'])
        )
        tickers_arr = leader_earn['ticker'].values
        transcripts_arr = leader_earn['transcript'].tolist()
        dates_arr = leader_earn['date'].values

        for i in tqdm(range(len(leader_earn)), desc="  Sentiments"):
            self.llm_analysis(
                tickers_arr[i],
                transcripts_arr[i],
                pd.Timestamp(dates_arr[i]).strftime('%Y-%m-%d')
            )
        self._build_sentiment_lookup()

        # ---- step 3: simulation loop ----
        print("Running backtest...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        self.entry_dates = {}
        portfolio_history = []
        BASE_TARGET = 5000

        for i, week_date in enumerate(sim.weekly_schedule):
            if verbose and i % 20 == 0:
                print(f"  Week {i+1}/{len(sim.weekly_schedule)}: {week_date}")

            current_prices = sim._get_current_prices(week_date)
            clusters  = self.weekly_clusters.get(week_date, {})
            leaders   = self.weekly_leaders.get(week_date, {})
            high_corr = self.high_corr_flag.get(week_date, False)

            # ==================== EXITS ====================
            for ticker in list(self.entry_dates.keys()):
                # Position already closed externally
                if ticker not in sim.portfolio.positions:
                    del self.entry_dates[ticker]
                    continue

                price = sim._get_price_on_date(ticker, week_date)
                if price is None:
                    continue

                entry = self.entry_dates[ticker]

                # Take-profit (1:3 R:R)
                if price >= entry['tp']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # Stop-loss (low_at_entry − 1.5 × ATR)
                if price <= entry['sl']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # EMA exit — close falls below EMA at date of entry
                if price < entry['ema']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # High-correlation regime — exit all positions
                if high_corr:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

            # ==================== ENTRIES ====================
            for cid, leader in leaders.items():
                la = self._latest(leader, week_date, lookup)
                if la is None:
                    continue

                l_close = la.get('close', 0)
                l_ema   = la.get('ema_20', 0)
                l_sav   = la.get('sav', 0)

                if l_ema <= 0:
                    continue
                if isinstance(l_sav, float) and np.isnan(l_sav):
                    continue

                # Entry signal: leader close > EMA  AND  SAV > 2
                if l_close > l_ema and l_sav > 2:

                    # --- sentiment-based bet sizing ---
                    mult = 1.0
                    sent = self._latest_sentiment(leader, week_date)
                    if sent is not None:
                        if sent > 0.5:
                            mult = 1.5
                        elif sent < 0.3:
                            mult = 0.5

                    target = BASE_TARGET * mult

                    # Buy every ticker in the cluster
                    for ticker in clusters.get(cid, []):
                        if ticker in sim.portfolio.positions:
                            continue

                        price = sim._get_price_on_date(ticker, week_date)
                        if price is None or price <= 0:
                            continue

                        ta = self._latest(ticker, week_date, lookup)
                        if ta is None:
                            continue

                        atr = ta.get('atr_14', 0)
                        low = ta.get('low', price)
                        ema = ta.get('ema_20', price)

                        if not atr or np.isnan(atr) or atr <= 0:
                            continue

                        # Stop-loss: entry-day low − 1.5 × ATR
                        sl   = low - 1.5 * atr
                        # Risk per share
                        risk = price - sl
                        if risk <= 0:
                            continue
                        # Take-profit: 1:3 risk-reward
                        tp = price + 3.0 * risk

                        sim.portfolio.buy_target(
                            ticker, price, week_date, target_value=target
                        )

                        # Record entry only if the buy actually went through
                        if ticker in sim.portfolio.positions:
                            self.entry_dates[ticker] = {
                                'date':  week_date,
                                'price': price,
                                'low':   low,
                                'atr':   atr,
                                'ema':   ema,
                                'tp':    tp,
                                'sl':    sl,
                            }

            # ---- record portfolio snapshot ----
            portfolio_history.append({
                'date':            week_date,
                'portfolio_value': sim.portfolio.get_value(current_prices),
                'cash':            sim.portfolio.cash,
                'positions':       len(sim.portfolio.positions),
            })

        # ---- return results in the expected format ----
        final_date   = sim.weekly_schedule[-1]
        final_prices = sim._get_current_prices(final_date)
        return {
            'trades':            sim.portfolio.trades,
            'portfolio_history': portfolio_history,
            'final_portfolio':   sim.portfolio.get_state(final_prices),
            'final_prices':      final_prices,
        }


print("EnhancedStrategy class loaded")

# Enchanced Strat V2 (20MA Crossover w trailing stop loss)

In [ ]:
# ============================================================
# CELL 13: Enhanced Strategy Implementation
# ============================================================

import re
import networkx as nx
from statsmodels.stats.diagnostic import acorr_ljungbox
from tqdm import tqdm

print("\n" + "="*70)
print("ENHANCED STRATEGY")
print("="*70)


class EnhancedStrategy(BaseStrategy):

    def __init__(self, finbert_pipeline=None):
        super().__init__(finbert_pipeline)
        self.entry_dates = {}                    # {ticker: {date, price, low, atr, ema, tp, sl}}
        self.inefficient_tickers = []
        self.efficient_tickers = []
        self.weekly_clusters = {}                # {date_str: {cid: [tickers]}}
        self.weekly_leaders = {}                 # {date_str: {cid: leader_ticker}}
        self.high_corr_flag = {}                 # {date_str: bool}
        self.avg_volume = {}                     # {ticker: float}
        self.sentiment_cache = {}                # {(ticker, date_str): float}
        self._sentiment_by_ticker = defaultdict(list)

    # ==============================================================
    # 1. DATA CLEANING
    # ==============================================================
    def clean_data(self, prices_df, earnings_df):
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]

        # Drop tickers with no earnings calls
        tickers_with_earnings = set(earnings['ticker'].unique())
        prices = prices[prices['ticker'].isin(tickers_with_earnings)]

        # Forward-fill close prices per ticker
        prices = prices.sort_values(['ticker', 'date'])
        prices['close'] = prices.groupby('ticker')['close'].ffill()

        return prices, earnings

    # ==============================================================
    # 2. ANALYTICS — Ljung-Box, Louvain, Correlation Flag, SAV, ATR, EMA
    # ==============================================================
    def calculate_analytics(self, prices_df):
        print("Computing enhanced analytics...")
        prices = prices_df.copy().sort_values(['ticker', 'date'])

        # ---- vectorised log returns ----
        prices['log_return'] = prices.groupby('ticker')['close'].transform(
            lambda x: np.log(x / x.shift(1))
        )

        # ---- LIST 1: Ljung-Box → inefficient / efficient tickers ----
        print("  [1/4] Ljung-Box test for market efficiency...")
        lb_pvals = {}
        for ticker, group in prices.groupby('ticker')['log_return']:
            lr = group.dropna()
            if len(lr) < 50:
                continue
            try:
                res = acorr_ljungbox(lr, lags=[20], return_df=True)
                lb_pvals[ticker] = res['lb_pvalue'].iloc[0]
            except Exception:
                continue

        alpha = 0.05
        self.inefficient_tickers = [t for t, p in lb_pvals.items() if p < alpha]
        self.efficient_tickers   = [t for t, p in lb_pvals.items() if p >= alpha]
        print(f"    Inefficient: {len(self.inefficient_tickers)}, "
              f"Efficient: {len(self.efficient_tickers)}")

        # ---- average volume for leader identification ----
        self.avg_volume = (
            prices[prices['ticker'].isin(self.inefficient_tickers)]
            .groupby('ticker')['volume'].mean().to_dict()
        )

        # ---- LIST 2 & 3 + BOOLEAN: weekly Louvain clusters,
        #      leaders, efficient-ticker correlation flag ----
        print("  [2/4] Weekly Louvain clustering & correlation regime flag...")
        returns_pivot = prices.pivot_table(
            index='date', columns='ticker', values='log_return'
        )
        ineff_cols = [t for t in self.inefficient_tickers if t in returns_pivot.columns]
        eff_cols   = [t for t in self.efficient_tickers   if t in returns_pivot.columns]
        ineff_ret  = returns_pivot[ineff_cols]
        eff_ret    = returns_pivot[eff_cols]

        weekly_dates = pd.date_range(
            prices['date'].min(), prices['date'].max(), freq='W-FRI'
        )
        CORR_WINDOW    = 60
        EDGE_THRESHOLD = 0.6

        for wd in tqdm(weekly_dates, desc="    Weeks"):
            ws = wd.strftime('%Y-%m-%d')

            # --- inefficient tickers → Louvain ---
            window = ineff_ret[ineff_ret.index <= wd].tail(CORR_WINDOW)
            valid  = window.columns[window.notna().sum() >= 20]

            if len(valid) < 3:
                self.weekly_clusters[ws] = {}
                self.weekly_leaders[ws]  = {}
                self.high_corr_flag[ws]  = False
                continue

            corr_m = window[valid].corr()
            tlist  = corr_m.columns.tolist()
            cvals  = corr_m.values
            ri, ci = np.triu_indices(len(tlist), k=1)
            rhos   = cvals[ri, ci]
            emask  = rhos >= EDGE_THRESHOLD

            G = nx.Graph()
            G.add_nodes_from(tlist)
            G.add_edges_from([
                (tlist[r], tlist[c], {'weight': float(rho)})
                for r, c, rho in zip(ri[emask], ci[emask], rhos[emask])
            ])

            iso = [n for n in G.nodes() if G.degree(n) == 0]
            Gc  = G.copy()
            Gc.remove_nodes_from(iso)

            if Gc.number_of_nodes() < 2:
                self.weekly_clusters[ws] = {}
                self.weekly_leaders[ws]  = {}
            else:
                comms = nx.community.louvain_communities(
                    Gc, weight='weight', seed=42
                )
                clusters, leaders = {}, {}
                for idx, comm in enumerate(sorted(comms, key=len, reverse=True)):
                    members = sorted(comm)
                    clusters[idx] = members
                    leaders[idx]  = max(
                        members, key=lambda t: self.avg_volume.get(t, 0)
                    )
                self.weekly_clusters[ws] = clusters
                self.weekly_leaders[ws]  = leaders

            # --- efficient tickers → correlation flag ---
            ew = eff_ret[eff_ret.index <= wd].tail(CORR_WINDOW)
            ve = ew.columns[ew.notna().sum() >= 20]
            if len(ve) < 2:
                self.high_corr_flag[ws] = False
            else:
                ec     = ew[ve].corr().values
                r2, c2 = np.triu_indices(len(ve), k=1)
                total  = len(r2)
                high   = (ec[r2, c2] > 0.85).sum()
                self.high_corr_flag[ws] = (high / total) >= 0.50

        # ---- LIST 4 & 5: per-ticker SAV, ATR, EMA (vectorised) ----
        print("  [3/4] SAV, ATR, EMA for inefficient tickers...")
        ip = prices[prices['ticker'].isin(self.inefficient_tickers)].copy()

        # 20-day rolling SAV
        ip['vol_med'] = ip.groupby('ticker')['volume'].transform(
            lambda x: x.rolling(20, min_periods=10).median()
        )
        ip['vol_std'] = ip.groupby('ticker')['volume'].transform(
            lambda x: x.rolling(20, min_periods=10).std()
        )
        ip['sav'] = (ip['volume'] - ip['vol_med']) / ip['vol_std']

        # 14-day ATR
        ip['prev_close'] = ip.groupby('ticker')['close'].shift(1)
        ip['tr'] = np.maximum(
            ip['high'] - ip['low'],
            np.maximum(
                np.abs(ip['high'] - ip['prev_close']),
                np.abs(ip['low']  - ip['prev_close'])
            )
        )
        ip['atr_14'] = ip.groupby('ticker')['tr'].transform(
            lambda x: x.rolling(14, min_periods=7).mean()
        )

        # 20-day EMA
        ip['ema_20'] = ip.groupby('ticker')['close'].transform(
            lambda x: x.ewm(span=20, adjust=False).mean()
        )

        result_df = ip[['ticker', 'date', 'close', 'low', 'high',
                         'volume', 'sav', 'atr_14', 'ema_20']].copy()
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')

        print(f"  [4/4] Analytics ready: {len(result_df):,} rows, "
              f"{result_df['ticker'].nunique()} tickers")
        return result_df

    # ==============================================================
    # 3. LLM ANALYSIS — split transcript, FinBERT confidence score
    # ==============================================================
    def llm_analysis(self, ticker, transcript, date):
        if transcript is None or self.finbert_pipeline is None:
            return None

        date_str = (date if isinstance(date, str)
                    else pd.Timestamp(date).strftime('%Y-%m-%d'))
        cache_key = (ticker, date_str)
        if cache_key in self.sentiment_cache:
            return {'confidence': self.sentiment_cache[cache_key]}

        pattern = re.compile(
            r'(question[- ]and[- ]answer session|q&a session|q n a)',
            re.IGNORECASE
        )
        match = pattern.search(transcript)

        if match:
            prepared = transcript[:match.start()][-2000:]
            qa       = transcript[match.start():][:2000]
        else:
            prepared = transcript[:2000]
            qa       = transcript[-2000:]

        try:
            res_p = self.finbert_pipeline(prepared, top_k=None)
            if isinstance(res_p[0], list):
                res_p = res_p[0]
            probs_p    = {r['label']: r['score'] for r in res_p}
            prep_score = probs_p.get('positive', 0) - probs_p.get('negative', 0)

            res_q = self.finbert_pipeline(qa, top_k=None)
            if isinstance(res_q[0], list):
                res_q = res_q[0]
            probs_q  = {r['label']: r['score'] for r in res_q}
            qa_score = probs_q.get('positive', 0) - probs_q.get('negative', 0)

            confidence = (prep_score + qa_score) / 2.0
        except Exception:
            confidence = 0.0

        self.sentiment_cache[cache_key] = confidence
        return {'confidence': confidence}

    # ==============================================================
    # HELPERS
    # ==============================================================
    def _build_fast_lookup(self, df):
        """Build {ticker: [(date_str, {col: val}), ...]} sorted by date."""
        df = df.sort_values(['ticker', 'date'])
        lookup = {}
        for ticker, grp in df.groupby('ticker'):
            records = grp.to_dict('records')
            lookup[ticker] = [(r['date'], r) for r in records]
        return lookup

    def _latest(self, ticker, date_str, lookup):
        """Return most recent analytics dict for ticker on or before date_str."""
        entries = lookup.get(ticker, [])
        best = None
        for d, rec in entries:
            if d <= date_str:
                best = rec
            else:
                break
        return best

    def _build_sentiment_lookup(self):
        """Organise cached sentiments by ticker for fast chronological search."""
        self._sentiment_by_ticker = defaultdict(list)
        for (ticker, ds), score in self.sentiment_cache.items():
            self._sentiment_by_ticker[ticker].append((ds, score))
        for t in self._sentiment_by_ticker:
            self._sentiment_by_ticker[t].sort()

    def _latest_sentiment(self, leader, date_str):
        """Most recent sentiment score for a leader on or before date_str."""
        best = None
        for ds, sc in self._sentiment_by_ticker.get(leader, []):
            if ds <= date_str:
                best = sc
            else:
                break
        return best

    # ==============================================================
    # 4. MAKE DECISION (stub — logic lives in evaluate)
    # ==============================================================
    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        return 'HOLD'

    # ==============================================================
    # 5. EVALUATE — custom backtest loop with dynamic bet sizing
    # ==============================================================
    def evaluate(self, verbose=False):
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        print("\n=== Enhanced Strategy Evaluation ===")

        # ---- step 1: compute all analytics ----
        analytics_df = self.calculate_analytics(self.prices)
        lookup       = self._build_fast_lookup(analytics_df)

        # ---- step 2: pre-compute leader sentiments ----
        print("Pre-computing leader sentiments...")
        all_leaders = set()
        for leaders_dict in self.weekly_leaders.values():
            all_leaders.update(leaders_dict.values())

        leader_earn = (
            self.earnings[self.earnings['ticker'].isin(all_leaders)]
            .sort_values(['ticker', 'date'])
        )
        tickers_arr = leader_earn['ticker'].values
        transcripts_arr = leader_earn['transcript'].tolist()
        dates_arr = leader_earn['date'].values

        for i in tqdm(range(len(leader_earn)), desc="  Sentiments"):
            self.llm_analysis(
                tickers_arr[i],
                transcripts_arr[i],
                pd.Timestamp(dates_arr[i]).strftime('%Y-%m-%d')
            )
        self._build_sentiment_lookup()

        # ---- step 3: simulation loop ----
        print("Running backtest...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        self.entry_dates = {}
        portfolio_history = []
        BASE_TARGET = 5000

        for i, week_date in enumerate(sim.weekly_schedule):
            if verbose and i % 20 == 0:
                print(f"  Week {i+1}/{len(sim.weekly_schedule)}: {week_date}")

            current_prices = sim._get_current_prices(week_date)
            clusters  = self.weekly_clusters.get(week_date, {})
            leaders   = self.weekly_leaders.get(week_date, {})
            high_corr = self.high_corr_flag.get(week_date, False)

            # ==================== EXITS ====================
            for ticker in list(self.entry_dates.keys()):
                # Position already closed externally
                if ticker not in sim.portfolio.positions:
                    del self.entry_dates[ticker]
                    continue

                price = sim._get_price_on_date(ticker, week_date)
                if price is None:
                    continue

                entry = self.entry_dates[ticker]

                # Update trailing stop: ratchet upward using current ATR
                ta_now = self._latest(ticker, week_date, lookup)
                if ta_now is not None:
                    cur_atr = ta_now.get('atr_14', entry['atr'])
                    if cur_atr and not np.isnan(cur_atr) and cur_atr > 0:
                        new_sl = price - 1.5 * cur_atr
                        if new_sl > entry['sl']:
                            entry['sl'] = new_sl

                # Take-profit (1:3 R:R)
                if price >= entry['tp']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # Trailing stop-loss
                if price <= entry['sl']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # EMA exit — close falls below EMA at date of entry
                if price < entry['ema']:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

                # High-correlation regime — exit all positions
                if high_corr:
                    sim.portfolio.sell(ticker, price, week_date)
                    del self.entry_dates[ticker]
                    continue

            # ==================== ENTRIES ====================
            for cid, leader in leaders.items():
                la = self._latest(leader, week_date, lookup)
                if la is None:
                    continue

                l_close = la.get('close', 0)
                l_ema   = la.get('ema_20', 0)
                l_sav   = la.get('sav', 0)

                if l_ema <= 0:
                    continue
                if isinstance(l_sav, float) and np.isnan(l_sav):
                    continue

                # Entry signal: leader close > EMA  AND  SAV > 2
                if l_close > l_ema and l_sav > 2:

                    # --- sentiment-based bet sizing ---
                    mult = 1.0
                    sent = self._latest_sentiment(leader, week_date)
                    if sent is not None:
                        if sent > 0.5:
                            mult = 2
                        elif sent < 0.3:
                            mult = 0.5

                    target = BASE_TARGET * mult

                    # Buy every ticker in the cluster
                    for ticker in clusters.get(cid, []):
                        if ticker in sim.portfolio.positions:
                            continue

                        price = sim._get_price_on_date(ticker, week_date)
                        if price is None or price <= 0:
                            continue

                        ta = self._latest(ticker, week_date, lookup)
                        if ta is None:
                            continue

                        atr = ta.get('atr_14', 0)
                        low = ta.get('low', price)
                        ema = ta.get('ema_20', price)

                        if not atr or np.isnan(atr) or atr <= 0:
                            continue

                        # Stop-loss: entry-day low − 1.5 × ATR
                        sl   = low - 1.5 * atr
                        # Risk per share
                        risk = price - sl
                        if risk <= 0:
                            continue
                        # Take-profit: 1:3 risk-reward
                        tp = price + 3.0 * risk

                        sim.portfolio.buy_target(
                            ticker, price, week_date, target_value=target
                        )

                        # Record entry only if the buy actually went through
                        if ticker in sim.portfolio.positions:
                            self.entry_dates[ticker] = {
                                'date':  week_date,
                                'price': price,
                                'low':   low,
                                'atr':   atr,
                                'ema':   ema,
                                'tp':    tp,
                                'sl':    sl,
                            }

            # ---- record portfolio snapshot ----
            portfolio_history.append({
                'date':            week_date,
                'portfolio_value': sim.portfolio.get_value(current_prices),
                'cash':            sim.portfolio.cash,
                'positions':       len(sim.portfolio.positions),
            })

        # ---- return results in the expected format ----
        final_date   = sim.weekly_schedule[-1]
        final_prices = sim._get_current_prices(final_date)
        return {
            'trades':            sim.portfolio.trades,
            'portfolio_history': portfolio_history,
            'final_portfolio':   sim.portfolio.get_state(final_prices),
            'final_prices':      final_prices,
        }


print("EnhancedStrategy class loaded")

# Enchanced Strat V3 (AMIHUD)

In [ ]:
"""
EnhancedStrategy: Multi-Sleeve Trading Strategy
=================================================
Extends BaseStrategy with:
  - Step 1: Universe Selection via Volatility + Amihud Illiquidity composite scoring (monthly)
  - Step 2: FinBERT Sentiment Acceleration (Earnings NLP)
  - Step 3: Dual-sleeve entry triggers (RSI Mean Reversion + Volume Momentum / Sentiment Acceleration)
  - Step 4: Time-based and trailing-stop exits
  - Step 5: Dynamic capital allocation with Leverage Effect sizing
"""

import re
import numpy as np
import pandas as pd
from collections import defaultdict


class EnhancedStrategy(BaseStrategy):
    """
    Multi-Sleeve Trading Strategy.

    Sleeve 1 (Tactical Weekly): RSI mean reversion + Volume-confirmed momentum.
        - Entry: Friday close — RSI<30 OR (positive return AND volume > 1.5x 20d MA)
        - Exit: Sell the following week (approximating Monday Close exit)
        - Sizing: 60% of capital, further attenuated 0.6x on red Fridays (leverage effect)

    Sleeve 2 (Fundamental Momentum): FinBERT Earnings Sentiment Acceleration.
        - Entry: When sentiment acceleration flag is True (Q0 net sentiment > Q-1)
        - Exit: 20-30 trading days (~4-6 weekly evaluations) OR 10% trailing stop
        - Sizing: 40% of capital, equally weighted across triggered stocks
    """

    def __init__(self, finbert_pipeline=None):
        self.finbert_pipeline = finbert_pipeline
        self.llm_cache = {}
        self.llm_cache_hits = 0
        self.llm_cache_misses = 0
        self.prices = None
        self.earnings = None

        # --- Universe selection state ---
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None  # store full analytics for universe updates

        # --- Sleeve 1: Tactical Weekly tracking ---
        self.sleeve1_positions = {}  # {ticker: entry_week_date}

        # --- Sleeve 2: Fundamental Momentum tracking ---
        self.sleeve2_positions = {}  # {ticker: {'entry_date': str, 'weeks_held': int, 'peak_price': float}}

        # --- Sentiment cache for acceleration detection ---
        self.sentiment_cache = {}  # {(ticker, quarter_key): net_sentiment}

        # --- Capital allocation ---
        self.sleeve1_allocation = 0.60  # 60% to Sleeve 1
        self.sleeve2_allocation = 0.40  # 40% to Sleeve 2

    # =====================================================================
    # DATA PREPARATION
    # =====================================================================

    def set_data(self, prices_df, earnings_df):
        """Set and preprocess data for evaluation. Call this before evaluate()."""
        print("Cleaning and preprocessing data...")
        self.prices, self.earnings = self.clean_data(prices_df, earnings_df)
        print(f"Data ready: {len(self.prices):,} price records")

    def clean_data(self, prices_df, earnings_df):
        """Clean and validate data with enhanced preprocessing."""
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]

        # Sort prices for proper rolling calculations
        prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

        return prices, earnings

    # =====================================================================
    # ANALYTICS CALCULATION (Step 1 metrics + technical indicators)
    # =====================================================================

    def calculate_analytics(self, prices_df):
        """
        Calculate all technical indicators needed for the strategy:
          - daily_return, vol_60 (annualized), amihud_60
          - rsi_14, volume_ma_20, ma_50
          - open, close, volume (passed through for decision logic)
        """
        print("Computing enhanced technical indicators...")
        results = []

        for ticker in prices_df['ticker'].unique():
            df = prices_df[prices_df['ticker'] == ticker].copy().sort_values('date')

            # --- Basic returns ---
            df['daily_return'] = df['close'].pct_change()
            df['abs_return'] = df['daily_return'].abs()

            # --- Step 1 metrics: Universe Selection ---
            # Volatility: annualized 60-day rolling std of daily returns
            df['vol_60'] = df['daily_return'].rolling(60, min_periods=30).std() * np.sqrt(252)

            # Amihud Illiquidity: avg(|return| / dollar_volume) over 60 days
            df['dollar_volume'] = df['close'] * df['volume']
            # Avoid division by zero: replace 0 dollar_volume with NaN
            df['amihud_daily'] = df['abs_return'] / df['dollar_volume'].replace(0, np.nan)
            df['amihud_60'] = df['amihud_daily'].rolling(60, min_periods=30).mean()

            # --- Step 3 metrics: Entry Triggers ---
            # RSI-14
            delta = df['close'].diff()
            gain = delta.where(delta > 0, 0.0)
            loss = (-delta).where(delta < 0, 0.0)
            avg_gain = gain.rolling(14, min_periods=14).mean()
            avg_loss = loss.rolling(14, min_periods=14).mean()
            rs = avg_gain / avg_loss.replace(0, np.nan)
            df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))

            # Volume 20-day MA
            df['volume_ma_20'] = df['volume'].rolling(20, min_periods=10).mean()

            # MA-50 (from base strategy)
            df['ma_50'] = df['close'].rolling(50, min_periods=1).mean()

            results.append(df[['ticker', 'date', 'open', 'close', 'volume',
                               'daily_return', 'vol_60', 'amihud_60',
                               'rsi_14', 'volume_ma_20', 'ma_50']])

        result_df = pd.concat(results, ignore_index=True)
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')
        print(f"Enhanced indicators computed: {len(result_df):,} rows")

        # Store for universe updates
        self.universe_analytics_df = result_df

        return result_df

    # =====================================================================
    # UNIVERSE SELECTION (Step 1: Monthly filter — top 20% composite score)
    # =====================================================================

    def _update_universe(self, current_date):
        """
        Recalculate the tradable universe monthly.
        Composite Score = Volatility Rank (ascending) + Amihud Rank (descending).
        Select top 20% by composite score.
        """
        if self.universe_analytics_df is None:
            return

        # Get latest analytics per ticker up to current_date
        df = self.universe_analytics_df[self.universe_analytics_df['date'] <= current_date]
        if df.empty:
            return

        latest = df.groupby('ticker').last().reset_index()

        # Drop tickers with missing metrics
        scored = latest.dropna(subset=['vol_60', 'amihud_60'])
        if scored.empty:
            return

        # Rank volatility ascending (low vol = rank 1 = good for Low Volatility Anomaly)
        scored = scored.copy()
        scored['vol_rank'] = scored['vol_60'].rank(ascending=True, method='average')

        # Rank Amihud descending (high illiquidity = rank 1 = good for Illiquidity Premium)
        scored['amihud_rank'] = scored['amihud_60'].rank(ascending=False, method='average')

        # Composite score (higher is better)
        scored['composite_score'] = scored['vol_rank'] + scored['amihud_rank']

        # Top 20%
        threshold = scored['composite_score'].quantile(0.80)
        self.universe = set(scored[scored['composite_score'] >= threshold]['ticker'].tolist())

        month_key = current_date[:7]  # YYYY-MM
        self.universe_last_updated_month = month_key
        print(f"  Universe updated ({month_key}): {len(self.universe)} stocks selected from {len(scored)} scored")

    # =====================================================================
    # LLM / FINBERT ANALYSIS (Step 2: Sentiment Acceleration)
    # =====================================================================

    def _get_quarter_key(self, date_str):
        """Convert a date string to a quarter key like '2015-Q3'."""
        dt = pd.to_datetime(date_str)
        return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"

    def llm_analysis(self, ticker, transcript, date):
        """
        FinBERT sentiment analysis with chunking, aggregation, and acceleration detection.

        Returns:
            dict with 'sentiment', 'net_sentiment', 'sentiment_acceleration'
            or None if no transcript/pipeline
        """
        if transcript is None or self.finbert_pipeline is None:
            return None

        # --- Cache check ---
        cache_key = f"{ticker}_{date}"
        if cache_key in self.llm_cache:
            self.llm_cache_hits += 1
            return self.llm_cache[cache_key]
        self.llm_cache_misses += 1

        try:
            # --- Chunking: split into sentences ---
            sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())
            # Group sentences into chunks of ~400 chars to stay within 512 token limit
            chunks = []
            current_chunk = ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) > 400:
                    if current_chunk:
                        chunks.append(current_chunk.strip())
                    current_chunk = sentence
                else:
                    current_chunk += " " + sentence
            if current_chunk.strip():
                chunks.append(current_chunk.strip())

            if not chunks:
                return None

            # --- Inference + Labeling ---
            positive_count = 0
            negative_count = 0
            neutral_count = 0

            for chunk in chunks:
                if not chunk or len(chunk) < 10:
                    continue
                try:
                    result = self.finbert_pipeline(chunk[:512])  # truncate to token limit
                    label = result[0]['label'].lower() if isinstance(result, list) and len(result) > 0 else 'neutral'
                    if isinstance(label, str):
                        if 'positive' in label:
                            positive_count += 1
                        elif 'negative' in label:
                            negative_count += 1
                        else:
                            neutral_count += 1
                except Exception:
                    neutral_count += 1

            total_valid = positive_count + negative_count + neutral_count
            if total_valid == 0:
                return None

            # --- Aggregation: Net Sentiment ---
            net_sentiment = (positive_count - negative_count) / total_valid

            # Dominant label
            if positive_count >= negative_count and positive_count >= neutral_count:
                dominant = 'positive'
            elif negative_count >= positive_count and negative_count >= neutral_count:
                dominant = 'negative'
            else:
                dominant = 'neutral'

            # --- Sentiment Acceleration ---
            quarter_key = self._get_quarter_key(date)
            self.sentiment_cache[(ticker, quarter_key)] = net_sentiment

            # Find previous quarter
            dt = pd.to_datetime(date)
            prev_quarter_dt = dt - pd.DateOffset(months=3)
            prev_quarter_key = f"{prev_quarter_dt.year}-Q{(prev_quarter_dt.month - 1) // 3 + 1}"

            prev_sentiment = self.sentiment_cache.get((ticker, prev_quarter_key))
            sentiment_acceleration = False
            if prev_sentiment is not None:
                sentiment_acceleration = (net_sentiment > prev_sentiment)

            result = {
                'sentiment': dominant,
                'net_sentiment': net_sentiment,
                'sentiment_acceleration': sentiment_acceleration,
                'positive_ratio': positive_count / total_valid if total_valid > 0 else 0,
                'chunks_processed': total_valid
            }

            self.llm_cache[cache_key] = result
            return result

        except Exception:
            return None

    # =====================================================================
    # DECISION LOGIC (Steps 3-5)
    # =====================================================================

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        """
        Multi-sleeve trading decision.

        Sleeve 1 (Tactical Weekly):
            Entry: RSI < 30 OR (positive return AND volume > 1.5x 20d MA) on Friday.
            Exit: Sell the following week.
            Sizing: 60% capital, 0.6x attenuation on red Fridays.

        Sleeve 2 (Fundamental Momentum):
            Entry: Sentiment Acceleration flag is True.
            Exit: ~5 weekly evaluations OR 10% trailing stop.
            Sizing: 40% capital, equally weighted.
        """
        price = analytics.get('close', 0)
        if price <= 0:
            return 'HOLD'

        has_position = ticker in portfolio_state.get('positions', {})

        # --- Step 0: Monthly universe update ---
        current_month = date[:7]
        if self.universe_last_updated_month != current_month:
            self._update_universe(date)

        # --- Step 2: Sleeve 1 EXIT — sell after 1 week holding ---
        if ticker in self.sleeve1_positions:
            pos_info = self.sleeve1_positions[ticker]
            entry_date = pos_info['entry_date']

            # Update peak price for the trailing stop
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            if price <= pos_info['peak_price'] * 0.93:
                del self.sleeve1_positions[ticker]
                return 'SELL'

            if date > entry_date:
                # Time to exit: sell on this week's Friday (approximating Monday Close)
                del self.sleeve1_positions[ticker]
                return 'SELL'

        # --- Step 3: Sleeve 2 EXIT — time-based or trailing stop ---
        if ticker in self.sleeve2_positions:
            pos_info = self.sleeve2_positions[ticker]
            pos_info['weeks_held'] += 1

            # Update peak price for trailing stop
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            # 10% trailing stop
            if price <= pos_info['peak_price'] * 0.93:
                del self.sleeve2_positions[ticker]
                return 'SELL'

            # Time-based exit: 5 weekly evaluations ≈ 25 trading days
            if pos_info['weeks_held'] >= 5:
                del self.sleeve2_positions[ticker]
                return 'SELL'

            return 'HOLD'

        # --- If ticker not in universe, don't open new positions ---
        in_universe = ticker in self.universe

        # If we have a position but ticker left the universe, sell it
        if has_position and not in_universe:
            # Clean up tracking if somehow in neither sleeve dict
            if ticker not in self.sleeve1_positions and ticker not in self.sleeve2_positions:
                return 'SELL'

        if not in_universe:
            return 'HOLD'

        # --- Don't enter if already holding this ticker ---
        if has_position:
            return 'HOLD'

        # --- Step 4: Sleeve 1 ENTRY (Tactical Weekly) ---
        rsi = analytics.get('rsi_14')
        daily_ret = analytics.get('daily_return')
        volume = analytics.get('volume')
        volume_ma = analytics.get('volume_ma_20')

        sleeve1_trigger = False
        if rsi is not None and not np.isnan(rsi):
            # Trigger A: RSI Mean Reversion
            if rsi < 30:
                sleeve1_trigger = True

        if not sleeve1_trigger and daily_ret is not None and volume is not None and volume_ma is not None:
            if not np.isnan(daily_ret) and not np.isnan(volume) and not np.isnan(volume_ma):
                # Trigger B: Volume-Confirmed Momentum
                if daily_ret > 0 and volume_ma > 0 and volume > 1.25 * volume_ma:
                    sleeve1_trigger = True

        if sleeve1_trigger:
            self.sleeve1_positions[ticker] = {
            'entry_date': date,
            'peak_price': price
            }
            return 'BUY'

        # --- Step 5: Sleeve 2 ENTRY (Fundamental Momentum / Sentiment Acceleration) ---
        if transcript is not None:
            sentiment_result = self.llm_analysis(ticker, transcript, date)
            if sentiment_result is not None and sentiment_result.get('sentiment_acceleration', False):
                self.sleeve2_positions[ticker] = {
                    'entry_date': date,
                    'weeks_held': 0,
                    'peak_price': price
                }
                return 'BUY'

        return 'HOLD'

    # =====================================================================
    # EVALUATION ORCHESTRATION
    # =====================================================================

    def _build_analytics_lookup(self, analytics_df):
        """Build O(1) lookup dict from analytics DataFrame."""
        lookup = defaultdict(list)
        for _, row in analytics_df.iterrows():
            lookup[row['ticker']].append((row['date'], row.to_dict()))
        for ticker in lookup:
            lookup[ticker].sort(key=lambda x: x[0])
        return lookup

    def evaluate(self, verbose=False):
        """Evaluate strategy, resetting sleeve state for clean run."""
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        # Reset sleeve tracking state for fresh evaluation
        self.sleeve1_positions = {}
        self.sleeve2_positions = {}
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None
        # Note: We keep sentiment_cache — it accumulates knowledge across the backtest

        print("Running evaluation...")

        # Calculate analytics
        analytics = self.calculate_analytics(self.prices)
        analytics_lookup = self._build_analytics_lookup(analytics)

        # Run backtest
        print("Running backtest simulation...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        results = sim.run(
            lambda t, d, tr, ps, a: self.make_decision(t, d, tr, ps, a),
            analytics_lookup, verbose
        )

        return results


print("EnhancedStrategy class loaded")


# Enchanced Strat V4 (AMIHUD + Market Breadth Drawdown management)

In [ ]:
"""
EnhancedStrategy: Multi-Sleeve Trading Strategy (Black Swan Defensive Edition)
=============================================================================
Extends BaseStrategy with:
  - Step 1: Universe Selection via Volatility + Amihud composite scoring (monthly)
  - Step 2: FinBERT Sentiment Acceleration (Earnings NLP)
  - Step 3: Dual-sleeve entry triggers (RSI < 35 + Volume > 1.25x MA)
  - Step 4: Chandelier Exits (3x ATR) and time-based exits
  - Step 5: Master Regime Filter, Volatility Acceleration Vetoes, and Systemic Circuit Breakers
"""

import re
import numpy as np
import pandas as pd
from collections import defaultdict

class EnhancedStrategy(BaseStrategy):
    """
    Multi-Sleeve Trading Strategy with robust drawdown defense.
    """

    def __init__(self, finbert_pipeline=None):
        self.finbert_pipeline = finbert_pipeline
        self.llm_cache = {}
        self.llm_cache_hits = 0
        self.llm_cache_misses = 0
        self.prices = None
        self.earnings = None

        # --- Universe selection state ---
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None  
        self.momentum_winners = set() # H1: Track Top Decile Winners to avoid skewness

        # --- Sleeve tracking ---
        self.sleeve1_positions = {}  
        self.sleeve2_positions = {}  

        # --- Sentiment cache ---
        self.sentiment_cache = {}  

        # --- Capital allocation ---
        self.sleeve1_allocation = 0.60  
        self.sleeve2_allocation = 0.40  

    # =====================================================================
    # DATA PREPARATION
    # =====================================================================

    def set_data(self, prices_df, earnings_df):
        print("Cleaning and preprocessing data...")
        self.prices, self.earnings = self.clean_data(prices_df, earnings_df)
        print(f"Data ready: {len(self.prices):,} price records")

    def clean_data(self, prices_df, earnings_df):
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]

        prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)
        return prices, earnings

    # =====================================================================
    # ANALYTICS CALCULATION (Including Market Regimes & ATR)
    # =====================================================================

    def calculate_analytics(self, prices_df):
        print("Computing advanced technicals and market regimes...")
        results = []

        for ticker in prices_df['ticker'].unique():
            df = prices_df[prices_df['ticker'] == ticker].copy().sort_values('date')

            # --- Basic returns & Moving Averages ---
            df['daily_return'] = df['close'].pct_change()
            df['abs_return'] = df['daily_return'].abs()
            df['ma_200'] = df['close'].rolling(200, min_periods=50).mean() # H7: Trend Filter
            df['ma_50'] = df['close'].rolling(50, min_periods=1).mean()
            df['ret_12m'] = df['close'].pct_change(252) # H1: 12-Month Momentum

            # --- Volatility metrics (H3 & Universe) ---
            df['vol_60'] = df['daily_return'].rolling(60, min_periods=30).std() * np.sqrt(252)
            df['vol_10'] = df['daily_return'].rolling(10, min_periods=5).std() * np.sqrt(252) # H3: Vol Acceleration

            # --- Amihud Illiquidity (H4 & H5) ---
            df['dollar_volume'] = df['close'] * df['volume']
            df['amihud_daily'] = df['abs_return'] / df['dollar_volume'].replace(0, np.nan)
            df['amihud_60'] = df['amihud_daily'].rolling(60, min_periods=30).mean()

            # --- ATR Calculation (Chandelier Exit) ---
            df['prev_close'] = df['close'].shift()
            if 'high' in df.columns and 'low' in df.columns:
                tr1 = df['high'] - df['low']
                tr2 = (df['high'] - df['prev_close']).abs()
                tr3 = (df['low'] - df['prev_close']).abs()
                df['tr'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
            else:
                df['tr'] = (df['close'] - df['prev_close']).abs()
            df['atr_14'] = df['tr'].rolling(14, min_periods=1).mean()

            # --- Entry Triggers ---
            delta = df['close'].diff()
            gain = delta.where(delta > 0, 0.0)
            loss = (-delta).where(delta < 0, 0.0)
            avg_gain = gain.rolling(14, min_periods=14).mean()
            avg_loss = loss.rolling(14, min_periods=14).mean()
            rs = avg_gain / avg_loss.replace(0, np.nan)
            df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))
            df['volume_ma_20'] = df['volume'].rolling(20, min_periods=10).mean()

            results.append(df[['ticker', 'date', 'open', 'close', 'volume',
                               'daily_return', 'vol_60', 'vol_10', 'amihud_60',
                               'rsi_14', 'volume_ma_20', 'ma_200', 'ret_12m', 'atr_14']])

        result_df = pd.concat(results, ignore_index=True)
        
        # --- MARKET REGIME CALCULATIONS (H2, H6) ---
        print("Calculating Systemic Market Breadth and Realized Volatility...")
        # 1. Market Breadth (% above 200 SMA)
        result_df['above_200'] = (result_df['close'] > result_df['ma_200']).astype(int)
        daily_breadth = result_df.groupby('date')['above_200'].mean().rename('market_breadth_pct')
        
        # 2. Market Realized Volatility (Equal weight synthetic index)
        daily_ret_market = result_df.groupby('date')['daily_return'].mean()
        market_vol = daily_ret_market.rolling(20, min_periods=10).std() * np.sqrt(252)
        market_vol = market_vol.rename('market_vol_20')

        # Merge Systemic indicators back to the main dataframe
        result_df = result_df.merge(daily_breadth, on='date', how='left')
        result_df = result_df.merge(market_vol, on='date', how='left')
        result_df.drop(columns=['above_200'], inplace=True)

        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')
        self.universe_analytics_df = result_df
        return result_df

    # =====================================================================
    # UNIVERSE SELECTION (Dynamic Regime Filters)
    # =====================================================================

    def _update_universe(self, current_date):
        if self.universe_analytics_df is None:
            return

        df = self.universe_analytics_df[self.universe_analytics_df['date'] <= current_date]
        if df.empty: return

        latest = df.groupby('ticker').last().reset_index()
        scored = latest.dropna(subset=['vol_60', 'amihud_60', 'ma_200']).copy()
        if scored.empty: return

        # Track Top Decile Momentum Winners (H1)
        scored['ret_12m_rank'] = scored['ret_12m'].rank(pct=True, na_option='bottom')
        self.momentum_winners = set(scored[scored['ret_12m_rank'] >= 0.90]['ticker'])

        # Get Current Market Breadth
        regime_breadth = latest['market_breadth_pct'].mean()

        # --- CORRECTION REGIME PENALTIES ---
        if regime_breadth < 0.50:
            # H7: Trend Filter (Drop stocks below 200 SMA)
            scored = scored[scored['close'] >= scored['ma_200']]
            
            # H5: Exclude High IVOL (Top 20% vol penalty)
            vol_thresh = scored['vol_60'].quantile(0.80)
            scored = scored[scored['vol_60'] <= vol_thresh]

            # H4: Exclude Severe Illiquidity (Top 20% Amihud penalty)
            amihud_thresh = scored['amihud_60'].quantile(0.80)
            scored = scored[scored['amihud_60'] <= amihud_thresh]

        # Composite score
        scored['vol_rank'] = scored['vol_60'].rank(ascending=True, method='average')
        scored['amihud_rank'] = scored['amihud_60'].rank(ascending=False, method='average')
        scored['composite_score'] = scored['vol_rank'] + scored['amihud_rank']

        # Widen Universe funnel to Top 40% (60th percentile)
        threshold = scored['composite_score'].quantile(0.60)
        self.universe = set(scored[scored['composite_score'] >= threshold]['ticker'].tolist())

        month_key = current_date[:7]
        self.universe_last_updated_month = month_key
        print(f"  Universe updated ({month_key}): {len(self.universe)} stocks selected. Breadth: {regime_breadth:.1%}")

    # =====================================================================
    # LLM / FINBERT ANALYSIS (Sentiment Acceleration)
    # =====================================================================

    def _get_quarter_key(self, date_str):
        dt = pd.to_datetime(date_str)
        return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"

    def llm_analysis(self, ticker, transcript, date):
        # ... (Existing FinBERT logic remains entirely unchanged) ...
        if transcript is None or self.finbert_pipeline is None:
            return None
        cache_key = f"{ticker}_{date}"
        if cache_key in self.llm_cache:
            self.llm_cache_hits += 1
            return self.llm_cache[cache_key]
        self.llm_cache_misses += 1

        try:
            sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())
            chunks = []
            current_chunk = ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) > 400:
                    if current_chunk: chunks.append(current_chunk.strip())
                    current_chunk = sentence
                else:
                    current_chunk += " " + sentence
            if current_chunk.strip(): chunks.append(current_chunk.strip())
            if not chunks: return None

            positive_count, negative_count, neutral_count = 0, 0, 0
            for chunk in chunks:
                if not chunk or len(chunk) < 10: continue
                try:
                    result = self.finbert_pipeline(chunk[:512])
                    label = result[0]['label'].lower() if isinstance(result, list) and len(result) > 0 else 'neutral'
                    if 'positive' in label: positive_count += 1
                    elif 'negative' in label: negative_count += 1
                    else: neutral_count += 1
                except Exception:
                    neutral_count += 1

            total_valid = positive_count + negative_count + neutral_count
            if total_valid == 0: return None
            net_sentiment = (positive_count - negative_count) / total_valid

            if positive_count >= negative_count and positive_count >= neutral_count: dominant = 'positive'
            elif negative_count >= positive_count and negative_count >= neutral_count: dominant = 'negative'
            else: dominant = 'neutral'

            quarter_key = self._get_quarter_key(date)
            self.sentiment_cache[(ticker, quarter_key)] = net_sentiment
            dt = pd.to_datetime(date)
            prev_quarter_dt = dt - pd.DateOffset(months=3)
            prev_quarter_key = f"{prev_quarter_dt.year}-Q{(prev_quarter_dt.month - 1) // 3 + 1}"
            prev_sentiment = self.sentiment_cache.get((ticker, prev_quarter_key))
            sentiment_acceleration = (net_sentiment > prev_sentiment) if prev_sentiment is not None else False

            result = {
                'sentiment': dominant, 'net_sentiment': net_sentiment,
                'sentiment_acceleration': sentiment_acceleration,
                'positive_ratio': positive_count / total_valid if total_valid > 0 else 0,
                'chunks_processed': total_valid
            }
            self.llm_cache[cache_key] = result
            return result
        except Exception:
            return None

    # =====================================================================
    # DECISION LOGIC (Drawdown Vetoes & Entry/Exit)
    # =====================================================================

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        price = analytics.get('close', 0)
        if price <= 0: return 'HOLD'
        has_position = ticker in portfolio_state.get('positions', {})

        # Monthly update
        current_month = date[:7]
        if self.universe_last_updated_month != current_month:
            self._update_universe(date)

        # Extraction of key risk metrics
        market_vol = analytics.get('market_vol_20', 0)
        market_breadth = analytics.get('market_breadth_pct', 1.0)
        vol_10 = analytics.get('vol_10')
        vol_60 = analytics.get('vol_60')
        atr_14 = analytics.get('atr_14', price * 0.05) # fallback to 5% if missing

        # --- H2 & H6: BLACK SWAN CIRCUIT BREAKER ---
        # Correlation Breakdown & Vol Target Defense
        if not np.isnan(market_vol) and not np.isnan(market_breadth):
            if market_vol > 0.35 or market_breadth < 0.15:
                # Systemic Shock: Liquidate everything immediately
                if ticker in self.sleeve1_positions:
                    del self.sleeve1_positions[ticker]
                    return 'SELL'
                if ticker in self.sleeve2_positions:
                    del self.sleeve2_positions[ticker]
                    return 'SELL'
                return 'HOLD' # Block all new entries

        # --- Sleeve 1 EXIT ---
        if ticker in self.sleeve1_positions:
            if date > self.sleeve1_positions[ticker]:
                del self.sleeve1_positions[ticker]
                return 'SELL'

        # --- Sleeve 2 EXIT (Chandelier) ---
        if ticker in self.sleeve2_positions:
            pos_info = self.sleeve2_positions[ticker]
            pos_info['weeks_held'] += 1
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            # Volatility-adjusted trailing stop (3 * ATR)
            if price <= pos_info['peak_price'] - (3 * atr_14):
                del self.sleeve2_positions[ticker]
                return 'SELL'

            if pos_info['weeks_held'] >= 5:
                del self.sleeve2_positions[ticker]
                return 'SELL'
            return 'HOLD'

        in_universe = ticker in self.universe
        if has_position and not in_universe:
            if ticker not in self.sleeve1_positions and ticker not in self.sleeve2_positions:
                return 'SELL'
        if not in_universe or has_position:
            return 'HOLD'

        # --- DEFENSIVE VETO LOGIC FOR NEW ENTRIES ---
        
        # H3: Volatility Acceleration Veto (Blocks 24% probability of >10% crash)
        vol_acceleration_veto = False
        if vol_10 is not None and vol_60 is not None and not np.isnan(vol_10) and not np.isnan(vol_60):
            if vol_10 > 1.5 * vol_60:
                vol_acceleration_veto = True

        # H1: Momentum Skewness Veto
        momentum_skewness_veto = False
        if market_vol > 0.20 and ticker in self.momentum_winners:
            momentum_skewness_veto = True

        # --- ENTRY: Tactical Weekly ---
        rsi = analytics.get('rsi_14')
        daily_ret = analytics.get('daily_return')
        volume = analytics.get('volume')
        volume_ma = analytics.get('volume_ma_20')

        sleeve1_trigger = False
        if rsi is not None and not np.isnan(rsi):
            # Relaxed RSI to 35, enforced by Vetoes
            if rsi < 35 and not vol_acceleration_veto and not momentum_skewness_veto:
                sleeve1_trigger = True

        if not sleeve1_trigger and daily_ret is not None and volume is not None and volume_ma is not None:
            if not np.isnan(daily_ret) and not np.isnan(volume) and not np.isnan(volume_ma):
                # Relaxed Momentum volume to 1.25x
                if daily_ret > 0 and volume_ma > 0 and volume > 1.25 * volume_ma and not vol_acceleration_veto:
                    sleeve1_trigger = True

        if sleeve1_trigger:
            self.sleeve1_positions[ticker] = date
            return 'BUY'

        # --- ENTRY: Fundamental Momentum ---
        if transcript is not None and not vol_acceleration_veto:
            sentiment_result = self.llm_analysis(ticker, transcript, date)
            if sentiment_result is not None and sentiment_result.get('sentiment_acceleration', False):
                self.sleeve2_positions[ticker] = {
                    'entry_date': date,
                    'weeks_held': 0,
                    'peak_price': price
                }
                return 'BUY'

        return 'HOLD'

    # =====================================================================
    # EVALUATION ORCHESTRATION
    # =====================================================================

    def _build_analytics_lookup(self, analytics_df):
        lookup = defaultdict(list)
        for _, row in analytics_df.iterrows():
            lookup[row['ticker']].append((row['date'], row.to_dict()))
        for ticker in lookup:
            lookup[ticker].sort(key=lambda x: x[0])
        return lookup

    def evaluate(self, verbose=False):
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        self.sleeve1_positions = {}
        self.sleeve2_positions = {}
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None
        self.momentum_winners = set()

        print("Running evaluation...")
        analytics = self.calculate_analytics(self.prices)
        analytics_lookup = self._build_analytics_lookup(analytics)

        print("Running backtest simulation...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        results = sim.run(
            lambda t, d, tr, ps, a: self.make_decision(t, d, tr, ps, a),
            analytics_lookup, verbose
        )
        return results

# Enchanced Strat V5 Final (AMIHUD + Volatility Acceleration Drawdown Management)

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict

class EnhancedStrategy(BaseStrategy):
    """
    Multi-Sleeve Trading Strategy with Minimalist Drawdown Defense.
    (CPU-Vectorised Pandas Edition)
    """

    def __init__(self, finbert_pipeline=None, rsi_threshold=40, vol_mult=2, s1_atr_mult=2, s2_atr_mult=2, sleeve2_exit_weeks=5):
        super().__init__(finbert_pipeline)
        self.rsi_threshold = rsi_threshold
        self.vol_mult = vol_mult
        self.s1_atr_mult = s1_atr_mult
        self.s2_atr_mult = s2_atr_mult
        self.sleeve2_exit_weeks = sleeve2_exit_weeks

        # --- Universe selection state ---
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None

        # --- Sleeve tracking (Updated for Peak Price Tracking) ---
        self.sleeve1_positions = {}  # {ticker: {'entry_date': str, 'peak_price': float}}
        self.sleeve2_positions = {}  # {ticker: {'entry_date': str, 'weeks_held': int, 'peak_price': float}}
        self.sentiment_cache = {}

    # =====================================================================
    # DATA PREPARATION
    # =====================================================================

    def clean_data(self, prices_df, earnings_df):
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].str.len() > 100]

        prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)
        return prices, earnings

    # =====================================================================
    # ANALYTICS CALCULATION (Vectorised via Pandas GroupBy)
    # =====================================================================

    def calculate_analytics(self, prices_df):
        print("Computing advanced technicals and market regimes using Pandas vectorisation...")
        df = prices_df.copy().sort_values(['ticker', 'date'])

        ticker_index = df['ticker']
        grouped = df.groupby('ticker', sort=False)
        annualisation = np.sqrt(252)

        daily_return = grouped['close'].pct_change()
        close_rolling = df['close'].groupby(ticker_index, sort=False)
        volume_rolling = df['volume'].groupby(ticker_index, sort=False)

        df['daily_return'] = daily_return
        df['ma_200'] = close_rolling.rolling(200, min_periods=50).mean().reset_index(level=0, drop=True)
        
        daily_return_rolling = daily_return.groupby(ticker_index, sort=False)
        df['vol_60'] = daily_return_rolling.rolling(60, min_periods=30).std().mul(annualisation).reset_index(level=0, drop=True)
        df['vol_10'] = daily_return_rolling.rolling(10, min_periods=5).std().mul(annualisation).reset_index(level=0, drop=True)

        dollar_volume = df['close'] * df['volume']
        amihud_daily = daily_return.abs().div(dollar_volume.replace(0, np.nan))
        df['amihud_60'] = amihud_daily.groupby(ticker_index, sort=False).rolling(60, min_periods=30).mean().reset_index(level=0, drop=True)

        prev_close = grouped['close'].shift()
        if 'high' in df.columns and 'low' in df.columns:
            true_range = pd.concat([
                df['high'] - df['low'],
                (df['high'] - prev_close).abs(),
                (df['low'] - prev_close).abs(),
            ], axis=1).max(axis=1)
        else:
            true_range = (df['close'] - prev_close).abs()
        df['atr_14'] = true_range.groupby(ticker_index, sort=False).rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

        delta = grouped['close'].diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)
        avg_gain = gain.groupby(ticker_index, sort=False).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)
        avg_loss = loss.groupby(ticker_index, sort=False).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)
        rs = avg_gain.div(avg_loss.replace(0, np.nan))
        df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))

        df['volume_ma_20'] = volume_rolling.rolling(20, min_periods=10).mean().reset_index(level=0, drop=True)

        result_df = df[[
            'ticker', 'date', 'open', 'close', 'volume', 'daily_return',
            'vol_60', 'vol_10', 'amihud_60', 'rsi_14', 'volume_ma_20',
            'ma_200', 'atr_14'
        ]].copy()
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')

        self.universe_analytics_df = result_df
        return result_df

    # =====================================================================
    # UNIVERSE SELECTION
    # =====================================================================

    def _update_universe(self, current_date):
        if self.universe_analytics_df is None:
            return

        df = self.universe_analytics_df[self.universe_analytics_df['date'] <= current_date]
        if df.empty: return

        latest = df.groupby('ticker').last().reset_index()
        scored = latest.dropna(subset=['vol_60', 'amihud_60']).copy()
        if scored.empty: return

        scored['vol_rank'] = scored['vol_60'].rank(ascending=True, method='average')
        scored['amihud_rank'] = scored['amihud_60'].rank(ascending=False, method='average')
        scored['composite_score'] = scored['vol_rank'] + scored['amihud_rank']

        # Expanded to Top 40%
        threshold = scored['composite_score'].quantile(0.80)
        self.universe = set(scored[scored['composite_score'] >= threshold]['ticker'].tolist())

        month_key = current_date[:7]
        self.universe_last_updated_month = month_key
        print(f"  Universe updated ({month_key}): {len(self.universe)} stocks selected.")

    # =====================================================================
    # LLM / FINBERT ANALYSIS
    # =====================================================================

    def _get_quarter_key(self, date_str):
        dt = pd.to_datetime(date_str)
        return f"{dt.year}-Q{(dt.month - 1) // 3 + 1}"

    def llm_analysis(self, ticker, transcript, date):
        if transcript is None or self.finbert_pipeline is None: return None
        cache_key = f"{ticker}_{date}"
        if cache_key in self.llm_cache:
            self.llm_cache_hits += 1
            return self.llm_cache[cache_key]

        self.llm_cache_misses += 1
        try:
            sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())
            chunks, current_chunk = [], ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) > 400:
                    if current_chunk: chunks.append(current_chunk.strip())
                    current_chunk = sentence
                else:
                    current_chunk += " " + sentence
            if current_chunk.strip(): chunks.append(current_chunk.strip())
            if not chunks: return None

            pos, neg, neu = 0, 0, 0
            for chunk in chunks:
                if not chunk or len(chunk) < 10: continue
                try:
                    result = self.finbert_pipeline(chunk[:512])
                    label = result[0]['label'].lower() if isinstance(result, list) and len(result) > 0 else 'neutral'
                    if 'positive' in label: pos += 1
                    elif 'negative' in label: neg += 1
                    else: neu += 1
                except Exception:
                    neu += 1

            total = pos + neg + neu
            if total == 0: return None
            net_sentiment = (pos - neg) / total
            dominant = 'positive' if pos >= neg and pos >= neu else ('negative' if neg >= pos and neg >= neu else 'neutral')

            quarter_key = self._get_quarter_key(date)
            self.sentiment_cache[(ticker, quarter_key)] = net_sentiment
            dt = pd.to_datetime(date)
            prev_dt = dt - pd.DateOffset(months=3)
            prev_key = f"{prev_dt.year}-Q{(prev_dt.month - 1) // 3 + 1}"
            prev_sentiment = self.sentiment_cache.get((ticker, prev_key))
            accel = (net_sentiment > prev_sentiment) if prev_sentiment is not None else False

            result = {
                'sentiment': dominant, 'net_sentiment': net_sentiment,
                'sentiment_acceleration': accel, 'positive_ratio': pos / total if total > 0 else 0,
                'chunks_processed': total
            }
            self.llm_cache[cache_key] = result
            return result
        except Exception:
            return None

    # =====================================================================
    # DECISION LOGIC (Minimalist Defense Integrated)
    # =====================================================================

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        price = analytics.get('close', 0)
        if price <= 0: return 'HOLD'
        has_pos = ticker in portfolio_state.get('positions', {})

        if self.universe_last_updated_month != date[:7]:
            self._update_universe(date)

        atr_14 = analytics.get('atr_14', price * 0.05)

        # --- EXIT LOGIC ---
        if ticker in self.sleeve1_positions:
            pos_info = self.sleeve1_positions[ticker]
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            # Sleeve 1 Exit: ATR Trailing Stop OR time-based
            if price <= pos_info['peak_price'] - (self.s1_atr_mult * atr_14) or date > pos_info['entry_date']:
                del self.sleeve1_positions[ticker]
                return 'SELL'

        if ticker in self.sleeve2_positions:
            pos_info = self.sleeve2_positions[ticker]
            pos_info['weeks_held'] += 1
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            # Sleeve 2 Exit: ATR Trailing Stop OR time-based
            if price <= pos_info['peak_price'] - (self.s2_atr_mult * atr_14) or pos_info['weeks_held'] >= self.sleeve2_exit_weeks:
                del self.sleeve2_positions[ticker]
                return 'SELL'
            return 'HOLD'

        in_univ = ticker in self.universe
        if has_pos and not in_univ and ticker not in self.sleeve1_positions and ticker not in self.sleeve2_positions: return 'SELL'
        if not in_univ or has_pos: return 'HOLD'

        # --- DEFENSIVE VETO LOGIC ---
        vol_10, vol_60 = analytics.get('vol_10'), analytics.get('vol_60')
        ma_200 = analytics.get('ma_200')

        # H3: Volatility Acceleration Veto (Blocks 24% probability of >10% crash)
        vol_veto = (vol_10 is not None and vol_60 is not None and not np.isnan(vol_10) and not np.isnan(vol_60) and vol_10 > 1.5 * vol_60)

        # H7: Trend Filter Veto
        is_downtrend = (ma_200 is not None and price < ma_200)

        # --- ENTRY LOGIC ---
        rsi = analytics.get('rsi_14')
        daily_ret = analytics.get('daily_return')
        vol = analytics.get('volume')
        vol_ma = analytics.get('volume_ma_20')

        s1_trigger = False

        # Trigger A: Mean Reversion (Allowed in downtrends, blocked by Vol Acceleration)
        if rsi is not None and not np.isnan(rsi) and rsi < self.rsi_threshold and not vol_veto:
            s1_trigger = True

        # Trigger B: Momentum (Strictly blocked by Downtrend and Vol Acceleration)
        if not s1_trigger and not is_downtrend and not vol_veto:
            if all(v is not None and not np.isnan(v) for v in (daily_ret, vol, vol_ma)):
                if daily_ret > 0 and vol_ma > 0 and vol > self.vol_mult * vol_ma:
                    s1_trigger = True

        if s1_trigger:
            self.sleeve1_positions[ticker] = {'entry_date': date, 'peak_price': price}
            return 'BUY'

        # Sleeve 2: Earnings Momentum (Strictly blocked by Downtrend and Vol Acceleration)
        if transcript is not None and not is_downtrend and not vol_veto:
            sent_res = self.llm_analysis(ticker, transcript, date)
            if sent_res is not None and sent_res.get('sentiment_acceleration', False):
                self.sleeve2_positions[ticker] = {'entry_date': date, 'weeks_held': 0, 'peak_price': price}
                return 'BUY'

        return 'HOLD'

    # =====================================================================
    # EVALUATION ORCHESTRATION
    # =====================================================================

    def evaluate(self, verbose=False):
        if self.prices is None or self.earnings is None: raise ValueError("Must call set_data() before evaluate()")
        self.sleeve1_positions, self.sleeve2_positions, self.universe = {}, {}, set()
        self.universe_last_updated_month, self.universe_analytics_df = None, None

        print("Running evaluation...")
        analytics = self.calculate_analytics(self.prices)
        analytics_lookup = self._build_analytics_lookup(analytics)

        print("Running backtest simulation...")
        # Assuming TradingSimulation and STARTING_CASH are imported in your master script
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        return sim.run(lambda t, d, tr, ps, a: self.make_decision(t, d, tr, ps, a), analytics_lookup, verbose)


# Optimised Strat V5

In [ ]:
"""
EnhancedStrategy: Multi-Sleeve Trading Strategy (GPU-Optimized)
================================================================
Extends BaseStrategy with:
  - Step 1: Universe selection via volatility + Amihud illiquidity composite scoring
  - Step 2: FinBERT sentiment acceleration with bulk batched inference
  - Step 3: Dual-sleeve entry triggers (RSI mean reversion + volume momentum / sentiment acceleration)
  - Step 4: Time-based and ATR-based trailing-stop exits

GPU optimizations applied:
  - Vectorized Pandas groupby for analytics
  - Bulk FinBERT transcript precomputation so inference can run in large GPU batches
  - Weekly-aligned analytics lookup to reduce repeated daily-history scans during the backtest
"""
# Depends on notebook globals: BaseStrategy, TradingSimulation, STARTING_CASH

import re
from collections import defaultdict

import numpy as np
import pandas as pd

from enhanced_helpers.lookups import build_analytics_lookup_vectorised
from enhanced_helpers.sentiment import get_quarter_key


SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')


class EnhancedStrategy(BaseStrategy):
    """
    Multi-Sleeve Trading Strategy with Minimalist Drawdown Defense.
    GPU work is pushed into batched FinBERT inference while portfolio execution
    stays sequential to preserve the original trading behavior.
    """

    def __init__(self, finbert_pipeline=None, rsi_threshold=40, vol_mult=2,
                 s1_atr_mult=2, s2_atr_mult=2, sleeve2_exit_weeks=5):
        super().__init__(finbert_pipeline)
        self.rsi_threshold = rsi_threshold
        self.vol_mult = vol_mult
        self.s1_atr_mult = s1_atr_mult
        self.s2_atr_mult = s2_atr_mult
        self.sleeve2_exit_weeks = sleeve2_exit_weeks

        # Universe selection state
        self.universe = set()
        self.universe_last_updated_month = None
        self.universe_analytics_df = None
        self.precomputed_universes = {}

        # Sleeve tracking
        self.sleeve1_positions = {}  # {ticker: {'entry_date': str, 'peak_price': float}}
        self.sleeve2_positions = {}  # {ticker: {'entry_date': str, 'weeks_held': int, 'peak_price': float}}
        self.sentiment_cache = {}

        # Bulk sentiment preprocessing state
        self.sentiment_batch_size = 64
        self.sentiment_super_batch_size = 4096
        self.precomputed_llm_results = {}
        self.weekly_schedule = []

    # =====================================================================
    # DATA PREPARATION
    # =====================================================================

    def clean_data(self, prices_df, earnings_df):
        prices = prices_df.copy().drop_duplicates()
        earnings = earnings_df.copy().drop_duplicates()
        prices['date'] = pd.to_datetime(prices['date'])
        earnings['date'] = pd.to_datetime(earnings['date'])
        earnings = earnings[earnings['transcript'].fillna('').str.len() > 100]

        prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)
        earnings = earnings.sort_values(['ticker', 'date']).reset_index(drop=True)
        return prices, earnings

    # =====================================================================
    # ANALYTICS CALCULATION
    # =====================================================================

    def calculate_analytics(self, prices_df):
        print("Computing advanced technicals using fully vectorized Pandas groupby...")
        df = prices_df.copy().sort_values(['ticker', 'date'])

        ticker_index = df['ticker']
        grouped = df.groupby('ticker', sort=False)
        annualisation = np.sqrt(252.0)

        daily_return = grouped['close'].pct_change()
        close_rolling = df['close'].groupby(ticker_index, sort=False)
        volume_rolling = df['volume'].groupby(ticker_index, sort=False)

        df['daily_return'] = daily_return
        df['ma_200'] = close_rolling.rolling(200, min_periods=50).mean().reset_index(level=0, drop=True)

        daily_return_rolling = daily_return.groupby(ticker_index, sort=False)
        df['vol_60'] = daily_return_rolling.rolling(60, min_periods=30).std().mul(annualisation).reset_index(level=0, drop=True)
        df['vol_10'] = daily_return_rolling.rolling(10, min_periods=5).std().mul(annualisation).reset_index(level=0, drop=True)

        dollar_volume = df['close'] * df['volume']
        amihud_daily = daily_return.abs().div(dollar_volume.replace(0, np.nan))
        df['amihud_60'] = amihud_daily.groupby(ticker_index, sort=False).rolling(60, min_periods=30).mean().reset_index(level=0, drop=True)

        prev_close = grouped['close'].shift()
        if 'high' in df.columns and 'low' in df.columns:
            true_range = pd.concat([
                df['high'] - df['low'],
                (df['high'] - prev_close).abs(),
                (df['low'] - prev_close).abs(),
            ], axis=1).max(axis=1)
        else:
            true_range = (df['close'] - prev_close).abs()
        df['atr_14'] = true_range.groupby(ticker_index, sort=False).rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)

        delta = grouped['close'].diff()
        gain = delta.clip(lower=0)
        loss = (-delta).clip(lower=0)
        avg_gain = gain.groupby(ticker_index, sort=False).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)
        avg_loss = loss.groupby(ticker_index, sort=False).rolling(14, min_periods=14).mean().reset_index(level=0, drop=True)
        rs = avg_gain.div(avg_loss.replace(0, np.nan))
        df['rsi_14'] = 100.0 - (100.0 / (1.0 + rs))

        df['volume_ma_20'] = volume_rolling.rolling(20, min_periods=10).mean().reset_index(level=0, drop=True)

        result_df = df[[
            'ticker', 'date', 'open', 'close', 'volume', 'daily_return',
            'vol_60', 'vol_10', 'amihud_60', 'rsi_14', 'volume_ma_20',
            'ma_200', 'atr_14'
        ]].copy()
        result_df['date'] = result_df['date'].dt.strftime('%Y-%m-%d')

        self.universe_analytics_df = result_df
        print(f"  Analytics computed: {len(result_df):,} rows for {result_df['ticker'].nunique()} tickers")
        return result_df

    def _build_weekly_schedule(self, prices_df):
        min_date = pd.to_datetime(prices_df['date']).min()
        max_date = pd.to_datetime(prices_df['date']).max()
        return pd.date_range(start=min_date, end=max_date, freq='W-FRI').strftime('%Y-%m-%d').tolist()

    def _build_weekly_analytics_lookup(self, analytics_df, weekly_schedule):
        """
        Compress daily analytics down to the simulator's weekly schedule.
        The simulation still executes sequentially, but it no longer walks
        daily analytics rows to find the latest value for each week.
        """
        print("Aligning analytics to weekly schedule...")
        if analytics_df.empty or not weekly_schedule:
            return {}

        sorted_df = analytics_df.copy().sort_values(['ticker', 'date'])
        sorted_df['date'] = pd.to_datetime(sorted_df['date'])
        week_dates = pd.to_datetime(pd.Index(weekly_schedule))
        week_values = week_dates.to_numpy()

        aligned_frames = []
        for ticker, group in sorted_df.groupby('ticker', sort=False):
            group = group.reset_index(drop=True)
            group_dates = group['date'].to_numpy()
            positions = group_dates.searchsorted(week_values, side='right') - 1
            valid_mask = positions >= 0
            if not valid_mask.any():
                continue

            aligned = group.iloc[positions[valid_mask]].copy()
            aligned['date'] = week_dates[valid_mask].strftime('%Y-%m-%d')
            aligned['ticker'] = ticker
            aligned_frames.append(aligned)

        if not aligned_frames:
            return {}

        weekly_df = pd.concat(aligned_frames, ignore_index=True)
        print(f"  Weekly analytics aligned: {len(weekly_df):,} rows")
        return build_analytics_lookup_vectorised(weekly_df)

    # =====================================================================
    # UNIVERSE SELECTION
    # =====================================================================

    def _select_universe(self, analytics_slice):
        latest = analytics_slice.groupby('ticker').last().reset_index()
        scored = latest.dropna(subset=['vol_60', 'amihud_60']).copy()
        if scored.empty:
            return set()

        scored['vol_rank'] = scored['vol_60'].rank(ascending=True, method='average')
        scored['amihud_rank'] = scored['amihud_60'].rank(ascending=False, method='average')
        scored['composite_score'] = scored['vol_rank'] + scored['amihud_rank']

        threshold = scored['composite_score'].quantile(0.80)
        return set(scored[scored['composite_score'] >= threshold]['ticker'].tolist())

    def _precompute_monthly_universes(self, weekly_schedule):
        self.precomputed_universes = {}
        if self.universe_analytics_df is None or not weekly_schedule:
            return

        print("Precomputing monthly universe snapshots...")
        first_week_by_month = {}
        for week_date in weekly_schedule:
            month_key = week_date[:7]
            if month_key not in first_week_by_month:
                first_week_by_month[month_key] = week_date

        for month_key, cutoff_date in first_week_by_month.items():
            analytics_slice = self.universe_analytics_df[self.universe_analytics_df['date'] <= cutoff_date]
            if analytics_slice.empty:
                continue
            self.precomputed_universes[month_key] = self._select_universe(analytics_slice)

        print(f"  Universe snapshots ready for {len(self.precomputed_universes)} months")

    def _update_universe(self, current_date):
        month_key = current_date[:7]
        if month_key in self.precomputed_universes:
            self.universe = self.precomputed_universes[month_key]
            self.universe_last_updated_month = month_key
            print(f"  Universe updated ({month_key}): {len(self.universe)} stocks selected.")
            return

        if self.universe_analytics_df is None:
            return

        analytics_slice = self.universe_analytics_df[self.universe_analytics_df['date'] <= current_date]
        if analytics_slice.empty:
            return

        self.universe = self._select_universe(analytics_slice)
        self.universe_last_updated_month = month_key
        print(f"  Universe updated ({month_key}): {len(self.universe)} stocks selected.")

    # =====================================================================
    # LLM / FINBERT ANALYSIS
    # =====================================================================

    def _chunk_transcript(self, transcript):
        if transcript is None:
            return []

        transcript = str(transcript).strip()
        if not transcript:
            return []

        sentences = SENTENCE_SPLIT_RE.split(transcript)
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) > 400:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
            else:
                current_chunk += " " + sentence

        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        return [chunk[:512] for chunk in chunks if chunk and len(chunk) >= 10]

    def _label_bucket(self, result):
        label = result['label'].lower() if isinstance(result, dict) and 'label' in result else 'neutral'
        if 'positive' in label:
            return 0
        if 'negative' in label:
            return 1
        return 2

    def _build_sentiment_payload(self, pos, neg, neu):
        total = pos + neg + neu
        if total == 0:
            return None

        net_sentiment = (pos - neg) / total
        dominant = 'positive' if pos >= neg and pos >= neu else (
            'negative' if neg >= pos and neg >= neu else 'neutral'
        )
        return {
            'sentiment': dominant,
            'net_sentiment': net_sentiment,
            'positive_ratio': pos / total if total > 0 else 0.0,
            'chunks_processed': total
        }

    def _infer_sentiment_counts(self, valid_chunks):
        pos, neg, neu = 0, 0, 0
        try:
            batch_results = self.finbert_pipeline(
                valid_chunks,
                batch_size=self.sentiment_batch_size,
                truncation=True,
                max_length=512
            )
            if isinstance(batch_results, dict):
                batch_results = [batch_results]

            for result in batch_results:
                bucket = self._label_bucket(result)
                if bucket == 0:
                    pos += 1
                elif bucket == 1:
                    neg += 1
                else:
                    neu += 1
        except Exception:
            neu = len(valid_chunks)
        return pos, neg, neu

    def _precompute_llm_analysis(self):
        self.precomputed_llm_results = {}
        if self.finbert_pipeline is None or self.earnings is None or self.earnings.empty:
            return

        print("Precomputing transcript sentiment in batched mode...")
        earnings_df = self.earnings[['ticker', 'date', 'transcript']].copy()
        earnings_df['date'] = pd.to_datetime(earnings_df['date']).dt.strftime('%Y-%m-%d')

        chunk_texts = []
        chunk_owners = []
        seen_cache_keys = set()

        for row in earnings_df.itertuples(index=False):
            cache_key = f"{row.ticker}_{row.date}"
            if cache_key in seen_cache_keys:
                continue

            valid_chunks = self._chunk_transcript(row.transcript)
            if not valid_chunks:
                continue

            seen_cache_keys.add(cache_key)
            chunk_texts.extend(valid_chunks)
            chunk_owners.extend([cache_key] * len(valid_chunks))

        if not chunk_texts:
            print("  No valid transcript chunks found for precomputation")
            return

        sentiment_counts = defaultdict(lambda: [0, 0, 0])
        for start in range(0, len(chunk_texts), self.sentiment_super_batch_size):
            batch_chunks = chunk_texts[start:start + self.sentiment_super_batch_size]
            batch_owners = chunk_owners[start:start + self.sentiment_super_batch_size]
            try:
                batch_results = self.finbert_pipeline(
                    batch_chunks,
                    batch_size=self.sentiment_batch_size,
                    truncation=True,
                    max_length=512
                )
                if isinstance(batch_results, dict):
                    batch_results = [batch_results]
            except Exception:
                batch_results = None

            if batch_results is None or len(batch_results) != len(batch_chunks):
                for owner in batch_owners:
                    sentiment_counts[owner][2] += 1
                continue

            for owner, result in zip(batch_owners, batch_results):
                bucket = self._label_bucket(result)
                sentiment_counts[owner][bucket] += 1

        for cache_key, counts in sentiment_counts.items():
            payload = self._build_sentiment_payload(*counts)
            if payload is not None:
                self.precomputed_llm_results[cache_key] = payload

        print(
            f"  Precomputed {len(self.precomputed_llm_results):,} transcript sentiments "
            f"from {len(chunk_texts):,} chunks"
        )

    def llm_analysis(self, ticker, transcript, date):
        if transcript is None or self.finbert_pipeline is None:
            return None

        cache_key = f"{ticker}_{date}"
        if cache_key in self.llm_cache:
            self.llm_cache_hits += 1
            return self.llm_cache[cache_key]

        self.llm_cache_misses += 1
        try:
            base_result = self.precomputed_llm_results.get(cache_key)
            if base_result is None:
                valid_chunks = self._chunk_transcript(transcript)
                if not valid_chunks:
                    return None

                pos, neg, neu = self._infer_sentiment_counts(valid_chunks)
                base_result = self._build_sentiment_payload(pos, neg, neu)
                if base_result is None:
                    return None
                self.precomputed_llm_results[cache_key] = base_result

            quarter_key = get_quarter_key(date)
            net_sentiment = base_result['net_sentiment']
            self.sentiment_cache[(ticker, quarter_key)] = net_sentiment

            dt = pd.to_datetime(date)
            prev_dt = dt - pd.DateOffset(months=3)
            prev_key = f"{prev_dt.year}-Q{(prev_dt.month - 1) // 3 + 1}"
            prev_sentiment = self.sentiment_cache.get((ticker, prev_key))
            accel = (net_sentiment > prev_sentiment) if prev_sentiment is not None else False

            result = dict(base_result)
            result['sentiment_acceleration'] = accel
            self.llm_cache[cache_key] = result
            return result
        except Exception:
            return None

    # =====================================================================
    # DECISION LOGIC
    # =====================================================================

    def make_decision(self, ticker, date, transcript, portfolio_state, analytics):
        price = analytics.get('close', 0)
        if price <= 0:
            return 'HOLD'
        has_pos = ticker in portfolio_state.get('positions', {})

        if self.universe_last_updated_month != date[:7]:
            self._update_universe(date)

        atr_14 = analytics.get('atr_14', price * 0.05)

        # Exit logic
        if ticker in self.sleeve1_positions:
            pos_info = self.sleeve1_positions[ticker]
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            if price <= pos_info['peak_price'] - (self.s1_atr_mult * atr_14) or date > pos_info['entry_date']:
                del self.sleeve1_positions[ticker]
                return 'SELL'

        if ticker in self.sleeve2_positions:
            pos_info = self.sleeve2_positions[ticker]
            pos_info['weeks_held'] += 1
            pos_info['peak_price'] = max(pos_info['peak_price'], price)

            if price <= pos_info['peak_price'] - (self.s2_atr_mult * atr_14) or pos_info['weeks_held'] >= self.sleeve2_exit_weeks:
                del self.sleeve2_positions[ticker]
                return 'SELL'
            return 'HOLD'

        in_univ = ticker in self.universe
        if has_pos and not in_univ and ticker not in self.sleeve1_positions and ticker not in self.sleeve2_positions:
            return 'SELL'
        if not in_univ or has_pos:
            return 'HOLD'

        # Defensive veto logic
        vol_10 = analytics.get('vol_10')
        vol_60 = analytics.get('vol_60')
        ma_200 = analytics.get('ma_200')

        vol_veto = (
            vol_10 is not None and vol_60 is not None
            and not np.isnan(vol_10) and not np.isnan(vol_60)
            and vol_10 > 1.5 * vol_60
        )
        is_downtrend = ma_200 is not None and price < ma_200

        # Entry logic
        rsi = analytics.get('rsi_14')
        daily_ret = analytics.get('daily_return')
        vol = analytics.get('volume')
        vol_ma = analytics.get('volume_ma_20')

        s1_trigger = False

        if rsi is not None and not np.isnan(rsi) and rsi < self.rsi_threshold and not vol_veto:
            s1_trigger = True

        if not s1_trigger and not is_downtrend and not vol_veto:
            if all(v is not None and not np.isnan(v) for v in (daily_ret, vol, vol_ma)):
                if daily_ret > 0 and vol_ma > 0 and vol > self.vol_mult * vol_ma:
                    s1_trigger = True

        if s1_trigger:
            self.sleeve1_positions[ticker] = {'entry_date': date, 'peak_price': price}
            return 'BUY'

        if transcript is not None and not is_downtrend and not vol_veto:
            sent_res = self.llm_analysis(ticker, transcript, date)
            if sent_res is not None and sent_res.get('sentiment_acceleration', False):
                self.sleeve2_positions[ticker] = {'entry_date': date, 'weeks_held': 0, 'peak_price': price}
                return 'BUY'

        return 'HOLD'

    # =====================================================================
    # EVALUATION ORCHESTRATION
    # =====================================================================

    def evaluate(self, verbose=False):
        if self.prices is None or self.earnings is None:
            raise ValueError("Must call set_data() before evaluate()")

        self.sleeve1_positions, self.sleeve2_positions, self.universe = {}, {}, set()
        self.universe_last_updated_month, self.universe_analytics_df = None, None
        self.precomputed_universes = {}
        self.precomputed_llm_results = {}
        self.weekly_schedule = self._build_weekly_schedule(self.prices)

        print("Running evaluation...")
        analytics = self.calculate_analytics(self.prices)
        self._precompute_llm_analysis()
        self._precompute_monthly_universes(self.weekly_schedule)
        analytics_lookup = self._build_weekly_analytics_lookup(analytics, self.weekly_schedule)

        print("Running backtest simulation...")
        sim = TradingSimulation(self.prices, self.earnings, STARTING_CASH)
        return sim.run(lambda t, d, tr, ps, a: self.make_decision(t, d, tr, ps, a), analytics_lookup, verbose)


In [ ]:
# ============================================================
# CELL 14: Enhanced Strategy Evaluation
# ============================================================
#
# Evaluate your enhanced strategy on the DEV split during development.
#
# IMPORTANT: Split Usage
# ----------------------
# - **Dev Split**: Use THIS split for development and hyperparameter tuning
#   - Iterate on your strategy implementation
#   - Test different RSI periods, stop-loss levels, signal thresholds
#   - Compare against baseline performance on dev
#
# - **Val Split**: Reserve for FINAL performance reporting only
#   - Do not evaluate on val until your strategy is finalized
#   - This is your "test" set for reporting final metrics
#
# UNCOMMENT THE CODE BELOW ONCE YOU'VE IMPLEMENTED YOUR STRATEGY IN CELL 13

print("\n" + "="*70)
print("ENHANCED STRATEGY EVALUATION (Development)")
print("="*70)

# Create enhanced strategy instance
enhanced = EnhancedStrategy(finbert_pipeline)

# Evaluate on DEV split (for development/tuning)
print("\n[DEV SPLIT - For Development]")
results_enhanced_dev = run_evaluation(enhanced_strategy=enhanced, strategy='enhanced', split='dev')

# Calculate metrics
metrics_enhanced_dev = calculate_metrics(results_enhanced_dev)
ENHANCED_METRICS_DEV = {
    'return': metrics_enhanced_dev['total_return'],
    'sharpe': metrics_enhanced_dev['sharpe_ratio'],
    'drawdown': metrics_enhanced_dev['max_drawdown'],
    'win_rate': metrics_enhanced_dev['win_rate'],
    'volatility': metrics_enhanced_dev['volatility'],
    'trades': metrics_enhanced_dev['num_trades']
}

# Display results
print(f"Return: {ENHANCED_METRICS_DEV['return']:.2%}")
print(f"Sharpe Ratio: {ENHANCED_METRICS_DEV['sharpe']:.2f}")
print(f"Max Drawdown: {ENHANCED_METRICS_DEV['drawdown']:.2%}")
print(f"Win Rate: {ENHANCED_METRICS_DEV['win_rate']:.1%}")
print(f"Volatility: {ENHANCED_METRICS_DEV['volatility']:.2%}")
print(f"Total Trades: {ENHANCED_METRICS_DEV['trades']:,}")

print("\n" + "="*70)
print("DEVELOPMENT TIPS")
print("="*70)
print("• Iterate on your strategy implementation in Cell 13")
print("• Re-run this cell to test changes on dev split")
print("• Compare against baseline dev performance (run baseline on dev if needed)")
print("• When satisfied, run on val split for final reporting")
print("="*70)

# Visualization
plot_results(results_enhanced_dev, metrics_enhanced_dev, title="Enhanced Strategy - Dev Split (Development)")
#
# # ============================================================
# # Val Split Evaluation
# # ============================================================
# # ONLY uncomment when your strategy is finalized and ready for final testing

print("\n" + "="*70)
print("FINAL PERFORMANCE")
print("="*70)

print("\n[VAL SPLIT - Final Performance]")
results_enhanced_val = run_evaluation(enhanced_strategy=enhanced, strategy='enhanced', split='val')

metrics_enhanced_val = calculate_metrics(results_enhanced_val)
ENHANCED_METRICS_VAL = {
    'return': metrics_enhanced_val['total_return'],
    'sharpe': metrics_enhanced_val['sharpe_ratio'],
    'drawdown': metrics_enhanced_val['max_drawdown'],
    'win_rate': metrics_enhanced_val['win_rate'],
    'volatility': metrics_enhanced_val['volatility'],
    'trades': metrics_enhanced_val['num_trades']
}

print(f"Return: {ENHANCED_METRICS_VAL['return']:.2%}")
print(f"Sharpe Ratio: {ENHANCED_METRICS_VAL['sharpe']:.2f}")
print(f"Max Drawdown: {ENHANCED_METRICS_VAL['drawdown']:.2%}")

plot_results(results_enhanced_val, metrics_enhanced_val, title="Enhanced Strategy - Val Split (Final)")

In [ ]:
# ============================================================
# CELL 15: Performance Comparison & Analysis
# ============================================================

# Once you've implemented and run your enhanced strategy (Cell 14),
# use the code below to compare performance against the baseline.

# The comparison functions are defined in Cell 8 and ready to use:
#   - plot_comparison(): Side-by-side visualizations
#   - print_detailed_comparison(): Detailed metrics table

# ============================================================
# USAGE EXAMPLES
# ============================================================
# Uncomment the code below once you have results from Cell 14

print_detailed_comparison(BASELINE_METRICS_VAL, ENHANCED_METRICS_VAL)
plot_comparison(results_baseline_val, metrics_baseline_val,
                results_enhanced_val, metrics_enhanced_val)